# Cross-Dataset Vascular Niche Analysis
## scRNA-seq Pericyte State Analysis + Spatial Program Enrichment
### Merged Notebook — Public scRNA-seq Validation & ND/T2D Stromal Remodeling

**Input:**
- `pancdbmes_subclusteredfiltered_nd_t2d_cleaned.h5ad` → `mesenchymalf`
  (Public pancreatic mesenchymal scRNA-seq, pre-clustered into MSL1/MSL2)
- `ND_merscopeROIinfo.h5ad` → `adata_ND` (MERSCOPE ND spatial data)
- `t2dcells_allsamples.h5ad` → `adata` (MERSCOPE T2D spatial data)

**Outputs:**
- `allvasc_cells.h5ad` → `vasc_adataf` (merged ND+T2D vascular object)
- `alladata_all_cells.h5ad` → `adata_all` (merged full dataset)
- Per-cell-type program enrichment CSVs
- MSL1/MSL2 DEG tables and pathway enrichment tables
- Publication-ready heatmaps, barplots, dotplots

---

### Pipeline Overview

**PART 1 — Public scRNA-seq Pericyte State Analysis (mesenchymalf)**

1. Load pre-clustered public pancreatic mesenchymal dataset
2. UMAP visualization: MSL1 vs MSL2 states, disease condition, donor
3. Load NABA Matrisome and Basement Membrane GMT files
4. Score ECM and BM gene sets per cell
5. Dotplot: ECM/BM program enrichment by cluster × disease
6. DEG: MSL2 T2D vs Control (Wilcoxon, use_raw=False)
7. Pathway enrichment on DEG gene lists:
   - GO Biological Process (Enrichr + GSEApy prerank)
   - MSigDB Hallmark (Enrichr)
   - KEGG 2021
   - Reactome 2024
   - WikiPathway 2023
8. 4-panel pathway figure: MSL1 vs MSL2 × Hallmark × Reactome
9. Custom Matrisome ECM category scoring per cell
10. Gene block heatmaps: functional modules
    (BM, ECM Remodeling, Contractility, Inflammatory Niche,
    Angiogenic Signaling)
11. DEG-driven heatmap: top 25 up + 15 down per state,
    pseudobulk mean, log1p + z-score, hierarchical clustering

**PART 2 — Merged Spatial Vascular Atlas (vasc_adataf)**

12. Load ND MERSCOPE + T2D MERSCOPE spatial objects
13. Subset vascular cells (Pericytes, ECs, Fibroblasts,
    Islet-associated Fibroblasts) → concat → `vasc_adataf`
14. Fix ArrowStringArray encoding issues before saving
15. Concord batch integration (domain_key = Sample) + UMAP
16. Leiden clustering (res=1.0) on Concord neighbor graph
17. Cluster merging into final vascular cell types:
    `merged_cell_type` in `vasc_adataf`
18. Relabel islet fibroblasts as Islet-associated Fibroblasts
    (Location == "Islet" filter)
19. DEG: T2D vs ND per cell type per compartment (Wilcoxon)
20. Vascular composition barplots (ND vs T2D, Islet + Exocrine)
21. Endothelial:Pericyte ratio analysis (donor-level MWU)
22. Islet fibroblast:Pericyte ratio analysis (donor-level MWU)
23. Exocrine fibroblast:Pericyte ratio analysis
24. Vascular:Endocrine ratio (cross-object, vasc_adataf + adata_all)
25. Spatial niche metrics:
    BM continuity, vascular coverage, local ECM intensity,
    islet edge organization (STRtree spatial indexing)

**PART 3 — Custom Program Enrichment (vasc_adataf)**

26. Score 6 custom gene programs per cell:
    Stress/ImmediateEarly, Contractile/Mural, ECM_Remodeling,
    Vascular_BM, Fibrillar_ECM, Pericyte_Identity
27. Donor-level pseudobulk (n=donors, not cells)
28. MWU: T2D vs ND per program per location (BH-FDR corrected)
29. Run separately for: Pericytes, Islet-assoc Fibroblasts,
    Endothelial Cells, Exocrine Fibroblasts
30. Combined heatmaps: delta(T2D-ND) per program × group
31. Z-score heatmaps: relative program states across
    ND/T2D × Islet/Exocrine conditions
32. Multi-compartment heatmap: 4 groups × 6 programs
33. Donor-level barplots per program per compartment

---

### Cluster → Cell Type Mapping (vasc_adataf)

| Leiden Clusters | Final Label |
|---|---|
| 1, 2, 4, 11, 12 | Endothelial Cells |
| 5, 9, 14 | Pericytes |
| 0, 6 | Fibroblasts |
| 10 | Islet-associated Fibroblasts |
| Fibroblasts in Islet | → Islet-associated Fibroblasts |
| 7, 13, 3, 15, 8 | Removed (ambiguous) |

### MSL States (mesenchymalf)

| Cluster | Interpretation |
|---|---|
| MSL1 | Fibroblast-like mesenchymal state (collagen high) |
| MSL2 | Contractile/pericyte-like mesenchymal state |

---

### Custom Program Definitions

| Program | Key Genes |
|---|---|
| Stress_ImmediateEarly | JUN, FOS, ATF3 |
| Contractile_Mural | MYL9, CALD1, TPM2, TAGLN, ACTA2 |
| ECM_Remodeling | THBS1, ADAM9, SERPINF1, MFGE8, SPARCL1, FBN1, MMP2 |
| Vascular_BM | COL4A1, COL4A2, LAMB1, LAMB2, LAMC1, HSPG2 |
| Fibrillar_ECM | COL1A1/2, COL3A1, COL5A1/2, COL6A1/2/3 |
| Pericyte_Identity | RGS5, PDGFRB, CSPG4, MCAM, NOTCH3 |
| Fibroblast_Identity | LUM, DCN, FBLN1, PRELP, SFRP2 |

All programs filtered to genes present in `var_names` at runtime.
Minimum 2 genes required to score a program.

---

### Gene Set Sources

| Source | Used For |
|---|---|
| NABA_CORE_MATRISOME.v2025.1.Hs.gmt | ECM matrisome gene universe |
| NABA_BASEMENT_MEMBRANES.v2025.1.Hs.gmt | BM-specific gene set |
| MSigDB Hallmark 2020 (Enrichr) | Broad pathway activity |
| KEGG 2021 Human (Enrichr) | Metabolic and signaling pathways |
| Reactome 2024 (Enrichr) | Detailed pathway hierarchies |
| WikiPathway 2023 Human (Enrichr) | Cross-database validation |
| GO Biological Process 2023 (Enrichr + GSEApy) | Functional annotation |
| Custom curated programs | Vascular niche biology |

---

### Statistical Framework

| Step | Method |
|---|---|
| DEG | Wilcoxon rank-sum, use_raw=False |
| Pathway enrichment | Enrichr (ORA) + GSEApy prerank (GSEA) |
| Program testing | Mann-Whitney U, donor pseudobulk |
| Multiple testing | Benjamini-Hochberg FDR |
| Pseudobulk | Donor-level mean score (avoids pseudoreplication) |

---

### Key Output Columns

| Column | Object | Description |
|---|---|---|
| `merged_cell_type` | vasc_adataf | Final vascular cell type labels |
| `mesenchymal_leiden` | mesenchymalf | MSL1 / MSL2 cluster labels |
| `disease_state` | mesenchymalf | Control / T2D |
| `Health_Status` | vasc_adataf | ND / T2D |
| `Location` | vasc_adataf | Islet / Exocrine |

---

### Color Palette

| Cell Type / State | Hex |
|---|---|
| MSL1 | #2230F4 |
| MSL2 | #df0a31 |
| Control | #6B5327 |
| T2D | #E378EB |
| Pericytes | #FFA500 |
| Endothelial Cells | #32CD32 |
| Islet-associated Fibroblasts | #FF69B4 |
| Fibroblasts | #00FFFF |
| Up in T2D | #c45b8a |
| Down in T2D | #8b5e3c |

In [ ]:
import concord as ccd
import scanpy as sc
import anndata as ad
import torch
import os
import pandas as pd 
import numpy as np
import seaborn as sns

In [ ]:
# =========================================================
# SECTION 1 — DATA LOADING
# Load pre-clustered public pancreatic mesenchymal scRNA-seq
# MSL1 = fibroblast-like, MSL2 = contractile/pericyte-like
# Source: pancreatic database, ND + T2D donors
# =========================================================

mesenchymalf= sc.read_h5ad('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdbmes_subclusteredfiltered_nd_t2d_cleaned.h5ad')

In [ ]:
mesenchymalf

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PALETTES
# =========================
cell_palette = {
    "MSL1": "#2230F4",
    "MSL2": "#df0a31"
}

location_palette = {
    "Control": "#6B5327",      # dark purple
    "T2D": "#E378EB"    # light purple
}

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(1, 2, figsize=(4, 2))

# ---- PANEL 1: CELL TYPE ----
sc.pl.embedding(
    mesenchymalf,
    basis="Concord_UMAP",
    color="mesenchymal_leiden",
    palette=cell_palette,
    size=1,
    frameon=False,
    legend_loc="center left",
    ax=axes[0],
    show=False,
    sort_order=True
)

# ---- PANEL 2: LOCATION ----
sc.pl.embedding(
    mesenchymalf,
    basis="Concord_UMAP",
    color="disease_state",
    palette=location_palette,
    size=1,
    frameon=False,
    legend_loc="center left",
    ax=axes[1],
    show=False,
    sort_order=True
)

# =========================
# FORMAT
# =========================
for ax in axes:
    ax.set_title("")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_linewidth(1)

    # rasterize points for small file size
    for col in ax.collections:
        col.set_rasterized(True)

plt.tight_layout()

# =========================
# SAVE
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/UMAP_vasc_celltype_location_"

os.makedirs(os.path.dirname(out_base), exist_ok=True)

plt.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
plt.savefig(f"{out_base}.pdf", bbox_inches="tight")

plt.show()

print("✔ DONE — clean UMAP exported")

In [ ]:
# --- core loaders ---
def load_gmt(path):
    """Return dict: set_name -> list of genes."""
    gs = {}
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                set_name = parts[0]
                genes = [g for g in parts[2:] if g]
                gs[set_name] = genes
    return gs


def genes_from_gmt(path, set_name):
    """Return gene list for one named set."""
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3 and parts[0] == set_name:
                return [g for g in parts[2:] if g]
    return []


def all_genes_flat(path):
    """Return one flat unique list of all genes in the GMT file."""
    seen, out = set(), []
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            for g in parts[2:]:
                if g and g not in seen:
                    seen.add(g)
                    out.append(g)
    return out


# --- use them ---
gmt_path = "/Users/kmeneses/Downloads/NABA_CORE_MATRISOME.v2025.1.Hs.gmt"

gene_sets = load_gmt(gmt_path)               # dict of sets
corematrisome_genes = genes_from_gmt(gmt_path, list(gene_sets.keys())[0])  # pick first set name
ecm_genes = all_genes_flat(gmt_path)         # one big list, de-duplicated

print(f"Found {len(gene_sets)} sets; first set has {len(corematrisome_genes)} genes; total unique genes = {len(ecm_genes)}")

In [ ]:
# --- core loaders ---
def load_gmt(path):
    """Return dict: set_name -> list of genes."""
    gs = {}
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                set_name = parts[0]
                genes = [g for g in parts[2:] if g]
                gs[set_name] = genes
    return gs


def genes_from_gmt(path, set_name):
    """Return gene list for one named set."""
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3 and parts[0] == set_name:
                return [g for g in parts[2:] if g]
    return []


def all_genes_flat(path):
    """Return one flat unique list of all genes in the GMT file."""
    seen, out = set(), []
    with open(path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            for g in parts[2:]:
                if g and g not in seen:
                    seen.add(g)
                    out.append(g)
    return out


# --- use them ---
gmt_path = "/Users/kmeneses/Downloads/NABA_BASEMENT_MEMBRANES.v2025.1.Hs.gmt"

gene_sets = load_gmt(gmt_path)               # dict of sets
bm_genes = genes_from_gmt(gmt_path, list(gene_sets.keys())[0])  # pick first set name
bmall_genes = all_genes_flat(gmt_path)         # one big list, de-duplicated

print(f"Found {len(gene_sets)} sets; first set has {len(bm_genes)} genes; total unique genes = {len(bmall_genes)}")

In [ ]:
# Convert to sets for fast operations
bm_set = set(bmall_genes)
ecm_set = set(ecm_genes)

# Remove BM genes from ECM
ecm_filtered = list(ecm_set - bm_set)

print(f"Original ECM genes: {len(ecm_genes)}")
print(f"Filtered ECM genes: {len(ecm_filtered)}")
print(f"Removed overlap: {len(ecm_set & bm_set)}")

In [ ]:
# SECTION 4 — LOAD NABA MATRISOME GENE SETS
# NABA_CORE_MATRISOME → ecm_genes (full matrisome)
# NABA_BASEMENT_MEMBRANES → bm_genes (BM subset)
# BM genes removed from ECM to avoid double-counting
# =========================================================

mpl.rcParams["svg.fonttype"] = "none"   # keeps text editable
mpl.rcParams["font.family"] = "Arial"   # optional but recommended
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

# Define properly as dictionary
gene_sets = {
    "ECM_Matrisome": ecm_filtered,
    "Basement_Membrane": bmall_genes
}

# Score gene sets
for gs_name, gene_list in gene_sets.items():

    valid_genes = [g for g in gene_list if g in mesenchymalf.var_names]

    if len(valid_genes) == 0:
        print(f"⚠️ Skipping {gs_name} (no valid genes found)")
        continue

    sc.tl.score_genes(
        mesenchymalf,
        gene_list=valid_genes,
        score_name=gs_name,
        use_raw=False
    )

# Collect successfully scored sets
scored_gene_sets = [
    gs for gs in gene_sets.keys()
    if gs in mesenchymalf.obs.columns
]

print("Scored gene sets:", scored_gene_sets)

# Plot
dp = sc.pl.dotplot(
    mesenchymalf,
    var_names=scored_gene_sets,
    groupby=['mesenchymal_leiden', 'disease_state'],
    title="Relative Enrichment of ECM Programs",
    colorbar_title="Z-score",
    cmap="viridis",
    standard_scale="var",
    return_fig=True   # <-- important
)

# Save as SVG
dp.savefig("/Users/kmeneses/Desktop/Figure4_plots/ECM_gene_sethadata_dotplot_clean.svg", dpi= 600, bbox_inches="tight")



In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# PALETTES
# =========================
cell_palette = {
    "MSL1": "#2230F4",
    "MSL2": "#df0a31"
}

disease_palette = {
    "Control": "#6B5327",
    "T2D": "#E378EB"
}

# =========================
# FIGURE
# =========================
fig, axes = plt.subplots(
    1,
    3,
    figsize=(6, 3)
)

# =========================
# PANEL 1 — MSL STATES
# =========================
sc.pl.embedding(
    mesenchymalf,
    basis="Concord_UMAP",
    color="mesenchymal_leiden",
    palette=cell_palette,
    size=5,
    frameon=False,
    legend_loc="right",
    ax=axes[0],
    show=False,
    sort_order=True
)

axes[0].set_title(
    "Pericyte states",
    fontsize=8
)

# =========================
# PANEL 2 — DISEASE STATE
# =========================
sc.pl.embedding(
    mesenchymalf,
    basis="Concord_UMAP",
    color="disease_state",
    palette=disease_palette,
    size=5,
    frameon=False,
    legend_loc="right",
    ax=axes[1],
    show=False,
    sort_order=True
)

axes[1].set_title(
    "Disease state",
    fontsize=8
)

# =========================
# PANEL 3 — DONOR
# =========================
sc.pl.embedding(
    mesenchymalf,
    basis="Concord_UMAP",
    color="donor_id",
    size=5,
    frameon=False,
    legend_loc='right',
    ax=axes[2],
    show=False,
    sort_order=True
)

axes[2].set_title(
    "Donor distribution",
    fontsize=8
)

# =========================
# FORMAT
# =========================
for ax in axes:

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    for spine in ax.spines.values():
        spine.set_visible(False)

    # rasterize points for smaller vector files
    for col in ax.collections:
        col.set_rasterized(True)

# =========================
# LAYOUT
# =========================
plt.tight_layout(
    w_pad=1
)

# =========================
# SAVE
# =========================
out_base = (
    "/Users/kmeneses/Desktop/"
    "Figure4_plots/SI/"
    "UMAP_vasc_celltype_location_donor"
)

os.makedirs(
    os.path.dirname(out_base),
    exist_ok=True
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ DONE — publication-ready UMAP exported")

In [ ]:
# Wilcoxon test on MSL2 subset (use_raw=False)
# Strict filter: FDR < 0.05, |LFC| > 0.25
# Removes: ribosomal, mitochondrial, contamination genes
# up_genes / dn_genes used for all pathway analyses below
# =========================================================

# ── SETTINGS ───────────────────────────────────────────────────────
CELLTYPE_COL = "mesenchymal_leiden"
DISEASE_COL  = "disease_state"
CLUSTER      = "MSL2"

# ── SUBSET ─────────────────────────────────────────────────────────
adata_sub = mesenchymalf[
    mesenchymalf.obs[CELLTYPE_COL] == CLUSTER
].copy()
print(adata_sub.obs[DISEASE_COL].value_counts())

# ── DEG ────────────────────────────────────────────────────────────
sc.tl.rank_genes_groups(
    adata_sub,
    groupby=DISEASE_COL,
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)
deg = sc.get.rank_genes_groups_df(adata_sub, group="T2D")
deg = deg.dropna(subset=["logfoldchanges"])
print(f"DEGs: {len(deg)}")

# ── DOWNLOAD GENE SETS ─────────────────────────────────────────────
url = "https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2023"
r   = requests.get(url)
library = {}
for line in r.text.strip().split("\n"):
    parts = line.split("\t")
    term  = parts[0]
    genes = [g for g in parts[2:] if g]
    if genes:
        library[term] = genes
print(f"Downloaded {len(library)} gene sets")

# ── RANKED LIST ────────────────────────────────────────────────────
rnk_df = (
    deg[["names", "logfoldchanges"]]
    .drop_duplicates(subset="names")
    .sort_values("logfoldchanges", ascending=False)
    .reset_index(drop=True)
)
rnk_df.columns = ["gene", "score"]

# ── RUN GSEA ───────────────────────────────────────────────────────
enr = gp.prerank(
    rnk=rnk_df,
    gene_sets=library,
    min_size=5,
    max_size=500,
    permutation_num=1000,
    seed=1,
    outdir=None,
    verbose=True
)

# ── RESULTS ────────────────────────────────────────────────────────
gsea_res = enr.res2d.copy()
print(gsea_res.columns.tolist())   # check column names first

In [ ]:
# diagnose
print("Total terms:", len(gsea_res))
print("NES range:", gsea_res["NES"].min(), "to", gsea_res["NES"].max())
print("FDR range:", gsea_res["FDR q-val"].min(), "to", gsea_res["FDR q-val"].max())
print("\nTop 10 by NES:")
print(gsea_res[["Term","NES","FDR q-val"]].head(10).to_string())
print("\nBottom 10 by NES:")
print(gsea_res[["Term","NES","FDR q-val"]].tail(10).to_string())

In [ ]:
# Downloads GO BP 2023 from Enrichr API
# Ranking metric: sign(LFC) * -log10(FDR)
# Better than LFC alone for low-expressed genes
# Runs GSEApy prerank (1000 permutations, seed=1)
# =========================================================

# upregulated in T2D
up_genes = deg[deg["logfoldchanges"] > 0]["names"].tolist()
dn_genes = deg[deg["logfoldchanges"] < 0]["names"].tolist()

enr_up = gp.enrichr(
    gene_list=up_genes,
    gene_sets="GO_Biological_Process_2023",
    organism="Human",
    outdir=None
)
enr_dn = gp.enrichr(
    gene_list=dn_genes,
    gene_sets="GO_Biological_Process_2023",
    organism="Human",
    outdir=None
)

res_up = enr_up.results[enr_up.results["Adjusted P-value"] < 0.05].head(10).copy()
res_dn = enr_dn.results[enr_dn.results["Adjusted P-value"] < 0.05].head(10).copy()

res_up["score"] =  -np.log10(res_up["Adjusted P-value"])
res_dn["score"] = -(-np.log10(res_dn["Adjusted P-value"]))

plot_df = pd.concat([res_up, res_dn]).sort_values("score")
plot_df["Term"] = plot_df["Term"].str.replace(r"\(GO:\d+\)", "", regex=True).str.strip()

colors = ["#d62728" if s > 0 else "#1f77b4" for s in plot_df["score"]]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(plot_df["Term"], plot_df["score"], color=colors)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Signed -log10(adj p-value)")
ax.set_title("MSL2: T2D vs Control\nGO Biological Process")
plt.tight_layout()

plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_vs_ND_enrichr.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_vs_ND_enrichr.pdf", bbox_inches="tight")
plt.show()

res_up.to_csv("/Users/kmeneses/Desktop/MSL2_T2D_UP_enrichr.csv", index=False)
res_dn.to_csv("/Users/kmeneses/Desktop/MSL2_T2D_DN_enrichr.csv", index=False)
print("✔ Done")

In [ ]:
# cleaner version
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(plot_df["Term"], plot_df["score"], color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Signed -log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nGO Biological Process", fontsize=9, loc="right")
ax.tick_params(axis="y", labelsize=7)
ax.tick_params(axis="x", labelsize=7)
ax.spines[["top","right"]].set_visible(False)

# add FDR label
ax.text(0.98, 0.01, "FDR < 0.05", transform=ax.transAxes,
        fontsize=6, color="gray", ha="right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_vs_ND_enrichr.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_vs_ND_enrichr.pdf", bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_vs_ND_enrichr.svg", bbox_inches="tight")
plt.show()

In [ ]:
# only use high confidence DEGs
deg_strict = deg[
    (deg["pvals_adj"] < 0.05) &
    (deg["logfoldchanges"].abs() > 0.5)  # meaningful fold change
]

print(f"Strict DEGs: {len(deg_strict)}")
print(f"Up: {(deg_strict['logfoldchanges']>0).sum()}")
print(f"Down: {(deg_strict['logfoldchanges']<0).sum()}")

up_genes = deg_strict[deg_strict["logfoldchanges"] > 0]["names"].tolist()
dn_genes = deg_strict[deg_strict["logfoldchanges"] < 0]["names"].tolist()

In [ ]:
# Tests up and down gene lists separately via Enrichr API
# Databases tested:
#   GO Biological Process 2023
#   KEGG 2021 Human
#   Reactome 2024
#   WikiPathway 2023 Human
# Noisy/irrelevant terms filtered before plottingfor gene_set in ["KEGG_2021_Human", "Reactome_Pathways_2024", "WikiPathway_2023_Human"]:
    enr_up = gp.enrichr(gene_list=up_genes, gene_sets=gene_set, organism="Human", outdir=None)
    enr_dn = gp.enrichr(gene_list=dn_genes, gene_sets=gene_set, organism="Human", outdir=None)
    
    sig_up = enr_up.results[enr_up.results["Adjusted P-value"] < 0.05]
    sig_dn = enr_dn.results[enr_dn.results["Adjusted P-value"] < 0.05]
    print(f"\n{gene_set}: {len(sig_up)} up, {len(sig_dn)} down")
    print(sig_up[["Term","Adjusted P-value"]].head(5).to_string())

In [ ]:
# get full reactome results
enr_up_r = gp.enrichr(gene_list=up_genes, gene_sets="Reactome_Pathways_2024",
                       organism="Human", outdir=None)
enr_dn_r = gp.enrichr(gene_list=dn_genes, gene_sets="Reactome_Pathways_2024",
                       organism="Human", outdir=None)

r_up = enr_up_r.results[enr_up_r.results["Adjusted P-value"]<0.05].head(10).copy()
r_dn = enr_dn_r.results[enr_dn_r.results["Adjusted P-value"]<0.05].head(10).copy()

def shorten(s, n=45):
    return s[:n]+"..." if len(s)>n else s

r_up["Term"] = r_up["Term"].apply(shorten)
r_dn["Term"] = r_dn["Term"].apply(shorten)
r_up["score"] =  -np.log10(r_up["Adjusted P-value"])
r_dn["score"] = -(-np.log10(r_dn["Adjusted P-value"]))

plot_df = pd.concat([r_dn, r_up]).sort_values("score")
colors  = ["#d62728" if s>0 else "#1f77b4" for s in plot_df["score"]]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(plot_df["Term"], plot_df["score"],
        color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)

for i, (_, row) in enumerate(plot_df.iterrows()):
    xpos = 0.1 if row["score"]>0 else -0.1
    ha   = "left" if row["score"]>0 else "right"
    ax.text(xpos, i, row["Overlap"], va="center",
            fontsize=6, color="white", fontweight="bold", ha=ha)

ax.set_xlabel("-log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nReactome 2024", fontsize=9, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
ax.tick_params(labelsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome.png", dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome.pdf", bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
# filter noisy terms
exclude = [
    "Cobalamin", "L13a", "Dietary",      # not relevant
    "mRNA Activation", "Translation Initiation",  # redundant with ribosome
    "Ribosomal Scanning",
    "Regulation of Expression of SLITs",  # not relevant
    "Signaling by ROBO"
]

r_up_clean = r_up[~r_up["Term"].apply(
    lambda x: any(e in x for e in exclude))].head(6)
r_dn_clean = r_dn[~r_dn["Term"].apply(
    lambda x: any(e in x for e in exclude))].head(8)

r_up_clean["score"] =  -np.log10(r_up_clean["Adjusted P-value"])
r_dn_clean["score"] = -(-np.log10(r_dn_clean["Adjusted P-value"]))

plot_df = pd.concat([r_dn_clean, r_up_clean]).sort_values("score")
colors  = ["#d62728" if s>0 else "#1f77b4" for s in plot_df["score"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(plot_df["Term"], plot_df["score"],
        color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)

for i, (_, row) in enumerate(plot_df.iterrows()):
    xpos = 0.1 if row["score"]>0 else -0.1
    ha   = "left" if row["score"]>0 else "right"
    ax.text(xpos, i, row["Overlap"], va="center",
            fontsize=6, color="white", fontweight="bold", ha=ha)

ax.set_xlabel("-log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nReactome 2024", fontsize=9, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
ax.tick_params(labelsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome_clean.png", dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome_clean.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── clarify which gene lists you're using ─────────────────────────
print("Which genes are you using?")
print(f"up2 (MSL2 pseudobulk nominal): {len(up2)}")
print(f"up_genes (old Wilcoxon):        {len(up_genes)}")

# ── run reactome with pseudobulk genes ────────────────────────────
e_up_r2 = gp.enrichr(gene_list=up2, gene_sets="Reactome_Pathways_2024",
                      organism="Human", outdir=None)
e_dn_r2 = gp.enrichr(gene_list=dn2, gene_sets="Reactome_Pathways_2024",
                      organism="Human", outdir=None)

r_up2 = e_up_r2.results[e_up_r2.results["Adjusted P-value"]<0.05].copy()
r_dn2 = e_dn_r2.results[e_dn_r2.results["Adjusted P-value"]<0.05].copy()

print(f"\nReactome pseudobulk MSL2: {len(r_up2)} up, {len(r_dn2)} down")
print("\nTOP UP:");  print(r_up2[["Term","Overlap","Adjusted P-value"]].head(10).to_string())
print("\nTOP DOWN:"); print(r_dn2[["Term","Overlap","Adjusted P-value"]].head(10).to_string())

In [ ]:
# ── what's actually in up2 vs up_genes ────────────────────────────
ecm_markers = ["COL1A1","COL1A2","COL3A1","MMP2","MMP14",
                "FN1","POSTN","FAP","LOXL2","TGFBI"]

print("ECM genes in pseudobulk up2:")
print([g for g in ecm_markers if g in up2])

print("\nECM genes in old Wilcoxon up_genes:")
print([g for g in ecm_markers if g in up_genes])

# check where these genes are in pseudobulk results
print("\nECM genes in MSL2 pseudobulk:")
print(res2[res2["gene"].isin(ecm_markers)]
      [["gene","log2FC","pval","padj"]].sort_values("log2FC",ascending=False).to_string())

In [ ]:
def shorten(s, n=45):
    return s[:n]+"..." if len(s)>n else s

r_up2_p = r_up2.head(4).copy()
r_dn2_p = r_dn2.head(8).copy()
r_up2_p["Term"] = r_up2_p["Term"].apply(shorten)
r_dn2_p["Term"] = r_dn2_p["Term"].apply(shorten)
r_up2_p["score"] =  -np.log10(r_up2_p["Adjusted P-value"])
r_dn2_p["score"] = -(-np.log10(r_dn2_p["Adjusted P-value"]))

react_df = pd.concat([r_dn2_p, r_up2_p]).sort_values("score")
colors_r = ["#d62728" if s>0 else "#1f77b4" for s in react_df["score"]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(react_df["Term"], react_df["score"],
        color=colors_r, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)
for i, (_, row) in enumerate(react_df.iterrows()):
    xpos = 0.1 if row["score"]>0 else -0.1
    ha   = "left" if row["score"]>0 else "right"
    ax.text(xpos, i, row["Overlap"], va="center",
            fontsize=6, color="white", fontweight="bold", ha=ha)
ax.set_xlabel("-log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nReactome 2024 — Pseudobulk",
             fontsize=9, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
ax.tick_params(labelsize=7)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False)
plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome_pseudobulk.png",
            dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome_pseudobulk.pdf",
            bbox_inches="tight")
plt.show()

In [ ]:
res_up_k = enr_up.results[enr_up.results["Adjusted P-value"] < 0.05].head(10).copy()
res_dn_k = enr_dn.results[enr_dn.results["Adjusted P-value"] < 0.05].head(10).copy()

# drop non-biological KEGG terms
exclude = ["Salmonella", "Hepatitis", "Influenza", "Herpes", 
           "Tuberculosis", "Chagas", "Leishmaniasis", "Malaria"]
res_up_k = res_up_k[~res_up_k["Term"].str.contains("|".join(exclude), case=False)]
res_dn_k = res_dn_k[~res_dn_k["Term"].str.contains("|".join(exclude), case=False)]

res_up_k["score"] =  -np.log10(res_up_k["Adjusted P-value"])
res_dn_k["score"] = -(-np.log10(res_dn_k["Adjusted P-value"]))

plot_df = pd.concat([res_up_k, res_dn_k]).sort_values("score")
plot_df["Term"] = plot_df["Term"].str.replace(r"WP\d+", "", regex=True).str.strip()

colors = ["#f0aaf3" if s > 0 else "#824516" for s in plot_df["score"]]

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(plot_df["Term"], plot_df["score"], color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Signed -log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nKEGG Pathways", fontsize=9, loc="right")
ax.tick_params(axis="y", labelsize=7)
ax.tick_params(axis="x", labelsize=7)
ax.spines[["top", "right"]].set_visible(False)
ax.text(0.98, 0.01, "FDR < 0.05", transform=ax.transAxes,
        fontsize=6, color="gray", ha="right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_vs_ND_KEGG.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_vs_ND_KEGG.pdf", bbox_inches="tight")
plt.show()

# save full results
enr_up.results.to_csv("/Users/kmeneses/Desktop/MSL2_T2D_UP_KEGG.csv", index=False)
enr_dn.results.to_csv("/Users/kmeneses/Desktop/MSL2_T2D_DN_KEGG.csv", index=False)

In [ ]:
print("TOP UPREGULATED in T2D:")
print(deg_strict[deg_strict["logfoldchanges"] > 0]
      .sort_values("logfoldchanges", ascending=False)
      [["names", "logfoldchanges", "pvals_adj"]].head(20).to_string())

print("\nTOP DOWNREGULATED in T2D:")
print(deg_strict[deg_strict["logfoldchanges"] < 0]
      .sort_values("logfoldchanges")
      [["names", "logfoldchanges", "pvals_adj"]].head(20).to_string())

print(f"\nTotal strict DEGs: {len(deg_strict)}")
print(f"Up: {(deg_strict['logfoldchanges']>0).sum()}")
print(f"Down: {(deg_strict['logfoldchanges']<0).sum()}")

In [ ]:
# Hallmark only — 50 non-redundant gene sets, minimum overlap filter
enr_up = gp.enrichr(
    gene_list=up_genes,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    outdir=None
)

res_up = enr_up.results.copy()

# filter: adjusted p < 0.05 AND at least 3 genes overlapping
res_up["overlap_n"] = res_up["Overlap"].apply(lambda x: int(x.split("/")[0]))
res_up = res_up[
    (res_up["Adjusted P-value"] < 0.05) &
    (res_up["overlap_n"] >= 3)
].sort_values("Adjusted P-value")

print(f"Significant Hallmark terms: {len(res_up)}")
print(res_up[["Term","Overlap","Adjusted P-value"]].to_string())

In [ ]:
res_up["score"] = -np.log10(res_up["Adjusted P-value"])
res_up["Term"]  = res_up["Term"].str.replace("HALLMARK_", "").str.replace("_", " ").str.title()
plot_df = res_up.sort_values("score")

fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(plot_df["Term"], plot_df["score"], color="#d62728", edgecolor="none", height=0.7)
ax.set_xlabel("-log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nMSigDB Hallmark — Upregulated Pathways", fontsize=9, loc="right")
ax.tick_params(labelsize=7)
ax.spines[["top","right"]].set_visible(False)
ax.axvline(0, color="black", linewidth=0.5)

# add overlap labels
for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.text(0.1, i, row["Overlap"], va="center", fontsize=6, color="white", fontweight="bold")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_Hallmark.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_Hallmark.pdf", bbox_inches="tight")
plt.show()

res_up.to_csv("/Users/kmeneses/Desktop/MSL2_T2D_Hallmark.csv", index=False)
print("✔ Done")

In [ ]:
print(f"Up genes: {len(up_genes)}")
print(f"Down genes: {len(dn_genes)}")

# try enrichr on down genes anyway
enr_dn = gp.enrichr(
    gene_list=dn_genes,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    outdir=None
)

res_dn = enr_dn.results.copy()
res_dn["overlap_n"] = res_dn["Overlap"].apply(lambda x: int(x.split("/")[0]))

print(res_dn[["Term","Overlap","Adjusted P-value"]].head(10).to_string())
print(f"\nSignificant with >=3 overlap: {len(res_dn[(res_dn['Adjusted P-value']<0.05) & (res_dn['overlap_n']>=3)])}")

In [ ]:
deg_strict2 = deg[
    (deg["pvals_adj"] < 0.05) &
    (deg["logfoldchanges"].abs() > 0.25)  # less strict than 0.5
]
dn_genes = deg_strict2[deg_strict2["logfoldchanges"] < 0]["names"].tolist()
print(f"Down genes at LFC>0.25: {len(dn_genes)}")

In [ ]:
deg_relaxed = deg[
    (deg["pvals_adj"] < 0.05) &
    (deg["logfoldchanges"] < -0.25)  # only down, relaxed LFC
]
dn_genes = deg_relaxed["names"].tolist()
print(f"Down genes: {len(dn_genes)}")

enr_dn = gp.enrichr(
    gene_list=dn_genes,
    gene_sets="MSigDB_Hallmark_2020",
    organism="Human",
    outdir=None
)

res_dn = enr_dn.results.copy()
res_dn["overlap_n"] = res_dn["Overlap"].apply(lambda x: int(x.split("/")[0]))
res_dn = res_dn[
    (res_dn["Adjusted P-value"] < 0.05) &
    (res_dn["overlap_n"] >= 3)
].sort_values("Adjusted P-value")

print(f"Significant down terms: {len(res_dn)}")
print(res_dn[["Term","Overlap","Adjusted P-value"]].to_string())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── data ───────────────────────────────────────────────────────────
kegg_up = res_up_k[res_up_k["Adjusted P-value"]<0.05].copy()
kegg_dn = res_dn_k[res_dn_k["Adjusted P-value"]<0.05].copy()

# get reactome and wikipathway results
enr_up_r = gp.enrichr(gene_list=up_genes, gene_sets="Reactome_Pathways_2024",
                       organism="Human", outdir=None)
enr_dn_r = gp.enrichr(gene_list=dn_genes, gene_sets="Reactome_Pathways_2024",
                       organism="Human", outdir=None)
enr_up_w = gp.enrichr(gene_list=up_genes, gene_sets="WikiPathway_2023_Human",
                       organism="Human", outdir=None)
enr_dn_w = gp.enrichr(gene_list=dn_genes, gene_sets="WikiPathway_2023_Human",
                       organism="Human", outdir=None)

react_up = enr_up_r.results[enr_up_r.results["Adjusted P-value"]<0.05].head(10).copy()
react_dn = enr_dn_r.results[enr_dn_r.results["Adjusted P-value"]<0.05].head(10).copy()
wiki_up  = enr_up_w.results[enr_up_w.results["Adjusted P-value"]<0.05].head(8).copy()
wiki_dn  = enr_dn_w.results[enr_dn_w.results["Adjusted P-value"]<0.05].head(8).copy()

# clean term names
def clean_terms(df):
    df = df.copy()
    df["Term"] = (df["Term"]
        .str.replace(r"\(GO:\d+\)","",regex=True)
        .str.replace(r"WP\d+","",regex=True)
        .str.strip())
    df["overlap_n"] = df["Overlap"].apply(lambda x: int(x.split("/")[0]))
    return df

kegg_up  = clean_terms(kegg_up)
react_up = clean_terms(react_up)
react_dn = clean_terms(react_dn)
wiki_up  = clean_terms(wiki_up)
wiki_dn  = clean_terms(wiki_dn)

# ── plot ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(1, 3, wspace=0.5)

# ── KEGG ───────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
kegg_up["score"] = -np.log10(kegg_up["Adjusted P-value"])
kegg_up_s = kegg_up.sort_values("score")
ax1.barh(kegg_up_s["Term"], kegg_up_s["score"],
         color="#d62728", edgecolor="none", height=0.6)
for i, (_, row) in enumerate(kegg_up_s.iterrows()):
    ax1.text(0.1, i, row["Overlap"], va="center",
             fontsize=6, color="white", fontweight="bold")
ax1.set_xlabel("-log10(adj p-value)", fontsize=8)
ax1.set_title("KEGG 2021\nUpregulated in T2D MSL2", fontsize=9, fontweight="bold")
ax1.spines[["top","right"]].set_visible(False)
ax1.tick_params(labelsize=7)

# ── Reactome ───────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
react_up["score"] =  -np.log10(react_up["Adjusted P-value"])
react_dn["score"] = -(-np.log10(react_dn["Adjusted P-value"]))
react_df = pd.concat([react_dn, react_up]).sort_values("score")
colors_r = ["#d62728" if s>0 else "#1f77b4" for s in react_df["score"]]
ax2.barh(react_df["Term"], react_df["score"],
         color=colors_r, edgecolor="none", height=0.6)
ax2.axvline(0, color="black", linewidth=0.8)
for i, (_, row) in enumerate(react_df.iterrows()):
    xpos = 0.1 if row["score"]>0 else -0.1
    ha   = "left" if row["score"]>0 else "right"
    ax2.text(xpos, i, row["Overlap"], va="center",
             fontsize=6, color="white", fontweight="bold", ha=ha)
ax2.set_xlabel("-log10(adj p-value)", fontsize=8)
ax2.set_title("Reactome 2024\nT2D MSL2", fontsize=9, fontweight="bold")
ax2.spines[["top","right"]].set_visible(False)
ax2.tick_params(labelsize=7)

# ── WikiPathway ────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])

# filter noisy wiki terms
exclude_wiki = ["SARS","Glomerulosclerosis","Cardiac"]
wiki_up_f = wiki_up[~wiki_up["Term"].str.contains("|".join(exclude_wiki), case=False)]
wiki_dn_f = wiki_dn[~wiki_dn["Term"].str.contains("|".join(exclude_wiki), case=False)]

wiki_up_f["score"] =  -np.log10(wiki_up_f["Adjusted P-value"])
wiki_dn_f["score"] = -(-np.log10(wiki_dn_f["Adjusted P-value"]))
wiki_df = pd.concat([wiki_dn_f, wiki_up_f]).sort_values("score")
colors_w = ["#d62728" if s>0 else "#1f77b4" for s in wiki_df["score"]]
ax3.barh(wiki_df["Term"], wiki_df["score"],
         color=colors_w, edgecolor="none", height=0.6)
ax3.axvline(0, color="black", linewidth=0.8)
for i, (_, row) in enumerate(wiki_df.iterrows()):
    xpos = 0.1 if row["score"]>0 else -0.1
    ha   = "left" if row["score"]>0 else "right"
    ax3.text(xpos, i, row["Overlap"], va="center",
             fontsize=6, color="white", fontweight="bold", ha=ha)
ax3.set_xlabel("-log10(adj p-value)", fontsize=8)
ax3.set_title("WikiPathway 2023\nT2D MSL2", fontsize=9, fontweight="bold")
ax3.spines[["top","right"]].set_visible(False)
ax3.tick_params(labelsize=7)

# ── shared legend ──────────────────────────────────────────────────
from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=8, frameon=False,
   loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))

fig.suptitle(f"{CLUSTER}: T2D vs Control — Pathway Enrichment",
             fontsize=11, fontweight="bold")

plt.savefig(f"/Users/kmeneses/Desktop/{CLUSTER}_pathways_3db.png",
            dpi=300, bbox_inches="tight")
plt.savefig(f"/Users/kmeneses/Desktop/{CLUSTER}_pathways_3db.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
CLUSTER = "MSL2"

# make sure up_genes/dn_genes are from MSL2 Wilcoxon
adata_sub = mesenchymalf[
    mesenchymalf.obs[CELLTYPE_COL] == CLUSTER
].copy()

sc.tl.rank_genes_groups(
    adata_sub,
    groupby=DISEASE_COL,
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg = sc.get.rank_genes_groups_df(adata_sub, group="T2D").dropna()
deg_strict = deg[
    (deg["pvals_adj"] < 0.05) &
    (deg["logfoldchanges"].abs() > 0.25)
]

up_genes = deg_strict[deg_strict["logfoldchanges"] > 0]["names"].tolist()
dn_genes = deg_strict[deg_strict["logfoldchanges"] < 0]["names"].tolist()

print(f"MSL2 — Up: {len(up_genes)}, Down: {len(dn_genes)}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from matplotlib.patches import Patch

def shorten(s, n=40):
    return s[:n]+"..." if len(s)>n else s

def prep(r_up, r_dn, n=8, exclude=None):
    ru = r_up.copy(); rd = r_dn.copy()
    if exclude:
        ru = ru[~ru["Term"].str.contains("|".join(exclude), case=False)]
        rd = rd[~rd["Term"].str.contains("|".join(exclude), case=False)]
    ru = ru.head(n); rd = rd.head(n)
    ru["Term"] = ru["Term"].apply(shorten)
    rd["Term"] = rd["Term"].apply(shorten)
    ru["score"] =  -np.log10(ru["Adjusted P-value"])
    rd["score"] = -(-np.log10(rd["Adjusted P-value"]))
    df = pd.concat([rd, ru]).sort_values("score")
    return df

def plot_panel(ax, df, title, show_overlap=True):
    colors = ["#d62728" if s>0 else "#1f77b4" for s in df["score"]]
    ax.barh(df["Term"], df["score"], color=colors, edgecolor="none", height=0.7)
    ax.axvline(0, color="black", linewidth=0.8)
    if show_overlap and "Overlap" in df.columns:
        for i, (_, row) in enumerate(df.iterrows()):
            xpos = 0.1 if row["score"]>0 else -0.1
            ha   = "left" if row["score"]>0 else "right"
            ax.text(xpos, i, row["Overlap"], va="center",
                    fontsize=5.5, color="white", fontweight="bold", ha=ha)
    ax.set_title(title, fontsize=8, fontweight="bold", pad=6)
    ax.set_xlabel("-log10(adj p-value)", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.spines[["top","right"]].set_visible(False)
    ax.axvline(0, color="black", linewidth=0.6)

# ── fetch all databases ────────────────────────────────────────────
print("Fetching gene sets...")

# MSigDB Hallmark
e_h_up = gp.enrichr(gene_list=up_genes, gene_sets="MSigDB_Hallmark_2020",
                     organism="Human", outdir=None)
e_h_dn = gp.enrichr(gene_list=dn_genes, gene_sets="MSigDB_Hallmark_2020",
                     organism="Human", outdir=None)
h_up = e_h_up.results[e_h_up.results["Adjusted P-value"]<0.05].copy()
h_dn = e_h_dn.results[e_h_dn.results["Adjusted P-value"]<0.05].copy()
h_up["overlap_n"] = h_up["Overlap"].apply(lambda x: int(x.split("/")[0]))
h_dn["overlap_n"] = h_dn["Overlap"].apply(lambda x: int(x.split("/")[0]))
h_up = h_up[h_up["overlap_n"]>=3]
h_dn = h_dn[h_dn["overlap_n"]>=3]

# KEGG
e_k_up = gp.enrichr(gene_list=up_genes, gene_sets="KEGG_2021_Human",
                     organism="Human", outdir=None)
e_k_dn = gp.enrichr(gene_list=dn_genes, gene_sets="KEGG_2021_Human",
                     organism="Human", outdir=None)
k_up = e_k_up.results[e_k_up.results["Adjusted P-value"]<0.05].copy()
k_dn = e_k_dn.results[e_k_dn.results["Adjusted P-value"]<0.05].copy()

# Reactome
e_r_up = gp.enrichr(gene_list=up_genes, gene_sets="Reactome_Pathways_2024",
                     organism="Human", outdir=None)
e_r_dn = gp.enrichr(gene_list=dn_genes, gene_sets="Reactome_Pathways_2024",
                     organism="Human", outdir=None)
r_up = e_r_up.results[e_r_up.results["Adjusted P-value"]<0.05].copy()
r_dn = e_r_dn.results[e_r_dn.results["Adjusted P-value"]<0.05].copy()

# WikiPathway
e_w_up = gp.enrichr(gene_list=up_genes, gene_sets="WikiPathway_2023_Human",
                     organism="Human", outdir=None)
e_w_dn = gp.enrichr(gene_list=dn_genes, gene_sets="WikiPathway_2023_Human",
                     organism="Human", outdir=None)
w_up = e_w_up.results[e_w_up.results["Adjusted P-value"]<0.05].copy()
w_dn = e_w_dn.results[e_w_dn.results["Adjusted P-value"]<0.05].copy()

print("All databases fetched ✔")

# ── build plot dataframes ──────────────────────────────────────────
noise = ["SARS","Glomerulosclerosis","Prostate","Salmonella",
         "Hepatitis","Chagas","Leishmaniasis","Malaria","Herpes"]

df_h = prep(h_up, h_dn, n=8)
df_k = prep(k_up, k_dn, n=6, exclude=noise)
df_r = prep(r_up, r_dn, n=8, exclude=["Cobalamin","Dietary","L13a",
                                        "Ribosomal Scanning"])
df_w = prep(w_up, w_dn, n=6, exclude=noise)

# ── figure ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 10))
gs  = gridspec.GridSpec(1, 4, wspace=0.55,
                        left=0.06, right=0.97,
                        top=0.88, bottom=0.12)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])
ax3 = fig.add_subplot(gs[2])
ax4 = fig.add_subplot(gs[3])

plot_panel(ax1, df_h, "MSigDB Hallmark")
plot_panel(ax2, df_k, "KEGG 2021")
plot_panel(ax3, df_r, "Reactome 2024")
plot_panel(ax4, df_w, "WikiPathway 2023")

fig.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=8, frameon=False,
   loc="lower center", ncol=2, bbox_to_anchor=(0.5, 0.01))

fig.suptitle(f"{CLUSTER}: T2D vs Control — Pathway Enrichment\n"
             f"MSigDB Hallmark · KEGG · Reactome · WikiPathway",
             fontsize=11, fontweight="bold", y=0.97)

plt.savefig(f"/Users/kmeneses/Desktop/{CLUSTER}_pathways_4db.png",
            dpi=300, bbox_inches="tight")
plt.savefig(f"/Users/kmeneses/Desktop/{CLUSTER}_pathways_4db.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import scanpy as sc
from matplotlib.patches import Patch

def shorten(s, n=40):
    return s[:n]+"..." if len(s)>n else s

def get_genes(adata, cluster, celltype_col, disease_col):
    sub = adata[adata.obs[celltype_col]==cluster].copy()
    print(f"\n{cluster}: {sub.obs[disease_col].value_counts().to_dict()}")
    sc.tl.rank_genes_groups(sub, groupby=disease_col,
                            groups=["T2D"], reference="Control",
                            method="wilcoxon", use_raw=False)
    deg = sc.get.rank_genes_groups_df(sub, group="T2D").dropna()
    strict = deg[(deg["pvals_adj"]<0.05)&(deg["logfoldchanges"].abs()>0.25)]
    up = strict[strict["logfoldchanges"]>0]["names"].tolist()
    dn = strict[strict["logfoldchanges"]<0]["names"].tolist()
    print(f"  Up: {len(up)}, Down: {len(dn)}")
    return up, dn

def fetch_enrichr(up, dn, gene_set):
    eu = gp.enrichr(gene_list=up, gene_sets=gene_set,
                    organism="Human", outdir=None)
    ed = gp.enrichr(gene_list=dn, gene_sets=gene_set,
                    organism="Human", outdir=None)
    ru = eu.results[eu.results["Adjusted P-value"]<0.05].copy()
    rd = ed.results[ed.results["Adjusted P-value"]<0.05].copy()
    return ru, rd

def prep(r_up, r_dn, n=8, exclude=None):
    ru = r_up.copy(); rd = r_dn.copy()
    if exclude:
        ru = ru[~ru["Term"].str.contains("|".join(exclude), case=False, regex=True)]
        rd = rd[~rd["Term"].str.contains("|".join(exclude), case=False, regex=True)]
    ru = ru.head(n); rd = rd.head(n)
    ru["Term"] = ru["Term"].apply(shorten)
    rd["Term"] = rd["Term"].apply(shorten)
    ru["score"] =  -np.log10(ru["Adjusted P-value"])
    rd["score"] = -(-np.log10(rd["Adjusted P-value"]))
    return pd.concat([rd, ru]).sort_values("score")

def plot_panel(ax, df, title):
    colors = ["#d62728" if s>0 else "#1f77b4" for s in df["score"]]
    ax.barh(df["Term"], df["score"], color=colors, edgecolor="none", height=0.7)
    ax.axvline(0, color="black", linewidth=0.8)
    if "Overlap" in df.columns:
        for i, (_, row) in enumerate(df.iterrows()):
            xpos = 0.1 if row["score"]>0 else -0.1
            ha   = "left" if row["score"]>0 else "right"
            ax.text(xpos, i, row["Overlap"], va="center",
                    fontsize=5.5, color="white", fontweight="bold", ha=ha)
    ax.set_title(title, fontsize=8, fontweight="bold", pad=6)
    ax.set_xlabel("-log10(adj p-value)", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.spines[["top","right"]].set_visible(False)

noise = ["SARS","Glomerulosclerosis","Prostate","Salmonella",
         "Hepatitis","Chagas","Cobalamin","Dietary","L13a","Ribosomal Scanning"]

# ── get genes for both clusters ────────────────────────────────────
up_msl1, dn_msl1 = get_genes(mesenchymalf, "MSL1", CELLTYPE_COL, DISEASE_COL)
up_msl2, dn_msl2 = get_genes(mesenchymalf, "MSL2", CELLTYPE_COL, DISEASE_COL)

# ── fetch enrichment ───────────────────────────────────────────────
print("\nFetching MSigDB Hallmark...")
h_up1, h_dn1 = fetch_enrichr(up_msl1, dn_msl1, "MSigDB_Hallmark_2020")
h_up2, h_dn2 = fetch_enrichr(up_msl2, dn_msl2, "MSigDB_Hallmark_2020")

# filter hallmark by overlap
for df in [h_up1, h_dn1, h_up2, h_dn2]:
    df["overlap_n"] = df["Overlap"].apply(lambda x: int(x.split("/")[0]))
h_up1 = h_up1[h_up1["overlap_n"]>=3]
h_dn1 = h_dn1[h_dn1["overlap_n"]>=3]
h_up2 = h_up2[h_up2["overlap_n"]>=3]
h_dn2 = h_dn2[h_dn2["overlap_n"]>=3]

print("Fetching Reactome...")
r_up1, r_dn1 = fetch_enrichr(up_msl1, dn_msl1, "Reactome_Pathways_2024")
r_up2, r_dn2 = fetch_enrichr(up_msl2, dn_msl2, "Reactome_Pathways_2024")

# ── build plot dfs ─────────────────────────────────────────────────
df_h1 = prep(h_up1, h_dn1, n=8)
df_h2 = prep(h_up2, h_dn2, n=8)
df_r1 = prep(r_up1, r_dn1, n=8, exclude=noise)
df_r2 = prep(r_up2, r_dn2, n=8, exclude=noise)

# ── figure: 2 rows x 2 cols ────────────────────────────────────────
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, wspace=0.5, hspace=0.45,
                        left=0.1, right=0.97,
                        top=0.92, bottom=0.08)

ax_h1 = fig.add_subplot(gs[0, 0])
ax_h2 = fig.add_subplot(gs[0, 1])
ax_r1 = fig.add_subplot(gs[1, 0])
ax_r2 = fig.add_subplot(gs[1, 1])

plot_panel(ax_h1, df_h1, "MSL1 — MSigDB Hallmark")
plot_panel(ax_h2, df_h2, "MSL2 — MSigDB Hallmark")
plot_panel(ax_r1, df_r1, "MSL1 — Reactome 2024")
plot_panel(ax_r2, df_r2, "MSL2 — Reactome 2024")

# column labels
fig.text(0.30, 0.94, "MSL1: T2D vs Control",
         ha="center", fontsize=11, fontweight="bold")
fig.text(0.73, 0.94, "MSL2: T2D vs Control",
         ha="center", fontsize=11, fontweight="bold")

fig.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=9, frameon=False,
   loc="lower center", ncol=2, bbox_to_anchor=(0.5, 0.01))

fig.suptitle("Pathway Enrichment: MSL1 vs MSL2 — T2D vs Control",
             fontsize=12, fontweight="bold", y=0.99)

plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_Hallmark_Reactome.png",
            dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_Hallmark_Reactome.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
UP_COLOR   = "#c45b8a"   # muted pink/mauve
DOWN_COLOR = "#8b5e3c"   # warm brown

def plot_panel(ax, df, title):
    colors = [UP_COLOR if s>0 else DOWN_COLOR for s in df["score"]]
    ax.barh(df["Term"], df["score"], color=colors, edgecolor="none", height=0.7)
    ax.axvline(0, color="black", linewidth=0.8)
    if "Overlap" in df.columns:
        for i, (_, row) in enumerate(df.iterrows()):
            xpos = 0.1 if row["score"]>0 else -0.1
            ha   = "left" if row["score"]>0 else "right"
            ax.text(xpos, i, row["Overlap"], va="center",
                    fontsize=5.5, color="white", fontweight="bold", ha=ha)
    ax.set_title(title, fontsize=8, fontweight="bold", pad=6)
    ax.set_xlabel("-log10(adj p-value)", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.spines[["top","right"]].set_visible(False)

fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, wspace=0.5, hspace=0.45,
                        left=0.1, right=0.97,
                        top=0.92, bottom=0.08)

ax_h1 = fig.add_subplot(gs[0, 0])
ax_h2 = fig.add_subplot(gs[0, 1])
ax_r1 = fig.add_subplot(gs[1, 0])
ax_r2 = fig.add_subplot(gs[1, 1])

plot_panel(ax_h1, df_h1, "MSL1 — MSigDB Hallmark")
plot_panel(ax_h2, df_h2, "MSL2 — MSigDB Hallmark")
plot_panel(ax_r1, df_r1, "MSL1 — Reactome 2024")
plot_panel(ax_r2, df_r2, "MSL2 — Reactome 2024")

fig.text(0.30, 0.94, "MSL1: T2D vs Control",
         ha="center", fontsize=11, fontweight="bold")
fig.text(0.73, 0.94, "MSL2: T2D vs Control",
         ha="center", fontsize=11, fontweight="bold")

fig.legend(handles=[
    Patch(facecolor=UP_COLOR,   label="Up in T2D"),
    Patch(facecolor=DOWN_COLOR, label="Down in T2D")
], fontsize=9, frameon=False,
   loc="lower center", ncol=2, bbox_to_anchor=(0.5, 0.01))

fig.suptitle("Pathway Enrichment: MSL1 vs MSL2 — T2D vs Control",
             fontsize=12, fontweight="bold", y=0.99)

plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_Hallmark_Reactome_v2.png",
            dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_Hallmark_Reactome_v2.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
UP_COLOR   = "#c45b8a"
DOWN_COLOR = "#8b5e3c"

def plot_panel(ax, df, title):
    colors = [UP_COLOR if s>0 else DOWN_COLOR for s in df["score"]]
    ax.barh(df["Term"], df["score"], color=colors, edgecolor="none", height=0.6)
    ax.axvline(0, color="black", linewidth=0.7)
    if "Overlap" in df.columns:
        for i, (_, row) in enumerate(df.iterrows()):
            xpos = 0.05 if row["score"]>0 else -0.05
            ha   = "left" if row["score"]>0 else "right"
            ax.text(xpos, i, row["Overlap"], va="center",
                    fontsize=4.5, color="white", fontweight="bold", ha=ha)
    ax.set_title(title, fontsize=7, fontweight="bold", pad=4)
    ax.set_xlabel("-log10(adj p-value)", fontsize=6)
    ax.tick_params(labelsize=5.5)
    ax.spines[["top","right"]].set_visible(False)

# ── reduce to top 6 per panel to keep compact ─────────────────────
df_h1_s = df_h1.iloc[list(range(min(6,len(df_h1)//2)))+
                      list(range(max(0,len(df_h1)-6), len(df_h1)))].sort_values("score")
df_h2_s = df_h2.iloc[list(range(min(6,len(df_h2)//2)))+
                      list(range(max(0,len(df_h2)-6), len(df_h2)))].sort_values("score")
df_r1_s = df_r1.iloc[list(range(min(6,len(df_r1)//2)))+
                      list(range(max(0,len(df_r1)-6), len(df_r1)))].sort_values("score")
df_r2_s = df_r2.iloc[list(range(min(6,len(df_r2)//2)))+
                      list(range(max(0,len(df_r2)-6), len(df_r2)))].sort_values("score")

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5),
                         gridspec_kw={"wspace": 0.2})

plot_panel(axes[0], df_h1_s, "MSL1\nMSigDB Hallmark")
plot_panel(axes[1], df_h2_s, "MSL2\nMSigDB Hallmark")
plot_panel(axes[2], df_r1_s, "MSL1\nReactome 2024")
plot_panel(axes[3], df_r2_s, "MSL2\nReactome 2024")

# vertical divider between MSL1 and MSL2
fig.add_artist(plt.Line2D([0.5, 0.5], [0.05, 0.93],
               transform=fig.transFigure,
               color="lightgrey", linewidth=1, linestyle="--"))

# column headers
fig.text(0.27, 0.97, "MSL1: T2D vs Control",
         ha="center", fontsize=9, fontweight="bold")
fig.text(0.73, 0.97, "MSL2: T2D vs Control",
         ha="center", fontsize=9, fontweight="bold")

fig.legend(handles=[
    Patch(facecolor=UP_COLOR,   label="Up in T2D"),
    Patch(facecolor=DOWN_COLOR, label="Down in T2D")
], fontsize=7, frameon=False,
   loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.05))

fig.suptitle("Pathway Enrichment: MSL1 vs MSL2 — T2D vs Control",
             fontsize=10, fontweight="bold", y=1.02)

plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_1row.png",
            dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_1row.svg",
            dpi=600,
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
UP_COLOR   = "#c45b8a"
DOWN_COLOR = "#8b5e3c"

def prep_combined(h_up, h_dn, r_up, r_dn, n=6, exclude=None):
    """Combine top hits from Hallmark + Reactome into one df"""
    noise = exclude or []
    
    # top n from each db
    hu = h_up[h_up["overlap_n"]>=3].head(n).copy()
    hd = h_dn[h_dn["overlap_n"]>=3].head(n).copy() if "overlap_n" in h_dn else h_dn.head(n).copy()
    ru = r_up[~r_up["Term"].str.contains("|".join(noise), case=False)].head(n).copy() if noise else r_up.head(n).copy()
    rd = r_dn[~r_dn["Term"].str.contains("|".join(noise), case=False)].head(n).copy() if noise else r_dn.head(n).copy()

    # tag source
    for df, tag in [(hu,"Hallmark"),(hd,"Hallmark"),(ru,"Reactome"),(rd,"Reactome")]:
        df["source"] = tag

    hu["score"] =  -np.log10(hu["Adjusted P-value"])
    hd["score"] = -(-np.log10(hd["Adjusted P-value"]))
    ru["score"] =  -np.log10(ru["Adjusted P-value"])
    rd["score"] = -(-np.log10(rd["Adjusted P-value"]))

    df = pd.concat([hd, rd, hu, ru]).sort_values("score")
    df["Term"] = df["Term"].apply(lambda s: s[:42]+"..." if len(s)>42 else s)
    df = df.drop_duplicates(subset="Term")
    return df

def plot_combined(ax, df, title):
    colors = [UP_COLOR if s>0 else DOWN_COLOR for s in df["score"]]
    bars = ax.barh(df["Term"], df["score"],
                   color=colors, edgecolor="none", height=0.7)

    # hatching for Reactome
    for bar, (_, row) in zip(bars, df.iterrows()):
        if row["source"] == "Reactome":
            bar.set_hatch("///")
            bar.set_edgecolor("white")
            bar.set_linewidth(0.3)

    ax.axvline(0, color="black", linewidth=0.8)

    if "Overlap" in df.columns:
        for i, (_, row) in enumerate(df.iterrows()):
            xpos = 0.1 if row["score"]>0 else -0.1
            ha   = "left" if row["score"]>0 else "right"
            ax.text(xpos, i, row["Overlap"], va="center",
                    fontsize=5, color="white", fontweight="bold", ha=ha)

    ax.set_title(title, fontsize=9, fontweight="bold", pad=6)
    ax.set_xlabel("-log10(adj p-value)", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.spines[["top","right"]].set_visible(False)

# ── build combined dfs ─────────────────────────────────────────────
noise = ["SARS","Cobalamin","Dietary","L13a","Ribosomal Scanning",
         "Prostate","Salmonella","Hepatitis"]

# add overlap_n to hallmark dfs if missing
for df in [h_up1, h_dn1, h_up2, h_dn2]:
    if "overlap_n" not in df.columns:
        df["overlap_n"] = df["Overlap"].apply(lambda x: int(x.split("/")[0]))

df_msl1 = prep_combined(h_up1, h_dn1, r_up1, r_dn1, n=6, exclude=noise)
df_msl2 = prep_combined(h_up2, h_dn2, r_up2, r_dn2, n=6, exclude=noise)

# ── figure: 1 row, 2 panels ────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7),
                                gridspec_kw={"wspace": 0.55})

plot_combined(ax1, df_msl1, "MSL1: T2D vs Control")
plot_combined(ax2, df_msl2, "MSL2: T2D vs Control")

# divider
fig.add_artist(plt.Line2D([0.5, 0.5], [0.05, 0.95],
               transform=fig.transFigure,
               color="lightgrey", linewidth=1, linestyle="--"))

# legend — colors
from matplotlib.patches import Patch
from matplotlib.patches import Patch as MPatch
import matplotlib.patches as mpatches

color_legend = [
    Patch(facecolor=UP_COLOR,   label="Up in T2D"),
    Patch(facecolor=DOWN_COLOR, label="Down in T2D"),
]

# legend — hatching for source
hatch_legend = [
    mpatches.Patch(facecolor="grey", label="MSigDB Hallmark"),
    mpatches.Patch(facecolor="grey", hatch="///",
                   edgecolor="white", label="Reactome 2024"),
]

leg1 = fig.legend(handles=color_legend, fontsize=8, frameon=False,
                  loc="lower left", bbox_to_anchor=(0.02, -0.04), ncol=2)
fig.legend(handles=hatch_legend, fontsize=8, frameon=False,
           loc="lower right", bbox_to_anchor=(0.98, -0.04), ncol=2)
fig.add_artist(leg1)

fig.suptitle("Pathway Enrichment: MSL1 vs MSL2 — T2D vs Control\n"
             "MSigDB Hallmark (solid) · Reactome 2024 (hatched)",
             fontsize=10, fontweight="bold")

plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.png",
            dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
def plot_combined(ax, df, title):
    colors = [UP_COLOR if s>0 else DOWN_COLOR for s in df["score"]]
    bars = ax.barh(df["Term"], df["score"],
                   color=colors, edgecolor="none", height=0.7)

    # hatching for Reactome
    for bar, (_, row) in zip(bars, df.iterrows()):
        if row["source"] == "Reactome":
            bar.set_hatch("///")
            bar.set_edgecolor("white")
            bar.set_linewidth(0.3)

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=9, fontweight="bold", pad=6)
    ax.set_xlabel("-log10(adj p-value)", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.spines[["top","right"]].set_visible(False)

# rest of code unchanged — rerun from df_msl1 = prep_combined(...) onwards
df_msl1 = prep_combined(h_up1, h_dn1, r_up1, r_dn1, n=6, exclude=noise)
df_msl2 = prep_combined(h_up2, h_dn2, r_up2, r_dn2, n=6, exclude=noise)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 2),
                                gridspec_kw={"wspace": 0.25})

plot_combined(ax1, df_msl1, "MSL1: T2D vs Control")
plot_combined(ax2, df_msl2, "MSL2: T2D vs Control")

fig.add_artist(plt.Line2D([0.5, 0.5], [0.05, 0.95],
               transform=fig.transFigure,
               color="lightgrey", linewidth=1, linestyle="--"))

import matplotlib.patches as mpatches
color_legend = [
    Patch(facecolor=UP_COLOR,   label="Up in T2D"),
    Patch(facecolor=DOWN_COLOR, label="Down in T2D"),
]
hatch_legend = [
    mpatches.Patch(facecolor="grey", label="MSigDB Hallmark"),
    mpatches.Patch(facecolor="grey", hatch="///",
                   edgecolor="white", label="Reactome 2024"),
]
leg1 = fig.legend(handles=color_legend, fontsize=8, frameon=False,
                  loc="lower left", bbox_to_anchor=(0.02, -0.04), ncol=2)
fig.legend(handles=hatch_legend, fontsize=8, frameon=False,
           loc="lower right", bbox_to_anchor=(0.98, -0.04), ncol=2)
fig.add_artist(leg1)

fig.suptitle("Pathway Enrichment: MSL1 vs MSL2 — T2D vs Control\n"
             "MSigDB Hallmark (solid) · Reactome 2024 (hatched)",
             fontsize=10, fontweight="bold")

plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.png",
            dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.svg",
            dpi=600,
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
def plot_combined(ax, df, title):
    colors = [UP_COLOR if s>0 else DOWN_COLOR for s in df["score"]]
    bars = ax.barh(df["Term"], df["score"],
                   color=colors, edgecolor="none", height=0.7)

    # hatching for Reactome — denser + visible at small size
    for bar, (_, row) in zip(bars, df.iterrows()):
        if row["source"] == "Reactome":
            bar.set_hatch("////")        # denser hatch
            bar.set_edgecolor("white")
            bar.set_linewidth(0.8)       # thicker lines
            bar.set_alpha(0.85)

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=7, fontweight="bold", pad=4)
    ax.set_xlabel("-log10(adj p-value)", fontsize=6)
    ax.tick_params(labelsize=5.5)
    ax.spines[["top","right"]].set_visible(False)

df_msl1 = prep_combined(h_up1, h_dn1, r_up1, r_dn1, n=6, exclude=noise)
df_msl2 = prep_combined(h_up2, h_dn2, r_up2, r_dn2, n=6, exclude=noise)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6,2),  # wider so hatch shows
                                gridspec_kw={"wspace": 0.15})

plot_combined(ax1, df_msl1, "MSL1: T2D vs Control")
plot_combined(ax2, df_msl2, "MSL2: T2D vs Control")

fig.add_artist(plt.Line2D([0.5, 0.5], [0.05, 0.95],
               transform=fig.transFigure,
               color="lightgrey", linewidth=1, linestyle="--"))

import matplotlib.patches as mpatches
color_legend = [
    Patch(facecolor=UP_COLOR,   label="Up in T2D"),
    Patch(facecolor=DOWN_COLOR, label="Down in T2D"),
]
hatch_legend = [
    mpatches.Patch(facecolor="grey", edgecolor="white",
                   label="MSigDB Hallmark"),
    mpatches.Patch(facecolor="grey", hatch="////",
                   edgecolor="white", label="Reactome 2024"),
]
leg1 = fig.legend(handles=color_legend, fontsize=7, frameon=False,
                  loc="lower left", bbox_to_anchor=(0.02, -0.05), ncol=2)
fig.legend(handles=hatch_legend, fontsize=7, frameon=False,
           loc="lower right", bbox_to_anchor=(0.98, -0.05), ncol=2)
fig.add_artist(leg1)

fig.suptitle("Pathway Enrichment: MSL1 vs MSL2 — T2D vs Control\n"
             "MSigDB Hallmark (solid) · Reactome 2024 (hatched)",
             fontsize=9, fontweight="bold")

plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.png",
            dpi=600, bbox_inches="tight")   # high dpi preserves hatch
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.pdf",
            bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.svg",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
def plot_combined(ax, df, title):
    colors = [UP_COLOR if s>0 else DOWN_COLOR for s in df["score"]]
    bars = ax.barh(df["Term"], df["score"],
                   color=colors, edgecolor="none", height=0.7)

    # hatching for Reactome — denser + visible at small size
    for bar, (_, row) in zip(bars, df.iterrows()):
        if row["source"] == "Reactome":      # denser hatch
            bar.set_edgecolor("white")
            bar.set_linewidth(0.8)       # thicker lines
            bar.set_alpha(0.85)

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=7, fontweight="bold", pad=4)
    ax.set_xlabel("-log10(adj p-value)", fontsize=6)
    ax.tick_params(labelsize=5.5)
    ax.spines[["top","right"]].set_visible(False)

df_msl1 = prep_combined(h_up1, h_dn1, r_up1, r_dn1, n=6, exclude=noise)
df_msl2 = prep_combined(h_up2, h_dn2, r_up2, r_dn2, n=6, exclude=noise)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5),  # wider so hatch shows
                                gridspec_kw={"wspace": 0.45})

plot_combined(ax1, df_msl1, "MSL1: T2D vs Control")
plot_combined(ax2, df_msl2, "MSL2: T2D vs Control")

fig.add_artist(plt.Line2D([0.5, 0.5], [0.05, 0.95],
               transform=fig.transFigure,
               color="lightgrey", linewidth=1, linestyle="--"))

import matplotlib.patches as mpatches
color_legend = [
    Patch(facecolor=UP_COLOR,   label="Up in T2D"),
    Patch(facecolor=DOWN_COLOR, label="Down in T2D"),
]
hatch_legend = [
    mpatches.Patch(facecolor="grey", edgecolor="white",
                   label="MSigDB Hallmark"),
    mpatches.Patch(facecolor="grey", 
                   edgecolor="white", label="Reactome 2024"),
]
leg1 = fig.legend(handles=color_legend, fontsize=7, frameon=False,
                  loc="lower left", bbox_to_anchor=(0.02, -0.05), ncol=2)
fig.legend(handles=hatch_legend, fontsize=7, frameon=False,
           loc="lower right", bbox_to_anchor=(0.98, -0.05), ncol=2)
fig.add_artist(leg1)

fig.suptitle("Pathway Enrichment: MSL1 vs MSL2 — T2D vs Control\n"
             "MSigDB Hallmark (solid) · Reactome 2024 (hatched)",
             fontsize=9, fontweight="bold")

plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.png",
            dpi=600, bbox_inches="tight")   # high dpi preserves hatch
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.pdf",
            bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_MSL2_combined.svg",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
res_up["score"] =  -np.log10(res_up["Adjusted P-value"])
res_dn["score"] = -(-np.log10(res_dn["Adjusted P-value"]))

plot_df = pd.concat([res_up, res_dn]).sort_values("score")
plot_df["Term"] = plot_df["Term"].str.strip()

colors = ["#d62728" if s > 0 else "#1f77b4" for s in plot_df["score"]]

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(plot_df["Term"], plot_df["score"],
        color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("-log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nMSigDB Hallmark", fontsize=9, loc="right")
ax.tick_params(labelsize=7)
ax.spines[["top", "right"]].set_visible(False)

# overlap labels
for i, (_, row) in enumerate(plot_df.iterrows()):
    xpos = 0.1 if row["score"] > 0 else -0.1
    ha   = "left" if row["score"] > 0 else "right"
    ax.text(xpos, i, row["Overlap"], va="center",
            fontsize=6, color="white", fontweight="bold", ha=ha)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_Hallmark_both.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_T2D_Hallmark_both.pdf", bbox_inches="tight")
plt.show()

In [ ]:
print(enr_dn.results[enr_dn.results["Adjusted P-value"] < 0.05][["Term","Adjusted P-value"]].to_string())

In [ ]:
# manual pathway activity scores using scanpy
# score each Hallmark gene set directly on your cells
import scanpy as sc

# parse hallmark into dict of gene lists
gene_sets = {}
for line in r.text.strip().split("\n"):
    parts = line.split("\t")
    term  = parts[0]
    genes = [g for g in parts[2:] if g]
    if genes:
        gene_sets[term] = genes

# score each pathway per cell
for term, genes in gene_sets.items():
    sc.tl.score_genes(
        adata_sub,
        gene_list=genes,
        score_name=term[:40],  # truncate name
        use_raw=False
    )

# compare scores between T2D and Control
import pandas as pd
score_cols = list(gene_sets.keys())
score_cols = [s[:40] for s in score_cols]

scores = adata_sub.obs[score_cols + [DISEASE_COL]]
mean_scores = scores.groupby(DISEASE_COL)[score_cols].mean()
diff = (mean_scores.loc["T2D"] - mean_scores.loc["Control"]).sort_values(ascending=False)

print("Top activated in T2D:")
print(diff.head(10))
print("\nTop suppressed:")
print(diff.tail(10))

# plot
fig, ax = plt.subplots(figsize=(6, 8))
colors = ["#d62728" if v > 0 else "#1f77b4" for v in diff]
ax.barh(diff.index, diff.values, color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean score (T2D - Control)", fontsize=8)
ax.set_title("MSL2: Hallmark Pathway Activity", fontsize=9, loc="right")
ax.tick_params(labelsize=7)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_scores.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_scores.pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))

colors = ["#d62728" if v > 0 else "#1f77b4" for v in diff]
ax.barh(diff.index, diff.values, color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean pathway score (T2D - Control)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nMSigDB Hallmark Activity", fontsize=9, loc="right")
ax.tick_params(labelsize=7)
ax.spines[["top", "right"]].set_visible(False)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_scores.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_scores.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# show only top/bottom 15 to reduce clutter
top15 = diff.head(15)
bot15 = diff.tail(15)
plot_df = pd.concat([bot15, top15]).sort_values()

colors = ["#d62728" if v > 0 else "#1f77b4" for v in plot_df]

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(plot_df.index, plot_df.values, color=colors, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean pathway score (T2D - Control)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nMSigDB Hallmark Activity", fontsize=9, loc="right")
ax.tick_params(labelsize=7)
ax.spines[["top", "right"]].set_visible(False)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_top15.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_top15.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Matrisome categories from Naba et al. 2012 — curated ECM gene sets
matrisome = {
    "ECM_Glycoproteins": [
        "FN1","LAMB1","LAMB2","LAMC1","LAMA1","LAMA2","LAMA3","LAMA4","LAMA5",
        "TNC","TNN","FBN1","FBN2","LTBP1","LTBP2","LTBP3","LTBP4",
        "COMP","THBS1","THBS2","THBS3","THBS4","TGFBI","POSTN","PERIOSTIN",
        "VCAN","ACAN","HAPLN1","MATN1","MATN2","MATN3","NID1","NID2",
        "AGRN","HSPG2","EMILIN1","EMILIN2","MMRN1","MMRN2"
    ],
    "Collagens": [
        "COL1A1","COL1A2","COL2A1","COL3A1","COL4A1","COL4A2","COL4A3",
        "COL4A4","COL4A5","COL4A6","COL5A1","COL5A2","COL5A3","COL6A1",
        "COL6A2","COL6A3","COL7A1","COL8A1","COL8A2","COL9A1","COL9A2",
        "COL9A3","COL10A1","COL11A1","COL11A2","COL12A1","COL14A1",
        "COL15A1","COL16A1","COL17A1","COL18A1","COL19A1","COL20A1",
        "COL21A1","COL22A1","COL23A1","COL24A1","COL25A1","COL27A1","COL28A1"
    ],
    "Proteoglycans": [
        "VCAN","ACAN","HSPG2","SDC1","SDC2","SDC3","SDC4","GPC1","GPC3",
        "GPC4","GPC6","DCN","BGN","FMOD","LUM","PRELP","OGN","OMD",
        "EPYC","CHAD","ASPN","ECM2","TNFRSF11B","HAPLN1","HAPLN3","HAPLN4"
    ],
    "ECM_Regulators": [
        "MMP1","MMP2","MMP3","MMP7","MMP8","MMP9","MMP10","MMP11","MMP12",
        "MMP13","MMP14","MMP15","MMP16","MMP17","MMP19","MMP20","MMP21",
        "MMP23B","MMP24","MMP25","MMP26","MMP27","MMP28",
        "TIMP1","TIMP2","TIMP3","TIMP4",
        "ADAM10","ADAM12","ADAM15","ADAM17","ADAM19","ADAM28",
        "ADAMTS1","ADAMTS2","ADAMTS3","ADAMTS4","ADAMTS5","ADAMTS8",
        "ADAMTS9","ADAMTS12","ADAMTS13","ADAMTS15","ADAMTS16"
    ],
    "ECM_Affiliated": [
        "LGALS1","LGALS3","LGALS7","LGALS8","LGALS9","LGALS12",
        "CD44","CD47","ITGA1","ITGA2","ITGA3","ITGA4","ITGA5","ITGA6",
        "ITGA7","ITGA8","ITGA9","ITGA10","ITGA11","ITGAV","ITGB1","ITGB2",
        "ITGB3","ITGB4","ITGB5","ITGB6","ITGB7","ITGB8"
    ],
    "Basement_Membrane": [
        "COL4A1","COL4A2","COL4A3","COL4A4","COL4A5","COL4A6",
        "LAMA1","LAMA2","LAMA3","LAMA4","LAMA5",
        "LAMB1","LAMB2","LAMB3","LAMC1","LAMC2","LAMC3",
        "NID1","NID2","HSPG2","AGRN","FBLN1","FBLN2","FBLN5",
        "LAMA5","ELN","EMILIN1"
    ],
    "Secreted_Factors": [
        "TGFB1","TGFB2","TGFB3","CTGF","CCN1","CCN2","CCN3","CCN4","CCN5",
        "VEGFA","VEGFB","VEGFC","FGF1","FGF2","FGF7","FGF10",
        "PDGFA","PDGFB","PDGFC","PDGFD","IGF1","IGF2",
        "WNT2","WNT4","WNT5A","WNT5B","WNT11","BMP2","BMP4","BMP7"
    ]
}

print(f"ECM gene sets: {len(matrisome)}")

# ── score each ECM category per cell ──────────────────────────────
import scanpy as sc

for term, genes in matrisome.items():
    sc.tl.score_genes(
        adata_sub,
        gene_list=genes,
        score_name=term,
        use_raw=False
    )

# ── compare T2D vs Control ─────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

score_cols = list(matrisome.keys())
scores = adata_sub.obs[score_cols + [DISEASE_COL]]
mean_scores = scores.groupby(DISEASE_COL)[score_cols].mean()
diff_ecm = (mean_scores.loc["T2D"] - mean_scores.loc["Control"]).sort_values()

print(diff_ecm)

# ── plot ───────────────────────────────────────────────────────────
colors = ["#d62728" if v > 0 else "#1f77b4" for v in diff_ecm]

fig, ax = plt.subplots(figsize=(5, 4))
ax.barh(diff_ecm.index, diff_ecm.values, color=colors, edgecolor="none", height=0.6)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Mean ECM score (T2D - Control)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nMatrisome ECM Categories", fontsize=9, loc="right")
ax.tick_params(labelsize=8)
ax.spines[["top","right"]].set_visible(False)
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False)

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_ECM_scores.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_ECM_scores.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# ── prep data ─────────────────────────────────────────────────────
top10_up = diff.head(10)
top10_dn = diff.tail(10)
hallmark_df = pd.concat([top10_dn, top10_up]).sort_values()

ecm_df = diff_ecm.sort_values()

# ── combined figure ───────────────────────────────────────────────
fig = plt.figure(figsize=(12, 6))
gs  = gridspec.GridSpec(1, 2, width_ratios=[2, 1.2], wspace=0.4)

# LEFT — Hallmark
ax1 = fig.add_subplot(gs[0])
colors1 = ["#d62728" if v > 0 else "#1f77b4" for v in hallmark_df]
ax1.barh(hallmark_df.index, hallmark_df.values,
         color=colors1, edgecolor="none", height=0.7)
ax1.axvline(0, color="black", linewidth=0.8)
ax1.set_xlabel("Mean pathway score (T2D - Control)", fontsize=8)
ax1.set_title("MSigDB Hallmark\nPathway Activity", fontsize=9)
ax1.tick_params(labelsize=7)
ax1.spines[["top","right"]].set_visible(False)

# RIGHT — ECM
ax2 = fig.add_subplot(gs[1])
colors2 = ["#d62728" if v > 0 else "#1f77b4" for v in ecm_df]
ax2.barh(ecm_df.index, ecm_df.values,
         color=colors2, edgecolor="none", height=0.7)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Mean ECM score (T2D - Control)", fontsize=8)
ax2.set_title("Matrisome\nECM Categories", fontsize=9)
ax2.tick_params(labelsize=7)
ax2.spines[["top","right"]].set_visible(False)

# shared legend + title
legend_elements = [
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
]
fig.legend(handles=legend_elements, fontsize=7, frameon=False,
           loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))

fig.suptitle("MSL2: T2D vs Control", fontsize=10, fontweight="bold", y=1.01)

plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_ECM_combined.png",
            dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Hallmark_ECM_combined.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
# ── SUBSET MSL1 ────────────────────────────────────────────────────
adata_msl1 = mesenchymalf[
    mesenchymalf.obs[CELLTYPE_COL] == "MSL1"
].copy()
print(adata_msl1.obs[DISEASE_COL].value_counts())

# ── DEG ────────────────────────────────────────────────────────────
sc.tl.rank_genes_groups(
    adata_msl1,
    groupby=DISEASE_COL,
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)
deg_msl1 = sc.get.rank_genes_groups_df(adata_msl1, group="T2D").dropna()

deg_strict_msl1 = deg_msl1[
    (deg_msl1["pvals_adj"] < 0.05) &
    (deg_msl1["logfoldchanges"].abs() > 0.25)
]
print(f"DEGs: {len(deg_strict_msl1)} — Up: {(deg_strict_msl1['logfoldchanges']>0).sum()}, Down: {(deg_strict_msl1['logfoldchanges']<0).sum()}")

up_msl1 = deg_strict_msl1[deg_strict_msl1["logfoldchanges"] > 0]["names"].tolist()
dn_msl1 = deg_strict_msl1[deg_strict_msl1["logfoldchanges"] < 0]["names"].tolist()

# ── HALLMARK SCORES ────────────────────────────────────────────────
for term, genes in gene_sets.items():
    sc.tl.score_genes(adata_msl1, gene_list=genes,
                      score_name=term, use_raw=False)

score_cols = list(gene_sets.keys())
scores_msl1 = adata_msl1.obs[score_cols + [DISEASE_COL]]
mean_msl1   = scores_msl1.groupby(DISEASE_COL)[score_cols].mean()
diff_msl1   = (mean_msl1.loc["T2D"] - mean_msl1.loc["Control"]).sort_values()

# ── ECM SCORES ─────────────────────────────────────────────────────
for term, genes in matrisome.items():
    sc.tl.score_genes(adata_msl1, gene_list=genes,
                      score_name=term, use_raw=False)

ecm_cols    = list(matrisome.keys())
ecm_msl1    = adata_msl1.obs[ecm_cols + [DISEASE_COL]]
mean_ecm_m1 = ecm_msl1.groupby(DISEASE_COL)[ecm_cols].mean()
diff_ecm_m1 = (mean_ecm_m1.loc["T2D"] - mean_ecm_m1.loc["Control"]).sort_values()

print("\nTop Hallmark UP:");   print(diff_msl1.tail(5))
print("Top Hallmark DOWN:");  print(diff_msl1.head(5))
print("\nECM scores:");        print(diff_ecm_m1.sort_values())

# ── COMBINED PLOT ──────────────────────────────────────────────────
top10_up = diff_msl1.tail(10)
top10_dn = diff_msl1.head(10)
hallmark_msl1 = pd.concat([top10_dn, top10_up]).sort_values()

fig = plt.figure(figsize=(12, 6))
gs  = gridspec.GridSpec(1, 2, width_ratios=[2, 1.2], wspace=0.4)

ax1 = fig.add_subplot(gs[0])
colors1 = ["#d62728" if v > 0 else "#1f77b4" for v in hallmark_msl1]
ax1.barh(hallmark_msl1.index, hallmark_msl1.values,
         color=colors1, edgecolor="none", height=0.7)
ax1.axvline(0, color="black", linewidth=0.8)
ax1.set_xlabel("Mean pathway score (T2D - Control)", fontsize=8)
ax1.set_title("MSigDB Hallmark\nPathway Activity", fontsize=9)
ax1.tick_params(labelsize=7)
ax1.spines[["top","right"]].set_visible(False)

ax2 = fig.add_subplot(gs[1])
ecm_sorted = diff_ecm_m1.sort_values()
colors2 = ["#d62728" if v > 0 else "#1f77b4" for v in ecm_sorted]
ax2.barh(ecm_sorted.index, ecm_sorted.values,
         color=colors2, edgecolor="none", height=0.7)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_xlabel("Mean ECM score (T2D - Control)", fontsize=8)
ax2.set_title("Matrisome\nECM Categories", fontsize=9)
ax2.tick_params(labelsize=7)
ax2.spines[["top","right"]].set_visible(False)

fig.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False,
   loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))

fig.suptitle("MSL1: T2D vs Control", fontsize=10, fontweight="bold", y=1.01)

plt.savefig("/Users/kmeneses/Desktop/MSL1_Hallmark_ECM_combined.png",
            dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_Hallmark_ECM_combined.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
# get full reactome results for MSL2
e_up_r2 = gp.enrichr(gene_list=up2, gene_sets="Reactome_Pathways_2024",
                      organism="Human", outdir=None)
e_dn_r2 = gp.enrichr(gene_list=dn2, gene_sets="Reactome_Pathways_2024",
                      organism="Human", outdir=None)

r_up2 = e_up_r2.results[e_up_r2.results["Adjusted P-value"]<0.05].head(8).copy()
r_dn2 = e_dn_r2.results[e_dn_r2.results["Adjusted P-value"]<0.05].head(8).copy()

def shorten(s, n=42):
    return s[:n]+"..." if len(s)>n else s

r_up2["Term"] = r_up2["Term"].apply(shorten)
r_dn2["Term"] = r_dn2["Term"].apply(shorten)
r_up2["score"] =  -np.log10(r_up2["Adjusted P-value"])
r_dn2["score"] = -(-np.log10(r_dn2["Adjusted P-value"]))

react2_df = pd.concat([r_dn2, r_up2]).sort_values("score")
colors_r2 = ["#d62728" if s>0 else "#1f77b4" for s in react2_df["score"]]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(react2_df["Term"], react2_df["score"],
        color=colors_r2, edgecolor="none", height=0.7)
ax.axvline(0, color="black", linewidth=0.8)

for i, (_, row) in enumerate(react2_df.iterrows()):
    xpos = 0.1 if row["score"]>0 else -0.1
    ha   = "left" if row["score"]>0 else "right"
    ax.text(xpos, i, row["Overlap"], va="center",
            fontsize=6, color="white", fontweight="bold", ha=ha)

ax.set_xlabel("-log10(adj p-value)", fontsize=8)
ax.set_title("MSL2: T2D vs Control\nReactome 2024 — Pseudobulk",
             fontsize=9, fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
ax.tick_params(labelsize=7)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#d62728", label="Up in T2D"),
    Patch(facecolor="#1f77b4", label="Down in T2D")
], fontsize=7, frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome_pseudobulk.png",
            dpi=300, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL2_Reactome_pseudobulk.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=False)

# MSL1 hallmark
top_msl1 = pd.concat([diff_msl1.head(10), diff_msl1.tail(10)]).sort_values()
colors_msl1 = ["#f489e6a9" if v > 0 else "#82521e" for v in top_msl1]
axes[0].barh(top_msl1.index, top_msl1.values, color=colors_msl1, edgecolor="none", height=0.7)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("MSL1: T2D vs Control\nMSigDB Hallmark", fontsize=9)
axes[0].set_xlabel("Mean pathway score (T2D - Control)", fontsize=8)
axes[0].tick_params(labelsize=7)
axes[0].spines[["top","right"]].set_visible(False)

# MSL2 hallmark
top_msl2 = pd.concat([diff.head(10), diff.tail(10)]).sort_values()
colors_msl2 = ["#f489e6a9" if v > 0 else "#82521e" for v in top_msl2]
axes[1].barh(top_msl2.index, top_msl2.values, color=colors_msl2, edgecolor="none", height=0.7)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("MSL2: T2D vs Control\nMSigDB Hallmark", fontsize=9)
axes[1].set_xlabel("Mean pathway score (T2D - Control)", fontsize=8)
axes[1].tick_params(labelsize=7)
axes[1].spines[["top","right"]].set_visible(False)

from matplotlib.patches import Patch
fig.legend(handles=[
    Patch(facecolor="#f489e6a9", label="Up in T2D"),
    Patch(facecolor="#82521e", label="Down in T2D")
], fontsize=7, frameon=False, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/MSL1_vs_MSL2_Hallmark.png", dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/MSL1_vs_MSL2_Hallmark.svg",dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import seaborn as sns

# ── run for ALL mesenchymal clusters ──────────────────────────────
clusters = mesenchymalf.obs[CELLTYPE_COL].unique()
all_diffs = {}

for cluster in clusters:
    print(f"Processing {cluster}...")
    sub = mesenchymalf[mesenchymalf.obs[CELLTYPE_COL] == cluster].copy()
    
    counts = sub.obs[DISEASE_COL].value_counts()
    if "T2D" not in counts or "Control" not in counts:
        print(f"  Skipping — missing group"); continue
    if counts.min() < 10:
        print(f"  Skipping — too few cells"); continue

    # score hallmark pathways
    for term, genes in gene_sets.items():
        sc.tl.score_genes(sub, gene_list=genes,
                          score_name=term, use_raw=False)

    score_cols = list(gene_sets.keys())
    means = sub.obs[score_cols + [DISEASE_COL]].groupby(DISEASE_COL)[score_cols].mean()
    
    if "T2D" in means.index and "Control" in means.index:
        all_diffs[cluster] = means.loc["T2D"] - means.loc["Control"]

# ── build matrix ───────────────────────────────────────────────────
heatmap_df = pd.DataFrame(all_diffs).T  # clusters x pathways
print(heatmap_df.shape)

# ── plot heatmap ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, len(heatmap_df) * 0.6 + 2))

sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    center=0,
    linewidths=0.3,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Mean score (T2D - Control)", "shrink": 0.5}
)

ax.set_title("Mesenchymal Clusters: T2D vs Control\nMSigDB Hallmark Pathway Activity",
             fontsize=10, fontweight="bold")
ax.set_xlabel("Hallmark Pathways", fontsize=8)
ax.set_ylabel("Cluster", fontsize=8)
ax.tick_params(axis="x", labelsize=6, rotation=90)
ax.tick_params(axis="y", labelsize=8, rotation=0)

plt.tight_layout()
plt.savefig("/Users/kmeneses/Desktop/AllClusters_Hallmark_heatmap.png",
            dpi=600, bbox_inches="tight")
plt.savefig("/Users/kmeneses/Desktop/AllClusters_Hallmark_heatmap.pdf",
            bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import zscore

# ==========================================
# STYLE
# ==========================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# ==========================================
# FUNCTIONAL GENE BLOCKS
# ==========================================

gene_blocks = {

    "ECM_BM": [
        "COL4A1",
        "COL4A2",
        "LAMA5",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "ECM_Remodeling": [
        "LOXL4",
        "PCOLCE",
        "TIMP3",
        "VCAN",
        "FN1",
        "COL6A3"
    ],

    "Contractility": [
        "ACTA2",
        "MYL9",
        "TAGLN",
        "CALD1",
        "RGS5"
    ],

    "Inflammatory_Niche": [
        "C1R",
        "C1S",
        "CXCL12",
        "IL6",
        "SERPINA1"
    ],

    "Angiogenic_Signaling": [
        "FLT1",
        "KDR",
        "ANGPT2",
        "PDGFRB",
        "ACE2"
    ],

    "Stress_Adaptive": [
        "DDIT3",
        "HSPA1A",
        "FOS",
        "JUN",
        "S100A6"
    ]
}

# ==========================================
# FLATTEN GENE LIST
# ==========================================

genes = []

for block in gene_blocks.values():
    genes.extend(block)

genes = list(dict.fromkeys(genes))

genes = [
    g for g in genes
    if g in mesenchymalf.var_names
]

# ==========================================
# SUBSET ONLY MSL1/MSL2
# ==========================================

adata_sub = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"].isin([
        "MSL1",
        "MSL2"
    ])
].copy()

# ==========================================
# CREATE GROUP COLUMN
# ==========================================

adata_sub.obs["group"] = (
    adata_sub.obs["mesenchymal_leiden"].astype(str)
    + "_"
    + adata_sub.obs["disease_state"].astype(str)
)

# ==========================================
# EXTRACT EXPRESSION
# ==========================================

X = adata_sub[:, genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

expr = pd.DataFrame(
    X,
    columns=genes,
    index=adata_sub.obs["group"]
)

# ==========================================
# PSEUDOBULK MEAN
# ==========================================

mean_expr = expr.groupby(level=0).mean()

# ==========================================
# Z-SCORE ACROSS GROUPS
# ==========================================

mean_expr_z = mean_expr.apply(
    zscore,
    axis=0
)

# ==========================================
# MANUAL COLUMN ORDER
# ==========================================

group_order = [
    "MSL1_ND",
    "MSL1_T2D",
    "MSL2_ND",
    "MSL2_T2D"
]

group_order = [
    x for x in group_order
    if x in mean_expr_z.index
]

mean_expr_z = mean_expr_z.loc[group_order]

# ==========================================
# MANUAL GENE ORDER
# ==========================================

ordered_genes = []

for block_name, block_genes in gene_blocks.items():

    valid = [
        g for g in block_genes
        if g in mean_expr_z.columns
    ]

    ordered_genes.extend(valid)

mean_expr_z = mean_expr_z[
    ordered_genes
]

# ==========================================
# TRANSPOSE FOR HEATMAP
# ==========================================

heatmap_df = mean_expr_z.T

# ==========================================
# PLOT
# ==========================================

plt.figure(figsize=(4.5,6))

sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={
        "label": "Z-score",
        "shrink": 0.6
    }
)

# ==========================================
# BLOCK SEPARATORS
# ==========================================

current = 0

for block_name, block_genes in gene_blocks.items():

    valid = [
        g for g in block_genes
        if g in heatmap_df.index
    ]

    current += len(valid)

    plt.axhline(
        current,
        color="black",
        linewidth=1
    )

# ==========================================
# FORMAT
# ==========================================

plt.title(
    "Functional remodeling programs",
    fontsize=10
)

plt.xlabel("")
plt.ylabel("")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.yticks(
    rotation=0
)

plt.tight_layout()

# ==========================================
# EXPORT
# ==========================================

out_base = (
    "/Users/kmeneses/Desktop/"
    "MSL_Functional_GeneBlock_Heatmap"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# ==========================================
# STYLE
# ==========================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# ==========================================
# FUNCTIONAL GENE BLOCKS
# ==========================================

gene_blocks = {

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMA5",
        "LAMB2",
        "HSPG2"
    ],

    "ECM_Remodeling": [
        "FN1",
        "VCAN",
        "COL6A3",
        "LOXL4",
        "PCOLCE"
    ],

    "Contractility": [
        "RGS5",
        "ACTA2",
        "TAGLN",
        "MYL9",
        "CALD1"
    ],

    "Inflammatory_Niche": [
        "C1R",
        "C1S",
        "CXCL12",
        "SERPINA1",
        "IL6"
    ],

    "Angiogenic_Signaling": [
        "FLT1",
        "KDR",
        "ANGPT2",
        "PDGFRB",
        "ACE2"
    ]
}

# ==========================================
# FLATTEN GENE LIST
# ==========================================

genes = []

for block in gene_blocks.values():
    genes.extend(block)

genes = list(dict.fromkeys(genes))

genes = [
    g for g in genes
    if g in mesenchymalf.var_names
]

# ==========================================
# SUBSET MSL1 + MSL2
# ==========================================

adata_sub = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"].isin([
        "MSL1",
        "MSL2"
    ])
].copy()

# ==========================================
# CREATE GROUPS
# ==========================================

adata_sub.obs["group"] = (
    adata_sub.obs["mesenchymal_leiden"].astype(str)
    + "_"
    + adata_sub.obs["disease_state"].astype(str)
)

# ==========================================
# EXTRACT EXPRESSION
# ==========================================

X = adata_sub[:, genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

expr = pd.DataFrame(
    X,
    columns=genes,
    index=adata_sub.obs["group"]
)

# ==========================================
# PSEUDOBULK MEAN
# ==========================================

mean_expr = expr.groupby(level=0).mean()

# ==========================================
# MANUAL COLUMN ORDER
# ==========================================

group_order = [
    "MSL1_Control",
    "MSL1_T2D",
    "MSL2_Control",
    "MSL2_T2D"
]

group_order = [
    x for x in group_order
    if x in mean_expr.index
]

mean_expr = mean_expr.loc[group_order]

# ==========================================
# MANUAL GENE ORDER
# ==========================================

ordered_genes = []

for block_name, block_genes in gene_blocks.items():

    valid = [
        g for g in block_genes
        if g in mean_expr.columns
    ]

    ordered_genes.extend(valid)

mean_expr = mean_expr[
    ordered_genes
]

# ==========================================
# LOG SCALE FOR VISUALIZATION
# ==========================================

heatmap_df = np.log1p(mean_expr.T)

# ==========================================
# OPTIONAL:
# SCALE EACH GENE ACROSS GROUPS
# ==========================================

heatmap_df = heatmap_df.sub(
    heatmap_df.mean(axis=1),
    axis=0
)

heatmap_df = heatmap_df.div(
    heatmap_df.std(axis=1),
    axis=0
)

# ==========================================
# PLOT
# ==========================================

plt.figure(figsize=(4.8,5.5))

sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    linewidths=0.3,
    linecolor="white",
    cbar_kws={
        "label": "Scaled expression",
        "shrink": 0.7
    }
)

# ==========================================
# BLOCK SEPARATORS
# ==========================================

current = 0

for block_name, block_genes in gene_blocks.items():

    valid = [
        g for g in block_genes
        if g in heatmap_df.index
    ]

    current += len(valid)

    plt.axhline(
        current,
        color="black",
        linewidth=1.2
    )

# ==========================================
# FORMAT
# ==========================================

plt.title(
    "Pericyte functional remodeling programs",
    fontsize=10
)

plt.xlabel("")
plt.ylabel("")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.yticks(
    rotation=0
)

plt.tight_layout()

# ==========================================
# EXPORT
# ==========================================

out_base = (
    "/Users/kmeneses/Desktop/"
    "Pericyte_Functional_GeneBlocks"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
adata_sub.obs['disease_state']

In [ ]:
print(
    adata_sub.uns["rank_genes_groups"]["names"].dtype.names
)

In [ ]:
sc.tl.rank_genes_groups(
    adata_sub,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

In [ ]:
# ==========================================
# GET TOP DEG GENES
# ==========================================

# Example:
# assumes you already ran:
# sc.tl.rank_genes_groups()

deg = sc.get.rank_genes_groups_df(
    adata_sub,
    group="T2D"
)

# ==========================================
# FILTER SIGNIFICANT
# ==========================================

deg = deg[
    (deg["pvals_adj"] < 0.05)
].copy()

# ==========================================
# REMOVE RIBOSOMAL / MT / LOW INFO
# ==========================================

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-",
    "MTRNR",
    "MALAT1"
)

deg = deg[
    ~deg["names"].str.startswith(remove_prefixes)
]

# ==========================================
# TOP UP + DOWN GENES
# ==========================================

top_up = (
    deg
    .sort_values(
        "logfoldchanges",
        ascending=False
    )
    .head(10)
)

top_down = (
    deg
    .sort_values(
        "logfoldchanges",
        ascending=True
    )
    .head(10)
)

# ==========================================
# COMBINE
# ==========================================

top_genes = pd.concat([
    top_up,
    top_down
])

# ==========================================
# VIEW
# ==========================================

print(
    top_genes[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# ==========================================
# GENE LIST
# ==========================================

genes = top_genes["names"].tolist()

print("\nTop DEG genes:")
print(genes)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list

# ======================================================
# STYLE
# ======================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# ======================================================
# CLEAN FUNCTIONAL MODULES
# Expanded from scRNA-seq biology
# ======================================================

gene_blocks = {

    "Basement_Membrane": [
        "COL4A1",
        "COL4A2",
        "LAMA5",
        "LAMB2",
        "HSPG2"
    ],

    "ECM_Remodeling": [
        "FN1",
        "VCAN",
        "COL6A3",
        "PCOLCE",
        "LOXL4"
    ],

    "Contractility": [
        "RGS5",
        "MYL9",
        "CALD1",
        "TAGLN",
        "ACTA2"
    ],

    "Inflammatory_Niche": [
        "C1R",
        "C1S",
        "CXCL12",
        "SERPINA1",
        "IL6"
    ],

    "Angiogenic_Signaling": [
        "FLT1",
        "KDR",
        "ANGPT2",
        "ACE2",
        "PDGFRB"
    ]
}

# ======================================================
# FLATTEN GENE LIST
# ======================================================

genes = []

for block in gene_blocks.values():
    genes.extend(block)

genes = list(dict.fromkeys(genes))

genes = [
    g for g in genes
    if g in mesenchymalf.var_names
]

print(f"\nGenes retained: {len(genes)}")

# ======================================================
# SUBSET CELLS
# ======================================================

adata_sub = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"].isin([
        "MSL1",
        "MSL2"
    ])
].copy()

# ======================================================
# GROUP LABELS
# ======================================================

adata_sub.obs["group"] = (
    adata_sub.obs["mesenchymal_leiden"].astype(str)
    + "_"
    + adata_sub.obs["disease_state"].astype(str)
)

# ======================================================
# EXTRACT EXPRESSION
# ======================================================

X = adata_sub[:, genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

expr = pd.DataFrame(
    X,
    columns=genes,
    index=adata_sub.obs["group"]
)

# ======================================================
# PSEUDOBULK MEAN
# ======================================================

mean_expr = expr.groupby(level=0).mean()

# ======================================================
# COLUMN ORDER
# ======================================================

group_order = [
    "MSL1_Control",
    "MSL1_T2D",
    "MSL2_Control",
    "MSL2_T2D"
]

group_order = [
    g for g in group_order
    if g in mean_expr.index
]

mean_expr = mean_expr.loc[group_order]

# ======================================================
# LOG1P
# ======================================================

heatmap_df = np.log1p(mean_expr.T)

# ======================================================
# SCALE EACH GENE
# ======================================================

heatmap_df = heatmap_df.sub(
    heatmap_df.mean(axis=1),
    axis=0
)

heatmap_df = heatmap_df.div(
    heatmap_df.std(axis=1),
    axis=0
)

# ======================================================
# CLUSTER GENES WITHIN MODULES
# ======================================================

ordered_genes = []

for block_name, block_genes in gene_blocks.items():

    valid = [
        g for g in block_genes
        if g in heatmap_df.index
    ]

    if len(valid) > 1:

        sub = heatmap_df.loc[valid]

        Z = linkage(
            sub.values,
            method="average",
            metric="euclidean"
        )

        order = leaves_list(Z)

        valid = list(
            sub.index[order]
        )

    ordered_genes.extend(valid)

# reorder
heatmap_df = heatmap_df.loc[
    ordered_genes
]

# ======================================================
# MODULE COLOR SIDEBAR
# ======================================================

module_colors = {
    "Basement_Membrane": "#8da0cb",
    "ECM_Remodeling": "#fc8d62",
    "Contractility": "#66c2a5",
    "Inflammatory_Niche": "#e78ac3",
    "Angiogenic_Signaling": "#ffd92f"
}

row_colors = []

for gene in heatmap_df.index:

    assigned = None

    for module, module_genes in gene_blocks.items():

        if gene in module_genes:
            assigned = module_colors[module]

    row_colors.append(assigned)

# ======================================================
# CLUSTERMAP
# ======================================================

g = sns.clustermap(
    heatmap_df,
    row_cluster=False,
    col_cluster=False,
    row_colors=row_colors,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    linewidths=0.3,
    linecolor="white",
    figsize=(5,6),
    cbar_kws={
        "label": "Scaled expression"
    }
)

# ======================================================
# REMOVE DENDROGRAM SPACE
# ======================================================

g.ax_row_dendrogram.set_visible(False)
g.ax_col_dendrogram.set_visible(False)

# ======================================================
# TITLE
# ======================================================

g.ax_heatmap.set_title(
    "Pericyte functional remodeling programs",
    fontsize=10,
    pad=12
)

# ======================================================
# FORMAT
# ======================================================

g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_ylabel("")

plt.setp(
    g.ax_heatmap.get_xticklabels(),
    rotation=45,
    ha="right"
)

plt.setp(
    g.ax_heatmap.get_yticklabels(),
    rotation=0
)

# ======================================================
# ADD MODULE SEPARATORS
# ======================================================

current = 0

for block_name, block_genes in gene_blocks.items():

    valid = [
        g for g in ordered_genes
        if g in block_genes
    ]

    current += len(valid)

    g.ax_heatmap.axhline(
        current,
        color="black",
        linewidth=1.2
    )

# ======================================================
# EXPORT
# ======================================================

out_base = (
    "/Users/kmeneses/Desktop/"
    "Pericyte_Functional_Modules_Clean"
)

g.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

g.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

g.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ======================================================
# MSL1: T2D vs Control
# ======================================================

adata_msl1 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL1"
].copy()

print(
    adata_msl1.obs["disease_state"]
    .value_counts()
)

sc.tl.rank_genes_groups(
    adata_msl1,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl1 = sc.get.rank_genes_groups_df(
    adata_msl1,
    group="T2D"
)

# ======================================================
# MSL2: T2D vs Control
# ======================================================

adata_msl2 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL2"
].copy()

print(
    adata_msl2.obs["disease_state"]
    .value_counts()
)

sc.tl.rank_genes_groups(
    adata_msl2,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl2 = sc.get.rank_genes_groups_df(
    adata_msl2,
    group="T2D"
)

# ======================================================
# FILTER
# ======================================================

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-",
    "MTRNR",
    "MALAT1"
)

deg_msl1 = deg_msl1[
    deg_msl1["pvals_adj"] < 0.05
]

deg_msl1 = deg_msl1[
    ~deg_msl1["names"].str.startswith(remove_prefixes)
]

deg_msl2 = deg_msl2[
    deg_msl2["pvals_adj"] < 0.05
]

deg_msl2 = deg_msl2[
    ~deg_msl2["names"].str.startswith(remove_prefixes)
]

# ======================================================
# TOP GENES
# ======================================================

top_msl1_up = (
    deg_msl1
    .sort_values(
        "logfoldchanges",
        ascending=False
    )
    .head(10)
)

top_msl1_down = (
    deg_msl1
    .sort_values(
        "logfoldchanges",
        ascending=True
    )
    .head(10)
)

top_msl2_up = (
    deg_msl2
    .sort_values(
        "logfoldchanges",
        ascending=False
    )
    .head(10)
)

top_msl2_down = (
    deg_msl2
    .sort_values(
        "logfoldchanges",
        ascending=True
    )
    .head(10)
)

# ======================================================
# VIEW
# ======================================================

print("\n====================")
print("MSL1 TOP DEGs")
print("====================")

print(
    pd.concat([
        top_msl1_up,
        top_msl1_down
    ])[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

print("\n====================")
print("MSL2 TOP DEGs")
print("====================")

print(
    pd.concat([
        top_msl2_up,
        top_msl2_down
    ])[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# ======================================================
# SAVE GENE LISTS
# ======================================================

msl1_genes = pd.concat([
    top_msl1_up,
    top_msl1_down
])["names"].tolist()

msl2_genes = pd.concat([
    top_msl2_up,
    top_msl2_down
])["names"].tolist()

print("\nMSL1 genes:")
print(msl1_genes)

print("\nMSL2 genes:")
print(msl2_genes)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scanpy as sc
from scipy.cluster.hierarchy import linkage, leaves_list

# ======================================================
# STYLE
# ======================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# ======================================================
# REMOVE CONTAMINATION GENES
# ======================================================

contam_genes = [

    # exocrine
    "CPA1","CPA2","CPB1","CTRC","CEL",
    "PRSS1","PRSS2","REG1A","REG1B",
    "REG3A","PNLIP","CLPS","SPINK1",
    "PLA2G1B",

    # endocrine
    "INS","IAPP","GCG","SST",
    "PPY","CHGA","CHGB",

    # ambient/common
    "KRT19","KRT8","KRT18","EPCAM"
]

# ======================================================
# MSL1 DEG
# ======================================================

adata_msl1 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL1"
].copy()

sc.tl.rank_genes_groups(
    adata_msl1,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl1 = sc.get.rank_genes_groups_df(
    adata_msl1,
    group="T2D"
)

# ======================================================
# MSL2 DEG
# ======================================================

adata_msl2 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL2"
].copy()

sc.tl.rank_genes_groups(
    adata_msl2,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl2 = sc.get.rank_genes_groups_df(
    adata_msl2,
    group="T2D"
)

# ======================================================
# FILTER
# ======================================================

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-",
    "MTRNR",
    "MALAT1"
)

def clean_deg(df):

    df = df.copy()

    df = df[
        df["pvals_adj"] < 0.05
    ]

    df = df[
        ~df["names"].str.startswith(remove_prefixes)
    ]

    df = df[
        ~df["names"].isin(contam_genes)
    ]

    return df

deg_msl1 = clean_deg(deg_msl1)
deg_msl2 = clean_deg(deg_msl2)

# ======================================================
# TAKE TOP 25 UP + DOWN
# ======================================================

top_msl1 = pd.concat([
    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(25),

    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(25)
])

top_msl2 = pd.concat([
    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(25),

    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(25)
])

# ======================================================
# COMBINE GENES
# ======================================================

genes = list(dict.fromkeys(
    top_msl1["names"].tolist()
    + top_msl2["names"].tolist()
))

genes = [
    g for g in genes
    if g in mesenchymalf.var_names
]

print(f"\nGenes retained: {len(genes)}")

# ======================================================
# SUBSET DATA
# ======================================================

adata_sub = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"].isin([
        "MSL1",
        "MSL2"
    ])
].copy()

adata_sub.obs["group"] = (
    adata_sub.obs["mesenchymal_leiden"].astype(str)
    + "_"
    + adata_sub.obs["disease_state"].astype(str)
)

# ======================================================
# EXTRACT EXPRESSION
# ======================================================

X = adata_sub[:, genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

expr = pd.DataFrame(
    X,
    columns=genes,
    index=adata_sub.obs["group"]
)

# ======================================================
# PSEUDOBULK MEAN
# ======================================================

mean_expr = expr.groupby(level=0).mean()

# ======================================================
# COLUMN ORDER
# ======================================================

group_order = [
    "MSL1_Control",
    "MSL1_T2D",
    "MSL2_Control",
    "MSL2_T2D"
]

group_order = [
    x for x in group_order
    if x in mean_expr.index
]

mean_expr = mean_expr.loc[group_order]

# ======================================================
# LOG + SCALE
# ======================================================

heatmap_df = np.log1p(mean_expr.T)

heatmap_df = heatmap_df.sub(
    heatmap_df.mean(axis=1),
    axis=0
)

heatmap_df = heatmap_df.div(
    heatmap_df.std(axis=1),
    axis=0
)

# ======================================================
# CLUSTER GENES
# ======================================================

Z = linkage(
    heatmap_df.values,
    method="average",
    metric="euclidean"
)

order = leaves_list(Z)

heatmap_df = heatmap_df.iloc[
    order
]

# ======================================================
# PLOT
# ======================================================

plt.figure(figsize=(4.5,8))

sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    linewidths=0.15,
    linecolor="white",
    cbar_kws={
        "label": "Scaled expression",
        "shrink": 0.6
    }
)

# ======================================================
# FORMAT
# ======================================================

plt.title(
    "MSL1/MSL2 diabetic remodeling programs",
    fontsize=10
)

plt.xlabel("")
plt.ylabel("")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.yticks(
    rotation=0,
    fontsize=6
)

plt.tight_layout()

# ======================================================
# EXPORT
# ======================================================

out_base = (
    "/Users/kmeneses/Desktop/"
    "MSL1_MSL2_DEG_Heatmap"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scanpy as sc
from scipy.cluster.hierarchy import linkage, leaves_list

# ======================================================
# STYLE
# ======================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# ======================================================
# REMOVE BAD / CONTAMINATION GENES
# ======================================================

remove_genes = [

    # sex
    "XIST","DDX3Y","EIF1AY",

    # ribo / mt
    "MALAT1",

    # endocrine/exocrine contamination
    "INS","IAPP","CHGA","CHGB",
    "CPA1","CPA2","CPB1","CTRC",
    "PRSS1","PRSS2","CEL","CLPS",
    "PNLIP","REG1A","REG1B",
    "SPINK1","PLA2G1B",

    # weak biology
    "CRYBA2","SCG2","MMAB","SCD",
    "NBDY","CHCHD2","MTX1",
    "ALDOC","PIR","ARF5",
    "PLEKHO1","DOT1L"
]

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-",
    "MTRNR"
)

# ======================================================
# RUN MSL1 DEG
# ======================================================

adata_msl1 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL1"
].copy()

sc.tl.rank_genes_groups(
    adata_msl1,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl1 = sc.get.rank_genes_groups_df(
    adata_msl1,
    group="T2D"
)

# ======================================================
# RUN MSL2 DEG
# ======================================================

adata_msl2 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL2"
].copy()

sc.tl.rank_genes_groups(
    adata_msl2,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl2 = sc.get.rank_genes_groups_df(
    adata_msl2,
    group="T2D"
)

# ======================================================
# CLEAN FUNCTION
# ======================================================

def clean_deg(df):

    df = df.copy()

    df = df[
        df["pvals_adj"] < 0.05
    ]

    df = df[
        ~df["names"].str.startswith(remove_prefixes)
    ]

    df = df[
        ~df["names"].isin(remove_genes)
    ]

    return df

deg_msl1 = clean_deg(deg_msl1)
deg_msl2 = clean_deg(deg_msl2)

# ======================================================
# PULL MORE GENES
# ======================================================

top_msl1 = pd.concat([

    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(40),

    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(40)

])

top_msl2 = pd.concat([

    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(40),

    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(40)

])

# ======================================================
# ADD IDENTITY GENES
# ======================================================

identity_genes = [
    "RGS5",
    "PDGFRB",
    "CSPG4",
    "COL4A1",
    "LAMA5"
]

# ======================================================
# COMBINE GENES
# ======================================================

genes = (
    identity_genes
    + top_msl1["names"].tolist()
    + top_msl2["names"].tolist()
)

# unique
genes = list(dict.fromkeys(genes))

# present only
genes = [
    g for g in genes
    if g in mesenchymalf.var_names
]

print(f"\nGenes retained: {len(genes)}")

# ======================================================
# SUBSET DATA
# ======================================================

adata_sub = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"].isin([
        "MSL1",
        "MSL2"
    ])
].copy()

adata_sub.obs["group"] = (
    adata_sub.obs["mesenchymal_leiden"].astype(str)
    + "_"
    + adata_sub.obs["disease_state"].astype(str)
)

# ======================================================
# EXTRACT EXPRESSION
# ======================================================

X = adata_sub[:, genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

expr = pd.DataFrame(
    X,
    columns=genes,
    index=adata_sub.obs["group"]
)

# ======================================================
# PSEUDOBULK
# ======================================================

mean_expr = expr.groupby(level=0).mean()

# ======================================================
# COLUMN ORDER
# ======================================================

group_order = [
    "MSL1_Control",
    "MSL1_T2D",
    "MSL2_Control",
    "MSL2_T2D"
]

group_order = [
    g for g in group_order
    if g in mean_expr.index
]

mean_expr = mean_expr.loc[group_order]

# ======================================================
# LOG + SCALE
# ======================================================

heatmap_df = np.log1p(mean_expr.T)

heatmap_df = heatmap_df.sub(
    heatmap_df.mean(axis=1),
    axis=0
)

heatmap_df = heatmap_df.div(
    heatmap_df.std(axis=1),
    axis=0
)

# ======================================================
# CLUSTER GENES
# ======================================================

Z = linkage(
    heatmap_df.values,
    method="average",
    metric="euclidean"
)

order = leaves_list(Z)

heatmap_df = heatmap_df.iloc[
    order
]

# ======================================================
# PLOT
# ======================================================

plt.figure(figsize=(5,10))

sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    linewidths=0.15,
    linecolor="white",
    cbar_kws={
        "label": "Scaled expression",
        "shrink": 0.5
    }
)

# ======================================================
# FORMAT
# ======================================================

plt.title(
    "MSL1/MSL2 diabetic remodeling programs",
    fontsize=10
)

plt.xlabel("")
plt.ylabel("")

plt.xticks(
    rotation=45,
    ha="right"
)

plt.yticks(
    rotation=0,
    fontsize=6
)

plt.tight_layout()

# ======================================================
# EXPORT
# ======================================================

out_base = (
    "/Users/kmeneses/Desktop/"
    "MSL1_MSL2_Refined_Heatmap"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scanpy as sc
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster

# ======================================================
# STYLE
# ======================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# ======================================================
# REMOVE BAD / CONTAMINATION GENES
# ======================================================

remove_genes = [

    # sex
    "XIST","DDX3Y","EIF1AY",

    # ribo / mt
    "MALAT1",

    # endocrine/exocrine contamination
    "INS","IAPP","CHGA","CHGB",
    "CPA1","CPA2","CPB1","CTRC",
    "PRSS1","PRSS2","CEL","CLPS",
    "PNLIP","REG1A","REG1B",
    "SPINK1","PLA2G1B",

    # weak/noisy
    "CRYBA2","SCG2","MMAB","SCD",
    "NBDY","CHCHD2","MTX1",
    "ALDOC","PIR","ARF5",
    "PLEKHO1","DOT1L",

    # additional cleanup
    "LINC00467",
    "SCARA3",
    "ABCC4",
    "CYGB",
    "GMFG",
    "PRELID1",
    "ATP6V0B"
]

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-",
    "MTRNR"
)

# ======================================================
# RUN MSL1 DEG
# ======================================================

adata_msl1 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL1"
].copy()

sc.tl.rank_genes_groups(
    adata_msl1,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl1 = sc.get.rank_genes_groups_df(
    adata_msl1,
    group="T2D"
)

# ======================================================
# RUN MSL2 DEG
# ======================================================

adata_msl2 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL2"
].copy()

sc.tl.rank_genes_groups(
    adata_msl2,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl2 = sc.get.rank_genes_groups_df(
    adata_msl2,
    group="T2D"
)

# ======================================================
# CLEAN FUNCTION
# ======================================================

def clean_deg(df):

    df = df.copy()

    df = df[
        df["pvals_adj"] < 0.05
    ]

    df = df[
        ~df["names"].str.startswith(remove_prefixes)
    ]

    df = df[
        ~df["names"].isin(remove_genes)
    ]

    return df

deg_msl1 = clean_deg(deg_msl1)
deg_msl2 = clean_deg(deg_msl2)

# ======================================================
# PULL BIOLOGICALLY STRONGER GENES
# ======================================================

top_msl1 = pd.concat([

    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(25),

    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(15)

])

top_msl2 = pd.concat([

    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(25),

    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(15)

])

# ======================================================
# ADD IDENTITY GENES
# ======================================================

identity_genes = [
    "RGS5",
    "PDGFRB",
    "CSPG4",
    "COL4A1",
    "LAMA5"
]

# ======================================================
# COMBINE GENES
# ======================================================

genes = (
    identity_genes
    + top_msl1["names"].tolist()
    + top_msl2["names"].tolist()
)

genes = list(dict.fromkeys(genes))

genes = [
    g for g in genes
    if g in mesenchymalf.var_names
]

print(f"\nGenes retained: {len(genes)}")

# ======================================================
# SUBSET DATA
# ======================================================

adata_sub = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"].isin([
        "MSL1",
        "MSL2"
    ])
].copy()

adata_sub.obs["group"] = (
    adata_sub.obs["mesenchymal_leiden"].astype(str)
    + "_"
    + adata_sub.obs["disease_state"].astype(str)
)

# ======================================================
# EXTRACT EXPRESSION
# ======================================================

X = adata_sub[:, genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

expr = pd.DataFrame(
    X,
    columns=genes,
    index=adata_sub.obs["group"]
)

# ======================================================
# PSEUDOBULK MEAN
# ======================================================

mean_expr = expr.groupby(level=0).mean()

# ======================================================
# COLUMN ORDER
# ======================================================

group_order = [
    "MSL1_Control",
    "MSL1_T2D",
    "MSL2_Control",
    "MSL2_T2D"
]

group_order = [
    g for g in group_order
    if g in mean_expr.index
]

mean_expr = mean_expr.loc[group_order]

# ======================================================
# LOG + Z-SCORE
# ======================================================

heatmap_df = np.log1p(mean_expr.T)

# gene-wise z-score
heatmap_df = heatmap_df.sub(
    heatmap_df.mean(axis=1),
    axis=0
)

heatmap_df = heatmap_df.div(
    heatmap_df.std(axis=1),
    axis=0
)

# ======================================================
# HIERARCHICAL CLUSTERING
# ======================================================

Z = linkage(
    heatmap_df.values,
    method="average",
    metric="euclidean"
)

order = leaves_list(Z)

heatmap_df = heatmap_df.iloc[order]

# ======================================================
# DEFINE MAJOR CLUSTERS
# ======================================================

clusters = fcluster(
    Z,
    t=4,
    criterion="maxclust"
)

clusters = clusters[order]

# ======================================================
# PLOT
# ======================================================

fig, ax = plt.subplots(
    figsize=(4.2, 8)
)

sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    linewidths=0.15,
    linecolor="white",
    cbar_kws={
        "label": "Gene-wise Z-score",
        "shrink": 0.45
    },
    ax=ax
)

# ======================================================
# ADD CLUSTER SEPARATORS
# ======================================================

for i in range(1, len(clusters)):

    if clusters[i] != clusters[i-1]:

        ax.axhline(
            i,
            color="black",
            linewidth=1.3
        )

# ======================================================
# FORMAT
# ======================================================

ax.set_title(
    "Distinct diabetic remodeling states in pancreatic pericytes",
    fontsize=10,
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.xticks(
    rotation=45,
    ha="right",
    fontsize=8
)

plt.yticks(
    rotation=0,
    fontsize=7
)

# cleaner spines
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()

# ======================================================
# EXPORT
# ======================================================

out_base = (
    "/Users/kmeneses/Desktop/"
    "MSL1_MSL2_Publication_Heatmap"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import scanpy as sc
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster

# ======================================================
# STYLE
# ======================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# ======================================================
# REMOVE ONLY TRUE BAD GENES
# keep biologically informative remodeling genes
# ======================================================

remove_genes = [

    # sex
    "XIST",
    "DDX3Y",
    "EIF1AY",

    # obvious contamination
    "INS","IAPP","CHGA","CHGB",
    "CPA1","CPA2","CPB1","CTRC",
    "PRSS1","PRSS2","CEL","CLPS",
    "PNLIP","REG1A","REG1B",
    "SPINK1","PLA2G1B",

    # noisy / weak
    "SCG2",
    "CRYBA2",
    "CYGB"
]

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-",
    "MTRNR"
)

# ======================================================
# RUN MSL1 DEG
# ======================================================

adata_msl1 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL1"
].copy()

sc.tl.rank_genes_groups(
    adata_msl1,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl1 = sc.get.rank_genes_groups_df(
    adata_msl1,
    group="T2D"
)

# ======================================================
# RUN MSL2 DEG
# ======================================================

adata_msl2 = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"] == "MSL2"
].copy()

sc.tl.rank_genes_groups(
    adata_msl2,
    groupby="disease_state",
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

deg_msl2 = sc.get.rank_genes_groups_df(
    adata_msl2,
    group="T2D"
)

# ======================================================
# CLEAN FUNCTION
# ======================================================

def clean_deg(df):

    df = df.copy()

    df = df[
        df["pvals_adj"] < 0.05
    ]

    df = df[
        ~df["names"].str.startswith(remove_prefixes)
    ]

    df = df[
        ~df["names"].isin(remove_genes)
    ]

    return df

deg_msl1 = clean_deg(deg_msl1)
deg_msl2 = clean_deg(deg_msl2)

# ======================================================
# TOP GENES
# ======================================================

top_msl1 = pd.concat([

    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(30),

    deg_msl1.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(20)

])

top_msl2 = pd.concat([

    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=False
    ).head(30),

    deg_msl2.sort_values(
        "logfoldchanges",
        ascending=True
    ).head(20)

])

# ======================================================
# ADD PERICYTE IDENTITY GENES
# ======================================================

identity_genes = [
    "RGS5",
    "PDGFRB",
    "CSPG4",
    "COL4A1",
    "LAMA5"
]

# ======================================================
# COMBINE GENES
# ======================================================

genes = (
    identity_genes
    + top_msl1["names"].tolist()
    + top_msl2["names"].tolist()
)

genes = list(dict.fromkeys(genes))

genes = [
    g for g in genes
    if g in mesenchymalf.var_names
]

print(f"\nGenes retained: {len(genes)}")

# ======================================================
# SUBSET DATA
# ======================================================

adata_sub = mesenchymalf[
    mesenchymalf.obs["mesenchymal_leiden"].isin([
        "MSL1",
        "MSL2"
    ])
].copy()

adata_sub.obs["group"] = (
    adata_sub.obs["mesenchymal_leiden"].astype(str)
    + "_"
    + adata_sub.obs["disease_state"].astype(str)
)

# ======================================================
# EXTRACT EXPRESSION
# ======================================================

X = adata_sub[:, genes].X

if hasattr(X, "toarray"):
    X = X.toarray()

expr = pd.DataFrame(
    X,
    columns=genes,
    index=adata_sub.obs["group"]
)

# ======================================================
# PSEUDOBULK MEAN
# ======================================================

mean_expr = expr.groupby(level=0).mean()

# ======================================================
# COLUMN ORDER
# ======================================================

group_order = [
    "MSL1_Control",
    "MSL1_T2D",
    "MSL2_Control",
    "MSL2_T2D"
]

group_order = [
    g for g in group_order
    if g in mean_expr.index
]

mean_expr = mean_expr.loc[group_order]

# ======================================================
# LOG + Z-SCORE
# ======================================================

heatmap_df = np.log1p(mean_expr.T)

# gene-wise z-score
heatmap_df = heatmap_df.sub(
    heatmap_df.mean(axis=1),
    axis=0
)

heatmap_df = heatmap_df.div(
    heatmap_df.std(axis=1),
    axis=0
)

# ======================================================
# CLUSTER GENES
# ======================================================

Z = linkage(
    heatmap_df.values,
    method="average",
    metric="euclidean"
)

order = leaves_list(Z)

heatmap_df = heatmap_df.iloc[order]

# ======================================================
# MAJOR CLUSTERS
# ======================================================

clusters = fcluster(
    Z,
    t=4,
    criterion="maxclust"
)

clusters = clusters[order]

# ======================================================
# PLOT
# ======================================================

fig, ax = plt.subplots(
    figsize=(4.5,10)
)

sns.heatmap(
    heatmap_df,
    cmap="RdBu_r",
    vmin=-2,
    vmax=2,
    linewidths=0.15,
    linecolor="white",
    cbar_kws={
        "label": "Gene-wise Z-score",
        "shrink": 0.45
    },
    ax=ax
)

# ======================================================
# CLUSTER SEPARATORS
# ======================================================

for i in range(1, len(clusters)):

    if clusters[i] != clusters[i-1]:

        ax.axhline(
            i,
            color="black",
            linewidth=1.3
        )

# ======================================================
# FORMAT
# ======================================================

ax.set_title(
    "Distinct diabetic remodeling states in pancreatic pericytes",
    fontsize=10,
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.xticks(
    rotation=45,
    ha="right",
    fontsize=8
)

plt.yticks(
    rotation=0,
    fontsize=7
)

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()

# ======================================================
# EXPORT
# ======================================================

out_base = (
    "/Users/kmeneses/Desktop/"
    "MSL1_MSL2_Final_Heatmap"
)

plt.savefig(
    f"{out_base}.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{out_base}.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
sns.clustermap(
    heatmap_sub,
    cmap="RdBu_r",
    center=0,
    linewidths=0.5,
    figsize=(10, 4),
    row_cluster=True,
    col_cluster=True,
    cbar_pos=(1.02, 0.3, 0.02, 0.4)
)
plt.savefig("/Users/kmeneses/Desktop/AllClusters_clustermap.png",
            dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================================
# CREATE ND vs T2D WITHIN EACH MSL
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# CREATE CONDITION-SPECIFIC GROUPS
# =========================================================

mesenchymalf.obs["MSL_condition"] = (
    mesenchymalf.obs["disease_state"].astype(str)
    + "_"
    + mesenchymalf.obs["mesenchymal_leiden"].astype(str)
)

print(
    mesenchymalf.obs["MSL_condition"].value_counts()
)

# should look like:
# ND_MSL1
# T2D_MSL1
# ND_MSL2
# T2D_MSL2

# =========================================================
# MSL1
# T2D vs ND
# =========================================================

sc.tl.rank_genes_groups(
    mesenchymalf,
    groupby="MSL_condition",
    groups=["T2D_MSL1"],
    reference="Control_MSL1",
    method="wilcoxon",
    use_raw=False
)

res_msl1 = sc.get.rank_genes_groups_df(
    mesenchymalf,
    group="T2D_MSL1"
)

# =========================================================
# CLEAN
# =========================================================

res_msl1["neglog10_padj"] = -np.log10(
    res_msl1["pvals_adj"] + 1e-300
)

# =========================================================
# MSL2
# T2D vs ND
# =========================================================

sc.tl.rank_genes_groups(
    mesenchymalf,
    groupby="MSL_condition",
    groups=["T2D_MSL2"],
    reference="Control_MSL2",
    method="wilcoxon",
    use_raw=False
)

res_msl2 = sc.get.rank_genes_groups_df(
    mesenchymalf,
    group="T2D_MSL2"
)

# =========================================================
# CLEAN
# =========================================================

res_msl2["neglog10_padj"] = -np.log10(
    res_msl2["pvals_adj"] + 1e-300
)

# =========================================================
# CHECK
# =========================================================

print("\nMSL1:")
print(
    res_msl1.head()
)

print("\nMSL2:")
print(
    res_msl2.head()
)

In [ ]:
# =========================================================
# COMBINED VOLCANO:
# MSL1 + MSL2
# SIGNIFICANT PANEL GENES ONLY
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================================================
# SETTINGS
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# PANEL GENES
# =========================================================

panel_genes = [

    # BM collagens
    "COL4A1",
    "COL4A2",
    "COL4A3",

    # laminins
    "LAMA4",
    "LAMA5",
    "LAMB1",
    "LAMB2",
    "LAMC1",

    # BM structure
    "HSPG2",
    "NID1",
    "NID2",

    # fibrillar ECM
    "COL1A1",
    "COL1A2",
    "COL3A1",
    "COL5A1",
    "COL5A2",
    "COL6A1",
    "COL6A2",
    "COL6A3",
    "COL14A1",
    "COL18A1",

    # glycoproteins
    "FN1",
    "SPARCL1",
    "THBS1",
    "TINAGL1",
    "POSTN",
    "FBN1",
    "VCAN",
    "MFGE8",

    # pericyte / mural
    "RGS5",
    "CSPG4",
    "PDGFRB",
    "MCAM",
    "NOTCH3",
    "DES",
    "ACTA2",
    "TAGLN",
    "MYL9",
    "TPM2",

    # ECM remodeling
    "LOXL4",
    "ADAM9",
    "SERPINF1",
    "MMP2",

    # signaling / adhesion
    "ITGA6",
    "SDC4",
    "MGP",
    "C1R",
    "CHST10"
]

# =========================================================
# PREP FUNCTION
# =========================================================

def prep_res(df, group_name):

    d = df.copy()

    # ---------------------------------------------
    # keep panel genes
    # ---------------------------------------------

    d = d[
        d["names"].isin(panel_genes)
    ].copy()

    # ---------------------------------------------
    # clean
    # ---------------------------------------------

    d = d.dropna(
        subset=[
            "logfoldchanges",
            "pvals_adj"
        ]
    )

    d = d[
        np.isfinite(d["logfoldchanges"])
    ]

    # ---------------------------------------------
    # significance
    # ---------------------------------------------

    d = d[
        (d["pvals_adj"] < PADJ_THRESH) &
        (np.abs(d["logfoldchanges"]) > LFC_THRESH)
    ].copy()

    # ---------------------------------------------
    # add metadata
    # ---------------------------------------------

    d["group"] = group_name

    d["neglog10_padj"] = -np.log10(
        d["pvals_adj"] + 1e-300
    )

    return d

# =========================================================
# PREP BOTH RESULTS
# =========================================================

msl1_df = prep_res(
    res_msl1,
    "MSL1"
)

msl2_df = prep_res(
    res_msl2,
    "MSL2"
)

# =========================================================
# COMBINE
# =========================================================

combined = pd.concat(
    [msl1_df, msl2_df],
    axis=0
)

# =========================================================
# PLOT
# =========================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(10,5),
    sharey=True
)

# =========================================================
# LOOP
# =========================================================

for ax, group in zip(
    axes,
    ["MSL1", "MSL2"]
):

    d = combined[
        combined["group"] == group
    ].copy()

    # -----------------------------------------------------
    # colors
    # -----------------------------------------------------

    colors = [
        "royalblue" if x < 0 else "red"
        for x in d["logfoldchanges"]
    ]

    # -----------------------------------------------------
    # scatter
    # -----------------------------------------------------

    ax.scatter(
        d["logfoldchanges"],
        d["neglog10_padj"],
        s=75,
        color=colors,
        edgecolor="black",
        linewidth=0.4,
        zorder=3
    )

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    for _, row in d.iterrows():

        x = row["logfoldchanges"]
        y = row["neglog10_padj"]

        ax.text(
            x + (0.05 if x > 0 else -0.05),
            y + 0.25,
            row["names"],
            fontsize=6,
            ha="left" if x > 0 else "right"
        )

    # -----------------------------------------------------
    # threshold lines
    # -----------------------------------------------------

    ax.axvline(
        LFC_THRESH,
        linestyle="--",
        color="black",
        linewidth=1
    )

    ax.axvline(
        -LFC_THRESH,
        linestyle="--",
        color="black",
        linewidth=1
    )

    ax.axhline(
        -np.log10(PADJ_THRESH),
        linestyle="--",
        color="black",
        linewidth=1
    )

    # -----------------------------------------------------
    # axes
    # -----------------------------------------------------

    ax.set_xlim(-3,3)

    ax.set_xlabel(
        "Log fold change (T2D vs ND)"
    )

    ax.set_title(
        group,
        fontsize=10,
        weight="bold"
    )

    # -----------------------------------------------------
    # clean
    # -----------------------------------------------------

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(width=1)

# =========================================================
# Y LABEL
# =========================================================

axes[0].set_ylabel(
    "-log10 adjusted p-value"
)

# =========================================================
# GLOBAL TITLE
# =========================================================

fig.suptitle(
    "MSL Remodeling Programs",
    fontsize=11,
    weight="bold"
)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_MSL2_combined_volcano.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_MSL2_combined_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

combined.to_csv(
    "/Users/kmeneses/Desktop/MSL1_MSL2_combined_volcano.csv",
    index=False
)

In [ ]:
# =========================================================
# SINGLE COMBINED VOLCANO
# MSL1 + MSL2 TOGETHER
# =========================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# THRESHOLDS
# =========================================================

LFC = 0.25
PADJ = 0.05

# =========================================================
# PANEL GENES
# =========================================================

panel_genes = [

    "COL4A1",
    "COL4A2",
    "COL1A1",
    "COL1A2",
    "COL3A1",
    "COL5A1",
    "COL5A2",
    "COL14A1",
    "COL18A1",

    "LAMA4",
    "LAMA5",
    "LAMB1",
    "LAMB2",
    "LAMC1",

    "HSPG2",
    "FN1",
    "VCAN",
    "POSTN",
    "SPARCL1",
    "LOXL4",
    "MGP",
    "SERPINF1",
    "FBN1",
    "CHST10",

    "RGS5",
    "CSPG4",
    "PDGFRB",

    "THBS1",
    "TINAGL1",
    "MFGE8",
    "ITGA6",
    "SDC4",

    "MYL9",
    "TPM2",
    "ADAM9"
]

# =========================================================
# PREP FUNCTION
# =========================================================

def prep_df(df, celltype):

    d = df.copy()

    d = d[
        d["names"].isin(panel_genes)
    ].copy()

    d = d.dropna(
        subset=[
            "logfoldchanges",
            "pvals_adj"
        ]
    )

    d["neglog10_padj"] = -np.log10(
        d["pvals_adj"] + 1e-300
    )

    # significant only
    d = d[
        (d["pvals_adj"] < PADJ) &
        (np.abs(d["logfoldchanges"]) > LFC)
    ].copy()

    d["celltype"] = celltype

    return d

# =========================================================
# PREP DATA
# =========================================================

msl1 = prep_df(
    res_msl1,
    "MSL1"
)

msl2 = prep_df(
    res_msl2,
    "MSL2"
)

plot_df = pd.concat(
    [msl1, msl2],
    axis=0
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(4.2,3.2)
)

# =========================================================
# COLORS
# =========================================================

color_map = {
    "MSL1": "royalblue",
    "MSL2": "red"
}

# =========================================================
# PLOT EACH CELL TYPE
# =========================================================

for ct in ["MSL1", "MSL2"]:

    d = plot_df[
        plot_df["celltype"] == ct
    ]

    ax.scatter(
        d["logfoldchanges"],
        d["neglog10_padj"],
        s=75,
        color=color_map[ct],
        edgecolor="black",
        linewidth=0.4,
        alpha=0.9,
        label=ct,
        zorder=3
    )

# =========================================================
# LABELS
# =========================================================

for _, row in plot_df.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    dx = 0.04 if x > 0 else -0.04

    ax.text(
        x + dx,
        y + 0.3,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLDS
# =========================================================

ax.axvline(
    LFC,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-3,3)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    "MSL Remodeling Programs",
    fontsize=10,
    weight="bold"
)

# =========================================================
# LEGEND
# =========================================================

ax.legend(
    frameon=False,
    fontsize=8
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/MSL_combined_volcano.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/MSL_combined_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "mesenchymal_leiden"

# change if needed
GROUP1 = "MSL1"

COND_COL = "disease_state"

# your panel genes
panel_genes = list(vasc_adata.var_names)

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = mesenchymalf[
    (mesenchymalf.obs[CELLTYPE_COL] == GROUP1) &
    (mesenchymalf.obs[COND_COL].isin(["Control", "T2D"]))
].copy()

print(ad)

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    ad,
    groupby=COND_COL,
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    ad,
    group="T2D"
)

# =========================================================
# FILTER TO PANEL GENES
# =========================================================

res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# REMOVE NAN
# =========================================================

res = res.dropna(
    subset=["logfoldchanges", "pvals_adj"]
)

# =========================================================
# VOLCANO METRICS
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# SIGNIFICANCE
# =========================================================

res["sig"] = (
    (res["pvals_adj"] < 0.05) &
    (np.abs(res["logfoldchanges"]) > 0.25)
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(4,4))

# all genes
ax.scatter(
    res["logfoldchanges"],
    res["neglog10_padj"],
    s=20,
    color="lightgray",
    linewidth=0
)

# significant
sig = res[res["sig"]]

ax.scatter(
    sig["logfoldchanges"],
    sig["neglog10_padj"],
    s=24,
    color="red",
    linewidth=0
)

# =========================================================
# LABEL TOP GENES
# =========================================================

top = res.sort_values(
    "pvals_adj"
).head(20)

for _, row in top.iterrows():

    ax.text(
        row["logfoldchanges"],
        row["neglog10_padj"],
        row["names"],
        fontsize=6
    )

# =========================================================
# THRESHOLD LINES
# =========================================================

ax.axvline(
    0.25,
    linestyle="--",
    linewidth=1,
    color="black"
)

ax.axvline(
    -0.25,
    linestyle="--",
    linewidth=1,
    color="black"
)

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    linewidth=1,
    color="black"
)

# =========================================================
# LABELS
# =========================================================

ax.set_xlabel("Log fold change (T2D vs Control)")
ax.set_ylabel("-log10 adjusted p-value")

ax.set_title(
    f"{GROUP1}: T2D vs Control",
    fontsize=9
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_T2D_vs_NvD_volcano.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_T2D_vs_NControlD_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT TABLE
# =========================================================

res.to_csv(
    f"/Users/kmeneses/Desktop/{GROUP1}_T2D_vs_ND_DControlGE.csv",
    index=False
)

print(res.head())

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================================================
# SETTINGS

# volcano thresholds
LFC_THRESH = 0.25
PADJ_THRESH = 0.05
CELLTYPE_COL = "mesenchymal_leiden"

# change if needed
GROUP1 = "MSL1"

COND_COL = "disease_state"

# your panel genes
panel_genes = list(vasc_adata.var_names)

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = mesenchymalf[
    (mesenchymalf.obs[CELLTYPE_COL] == GROUP1) &
    (mesenchymalf.obs[COND_COL].isin(["Control", "T2D"]))
].copy()

# =========================================================
# STYLE
# =========================================================



# =========================================================
# FILTER LOW DETECTION GENES
# =========================================================

sc.pp.filter_genes(
    ad,
    min_cells=10
)

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    ad,
    groupby=COND_COL,
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    ad,
    group="T2D"
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=["logfoldchanges", "pvals_adj"]
)

# remove infinite values
res = res[
    np.isfinite(res["logfoldchanges"])
]

# =========================================================
# OPTIONAL: REMOVE EXTREME FC OUTLIERS
# =========================================================

res = res[
    np.abs(res["logfoldchanges"]) < 5
]

# =========================================================
# METRICS
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# significance
res["sig"] = (
    (res["pvals_adj"] < PADJ_THRESH) &
    (np.abs(res["logfoldchanges"]) > LFC_THRESH)
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(4.5,4.5))

# nonsig
nonsig = res[~res["sig"]]

ax.scatter(
    nonsig["logfoldchanges"],
    nonsig["neglog10_padj"],
    s=18,
    color="lightgray",
    linewidth=0,
    alpha=0.7
)

# sig
sig = res[res["sig"]]

ax.scatter(
    sig["logfoldchanges"],
    sig["neglog10_padj"],
    s=22,
    color="red",
    linewidth=0
)

# =========================================================
# LABEL TOP GENES
# =========================================================

top = sig.sort_values(
    "pvals_adj"
).head(15)

for _, row in top.iterrows():

    ax.text(
        row["logfoldchanges"] + 0.03,
        row["neglog10_padj"],
        row["names"],
        fontsize=6
    )

# =========================================================
# THRESHOLDS
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# LIMITS
# =========================================================

ax.set_xlim(-3, 3)

# =========================================================
# LABELS
# =========================================================

ax.set_xlabel("Log fold change (T2D vs ND)")
ax.set_ylabel("-log10 adjusted p-value")

ax.set_title(
    f"{GROUP1.upper()}: T2D vs Control",
    fontsize=9
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_volcano_clean.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_volcano_clean.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# ECM / PANEL GENES ONLY
# =========================================================

panel_genes = [
    g for g in vasc_adata.var_names
    if g in res["names"].values
]

# keep only genes present
panel_genes = [
    g for g in panel_genes
    if g in res["names"].values
]

# subset volcano dataframe
panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(4.5,4.5))

# background genes
ax.scatter(
    res["logfoldchanges"],
    res["neglog10_padj"],
    s=10,
    color="lightgray",
    alpha=0.4,
    linewidth=0
)

# panel genes
ax.scatter(
    panel_res["logfoldchanges"],
    panel_res["neglog10_padj"],
    s=28,
    color="red",
    linewidth=0
)

# =========================================================
# LABEL PANEL GENES
# =========================================================

for _, row in panel_res.iterrows():

    ax.text(
        row["logfoldchanges"] + 0.03,
        row["neglog10_padj"],
        row["names"],
        fontsize=6
    )

# =========================================================
# THRESHOLDS
# =========================================================

ax.axvline(
    0.25,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -0.25,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-3,3)

ax.set_xlabel("Log fold change (T2D vs ND)")
ax.set_ylabel("-log10 adjusted p-value")

ax.set_title(
    "MSL1: ECM / BM Panel Genes",
    fontsize=9
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_ECM_panel_volcano.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_ECM_panel_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT TABLE
# =========================================================

panel_res.sort_values(
    "pvals_adj"
).to_csv(
    "/Users/kmeneses/Desktop/MSL1_ECM_panel_DGE.csv",
    index=False
)

print(panel_res.sort_values("pvals_adj"))

In [ ]:
# =========================================================
# SHOW ONLY SIGNIFICANT GENES
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# PANEL GENES
# =========================================================

panel_genes = [
    g for g in vasc_adata.var_names
    if g in res["names"].values
]

# =========================================================
# SUBSET TO PANEL
# =========================================================

panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# SIGNIFICANT ONLY
# =========================================================

panel_res = panel_res[
    (panel_res["pvals_adj"] < PADJ_THRESH) &
    (np.abs(panel_res["logfoldchanges"]) > LFC_THRESH)
].copy()

# =========================================================
# SORT
# =========================================================

panel_res = panel_res.sort_values(
    "logfoldchanges"
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(5,5))

# ---------------------------------------------------------
# BACKGROUND = ALL GENES
# ---------------------------------------------------------

ax.scatter(
    res["logfoldchanges"],
    res["neglog10_padj"],
    s=8,
    color="lightgray",
    alpha=0.12,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# COLORS
# ---------------------------------------------------------

colors = [
    "royalblue" if x < 0 else "red"
    for x in panel_res["logfoldchanges"]
]

# ---------------------------------------------------------
# SIGNIFICANT GENES
# ---------------------------------------------------------

ax.scatter(
    panel_res["logfoldchanges"],
    panel_res["neglog10_padj"],
    s=70,
    color=colors,
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in panel_res.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    ax.text(
        x + (0.05 if x > 0 else -0.05),
        y + 0.3,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLDS
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-3,3)

ax.set_ylim(
    0,
    panel_res["neglog10_padj"].max() * 1.08
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    "MSL1 Significant Panel Genes",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(width=1)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_sig_panel_volcano.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_sig_panel_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

panel_res.to_csv(
    "/Users/kmeneses/Desktop/MSL1_sig_panel_DGE.csv",
    index=False
)

print(
    panel_res[
        ["names", "logfoldchanges", "pvals_adj"]
    ]
)

In [ ]:
# =========================================================
# ECM / BM PANEL
# ONLY SIGNIFICANT GENES
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# PANEL GENES
# =========================================================

panel_genes = [
    "COL4A1",
    "COL4A2",
    "COL1A1",
    "COL1A2",
    "COL3A1",
    "COL5A1",
    "COL5A2",
    "COL14A1",
    "COL18A1",
    "LAMB1",
    "LAMB2",
    "LAMC1",
    "LAMA4",
    "LAMA5",
    "HSPG2",
    "FN1",
    "VCAN",
    "POSTN",
    "SPARCL1",
    "LOXL4",
    "MGP",
    "SERPINF1",
    "FBN1",
    "CHST10",
    "RGS5",
    "CSPG4",
    "PDGFRB",
    "THBS1",
    "TINAGL1",
    "MFGE8",
    "ITGA6",
    "SDC4"
]

# =========================================================
# FILTER TO PANEL
# =========================================================

panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# SIGNIFICANT ONLY
# =========================================================

sig = panel_res[
    (panel_res["pvals_adj"] < PADJ_THRESH) &
    (np.abs(panel_res["logfoldchanges"]) > LFC_THRESH)
].copy()

# =========================================================
# SORT
# =========================================================

sig = sig.sort_values(
    "logfoldchanges"
)

print(sig[
    ["names", "logfoldchanges", "pvals_adj"]
])

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(4.5,4.5))

# ---------------------------------------------------------
# ALL GENES BACKGROUND
# ---------------------------------------------------------

ax.scatter(
    res["logfoldchanges"],
    res["neglog10_padj"],
    s=8,
    color="lightgray",
    alpha=0.15,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# SIGNIFICANT PANEL GENES
# ---------------------------------------------------------

colors = [
    "royalblue" if x < 0 else "red"
    for x in sig["logfoldchanges"]
]

ax.scatter(
    sig["logfoldchanges"],
    sig["neglog10_padj"],
    s=55,
    color=colors,
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in sig.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    ax.text(
        x + (0.04 if x > 0 else -0.04),
        y + 0.5,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLDS
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-2.5, 2.5)

ax.set_ylim(
    0,
    min(
        45,
        sig["neglog10_padj"].max() * 1.08
    )
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    "MSL1 Significant ECM / BM Changes",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(width=1)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_sig_ECM_volcano.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_sig_ECM_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# BETTER ECM / BM FILTERING
# KEEP ALL PANEL GENES THAT:
# 1) are significant OR
# 2) are strong effect size genes
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# EXPANDED ECM / BM / PERICYTE PANEL
# =========================================================
panel_genes = [
    g for g in vasc_adata.var_names
    if g in res["names"].values
]


# =========================================================
# KEEP PRESENT GENES
# =========================================================

panel_genes = [
    g for g in panel_genes
    if g in res["names"].values
]

panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# KEEP:
# SIGNIFICANT OR LARGE EFFECT
# =========================================================

plot_df = panel_res[
    (
        panel_res["pvals_adj"] < PADJ_THRESH
    ) |
    (
        np.abs(panel_res["logfoldchanges"]) > 0.4
    )
].copy()

# =========================================================
# SORT
# =========================================================

plot_df = plot_df.sort_values(
    "logfoldchanges"
)

print(
    plot_df[
        ["names", "logfoldchanges", "pvals_adj"]
    ]
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(5.2,5))

# ---------------------------------------------------------
# BACKGROUND
# ---------------------------------------------------------

ax.scatter(
    res["logfoldchanges"],
    res["neglog10_padj"],
    s=8,
    color="lightgray",
    alpha=0.12,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# COLOR BY DIRECTION
# ---------------------------------------------------------

colors = []

for x in plot_df["logfoldchanges"]:

    if x < 0:
        colors.append("royalblue")
    else:
        colors.append("red")

# ---------------------------------------------------------
# PANEL GENES
# ---------------------------------------------------------

ax.scatter(
    plot_df["logfoldchanges"],
    plot_df["neglog10_padj"],
    s=60,
    color=colors,
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in plot_df.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    dx = 0.05 if x > 0 else -0.05

    ax.text(
        x + dx,
        y + 0.4,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLDS
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-2.5, 2.5)

ax.set_ylim(
    0,
    min(
        45,
        plot_df["neglog10_padj"].max() * 1.08
    )
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    "MSL1 ECM / BM / Pericyte Remodeling",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(width=1)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_ECM_BM_remodeling.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/MSL1_ECM_BM_remodeling.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

plot_df.to_csv(
    "/Users/kmeneses/Desktop/MSL1_ECM_BM_remodeling.csv",
    index=False
)

In [ ]:
# =========================================================
# ALL GENES VOLCANO
# SHOW ONLY SIGNIFICANT GENES
# =========================================================

import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# THRESHOLDS
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# CLEAN RESULTS
# =========================================================

plot_df = res.copy()

plot_df = plot_df.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

plot_df = plot_df[
    np.isfinite(plot_df["logfoldchanges"])
]

# =========================================================
# REMOVE HOUSEKEEPING
# =========================================================

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-"
)

plot_df = plot_df[
    ~plot_df["names"].str.startswith(remove_prefixes)
]

# =========================================================
# ADD NEG LOG10
# =========================================================

plot_df["neglog10_padj"] = -np.log10(
    plot_df["pvals_adj"] + 1e-300
)

# =========================================================
# SIGNIFICANT GENES ONLY
# =========================================================

sig = plot_df[
    (plot_df["pvals_adj"] < PADJ_THRESH) &
    (np.abs(plot_df["logfoldchanges"]) > LFC_THRESH)
].copy()

# =========================================================
# SORT
# =========================================================

sig = sig.sort_values(
    "logfoldchanges"
)

# =========================================================
# PRINT TABLE
# =========================================================

print(
    sig[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(6,6)
)

# ---------------------------------------------------------
# BACKGROUND = ALL GENES
# ---------------------------------------------------------

ax.scatter(
    plot_df["logfoldchanges"],
    plot_df["neglog10_padj"],
    s=8,
    color="lightgray",
    alpha=0.10,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# COLORS
# ---------------------------------------------------------

colors = [
    "royalblue" if x < 0 else "red"
    for x in sig["logfoldchanges"]
]

# ---------------------------------------------------------
# SIGNIFICANT GENES
# ---------------------------------------------------------

ax.scatter(
    sig["logfoldchanges"],
    sig["neglog10_padj"],
    s=75,
    color=colors,
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in sig.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    ax.text(
        x + (0.05 if x > 0 else -0.05),
        y + 0.3,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLD LINES
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-3,3)

ax.set_ylim(
    0,
    sig["neglog10_padj"].max() * 1.08
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    f"{GROUP1.upper()} Significant Remodeling Genes",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(width=1)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_ALL_sig_volcano.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_ALL_sig_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

sig.to_csv(
    f"/Users/kmeneses/Desktop/{GROUP1}_ALL_sig_volcano.csv",
    index=False
)

In [ ]:
# =========================================================
# SETTINGS
# =========================================================

GROUP1 = "MSL2"

# =========================================================
# SUBSET
# =========================================================

ad = mesenchymalf[
    (mesenchymalf.obs[CELLTYPE_COL] == GROUP1) &
    (mesenchymalf.obs[COND_COL].isin(["Control", "T2D"]))
].copy()

# =========================================================
# FILTER LOW DETECTION GENES
# =========================================================

sc.pp.filter_genes(
    ad,
    min_cells=10
)

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    ad,
    groupby=COND_COL,
    groups=["T2D"],
    reference="Control",
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    ad,
    group="T2D"
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=["logfoldchanges", "pvals_adj"]
)

res = res[
    np.isfinite(res["logfoldchanges"])
]

res = res[
    np.abs(res["logfoldchanges"]) < 5
]

# =========================================================
# METRICS
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# THRESHOLDS
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# EXPANDED ECM / BM / PERICYTE PANEL
# =========================================================

panel_genes = [

    # BM collagens
    "COL4A1",
    "COL4A2",
    "COL4A3",

    # laminins
    "LAMA4",
    "LAMA5",
    "LAMB1",
    "LAMB2",
    "LAMC1",

    # BM structure
    "HSPG2",
    "NID1",
    "NID2",

    # fibrillar ECM
    "COL1A1",
    "COL1A2",
    "COL3A1",
    "COL5A1",
    "COL5A2",
    "COL6A1",
    "COL6A2",
    "COL6A3",
    "COL14A1",
    "COL18A1",

    # glycoproteins
    "FN1",
    "SPARCL1",
    "THBS1",
    "TINAGL1",
    "POSTN",
    "FBN1",
    "VCAN",
    "MFGE8",

    # pericyte / mural
    "RGS5",
    "CSPG4",
    "PDGFRB",
    "MCAM",
    "NOTCH3",
    "DES",
    "ACTA2",
    "TAGLN",
    "MYL9",
    "TPM2",

    # ECM remodeling
    "LOXL4",
    "ADAM9",
    "SERPINF1",
    "MMP2",

    # signaling / adhesion
    "ITGA6",
    "SDC4",
    "MGP",
    "C1R",
    "CHST10"
]

# =========================================================
# KEEP PRESENT GENES
# =========================================================

panel_genes = [
    g for g in panel_genes
    if g in res["names"].values
]

panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# KEEP:
# SIGNIFICANT OR LARGE EFFECT
# =========================================================

plot_df = panel_res[
    (
        panel_res["pvals_adj"] < PADJ_THRESH
    ) |
    (
        np.abs(panel_res["logfoldchanges"]) > 0.4
    )
].copy()

# =========================================================
# SORT
# =========================================================

plot_df = plot_df.sort_values(
    "logfoldchanges"
)

print(
    plot_df[
        ["names", "logfoldchanges", "pvals_adj"]
    ]
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(5.2,5))

# ---------------------------------------------------------
# BACKGROUND
# ---------------------------------------------------------

ax.scatter(
    res["logfoldchanges"],
    res["neglog10_padj"],
    s=8,
    color="lightgray",
    alpha=0.12,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# COLOR BY DIRECTION
# ---------------------------------------------------------

colors = []

for x in plot_df["logfoldchanges"]:

    if x < 0:
        colors.append("royalblue")
    else:
        colors.append("red")

# ---------------------------------------------------------
# PANEL GENES
# ---------------------------------------------------------

ax.scatter(
    plot_df["logfoldchanges"],
    plot_df["neglog10_padj"],
    s=60,
    color=colors,
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in plot_df.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    dx = 0.05 if x > 0 else -0.05

    ax.text(
        x + dx,
        y + 0.4,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLDS
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-2.5, 2.5)

ax.set_ylim(
    0,
    min(
        45,
        plot_df["neglog10_padj"].max() * 1.08
    )
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    "MSL2 ECM / BM / Pericyte Remodeling",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(width=1)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/MSL2_ECM_BM_remodeling.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/MSL2_ECM_BM_remodeling.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

plot_df.to_csv(
    "/Users/kmeneses/Desktop/MSL2_ECM_BM_remodeling.csv",
    index=False
)

In [ ]:
# =========================================================
# USE FULL 300-GENE PANEL
# FILTER TO ONLY PANEL GENES
# =========================================================

# assumes:
# - panel_genes_full = list of all 300 panel genes
# OR:
vasc_adata.var_names  
# =========================================================
# OPTION 1:
# USE ALL GENES IN THE CURRENT ANN DATA
# =========================================================

panel_genes = res["names"].tolist()

# =========================================================
# OPTION 2:
# IF YOU HAVE ORIGINAL PANEL LIST
# uncomment this instead
# =========================================================

# panel_genes = [
#     g for g in panel_genes_full
#     if g in res["names"].values
# ]

# =========================================================
# FILTER TO PANEL GENES
# =========================================================

panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# KEEP:
# SIGNIFICANT OR STRONG EFFECT
# =========================================================

plot_df = panel_res[
    (
        panel_res["pvals_adj"] < 0.05
    ) |
    (
        np.abs(panel_res["logfoldchanges"]) > 0.4
    )
].copy()

# =========================================================
# REMOVE RIBOSOMAL / MT / HOUSEKEEPING
# OPTIONAL BUT HELPS A LOT
# =========================================================

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-"
)

plot_df = plot_df[
    ~plot_df["names"].str.startswith(remove_prefixes)
]

# =========================================================
# SORT
# =========================================================

plot_df = plot_df.sort_values(
    "logfoldchanges"
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(5.5,5.5))

# ---------------------------------------------------------
# BACKGROUND
# ---------------------------------------------------------

ax.scatter(
    res["logfoldchanges"],
    res["neglog10_padj"],
    s=8,
    color="lightgray",
    alpha=0.10,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# COLOR BY DIRECTION
# ---------------------------------------------------------

colors = [
    "royalblue" if x < 0 else "red"
    for x in plot_df["logfoldchanges"]
]

# ---------------------------------------------------------
# HIGHLIGHT GENES
# ---------------------------------------------------------

ax.scatter(
    plot_df["logfoldchanges"],
    plot_df["neglog10_padj"],
    s=65,
    color=colors,
    edgecolor="black",
    linewidth=0.4,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in plot_df.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    ax.text(
        x + (0.05 if x > 0 else -0.05),
        y + 0.4,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLD LINES
# =========================================================

ax.axvline(
    0.25,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -0.25,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(0.05),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-3, 3)

ax.set_ylim(
    0,
    min(
        45,
        plot_df["neglog10_padj"].max() * 1.05
    )
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    f"{GROUP1.upper()} Remodeling Programs",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_300panel_volcano.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_300panel_volcano.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

plot_df.to_csv(
    f"/Users/kmeneses/Desktop/{GROUP1}_300panel_volcano.csv",
    index=False
)

print(plot_df[
    ["names", "logfoldchanges", "pvals_adj"]
])

In [ ]:
# =========================================================
# SIGNIFICANT GENES ONLY
# STRICT FILTER VERSION
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# USE FULL 300 GENE PANEL
# =========================================================

panel_genes = [
    g for g in vasc_adata.var_names
    if g in res["names"].values
]

# =========================================================
# FILTER PANEL GENES
# =========================================================

panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# CLEAN
# =========================================================

panel_res = panel_res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

panel_res = panel_res[
    np.isfinite(panel_res["logfoldchanges"])
]

# =========================================================
# REMOVE HOUSEKEEPING
# =========================================================

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-"
)

panel_res = panel_res[
    ~panel_res["names"].str.startswith(remove_prefixes)
]

# =========================================================
# ADD NEG LOG10
# =========================================================

panel_res["neglog10_padj"] = -np.log10(
    panel_res["pvals_adj"] + 1e-300
)

# =========================================================
# SIGNIFICANT ONLY
# =========================================================

sig = panel_res[
    (panel_res["pvals_adj"] < PADJ_THRESH) &
    (np.abs(panel_res["logfoldchanges"]) > LFC_THRESH)
].copy()

# =========================================================
# SORT
# =========================================================

sig = sig.sort_values(
    "logfoldchanges"
)

# =========================================================
# PRINT TABLE
# =========================================================

print(
    sig[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(5,5)
)

# ---------------------------------------------------------
# BACKGROUND
# ---------------------------------------------------------

ax.scatter(
    panel_res["logfoldchanges"],
    panel_res["neglog10_padj"],
    s=10,
    color="lightgray",
    alpha=0.10,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# COLORS
# ---------------------------------------------------------

colors = [
    "royalblue" if x < 0 else "red"
    for x in sig["logfoldchanges"]
]

# ---------------------------------------------------------
# SIGNIFICANT GENES
# ---------------------------------------------------------

ax.scatter(
    sig["logfoldchanges"],
    sig["neglog10_padj"],
    s=80,
    color=colors,
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in sig.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    ax.text(
        x + (0.06 if x > 0 else -0.06),
        y + 0.35,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLD LINES
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-3,3)

ax.set_ylim(
    0,
    sig["neglog10_padj"].max() * 1.08
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    f"{GROUP1.upper()} Significant 300-Gene Panel Changes",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(width=1)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_sig_only_300panel.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_sig_only_300panel.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

sig.to_csv(
    f"/Users/kmeneses/Desktop/{GROUP1}_sig_only_300panel.csv",
    index=False
)

In [ ]:
# =========================================================
# SIGNIFICANT GENES ONLY
# STRICT FILTER VERSION
# =========================================================

LFC_THRESH = 0.25
PADJ_THRESH = 0.05

# =========================================================
# USE FULL 300 GENE PANEL
# =========================================================

panel_genes = [
    g for g in vasc_adata.var_names
    if g in res["names"].values
]

# =========================================================
# FILTER PANEL GENES
# =========================================================

panel_res = res[
    res["names"].isin(panel_genes)
].copy()

# =========================================================
# CLEAN
# =========================================================

panel_res = panel_res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

panel_res = panel_res[
    np.isfinite(panel_res["logfoldchanges"])
]

# =========================================================
# REMOVE HOUSEKEEPING
# =========================================================

remove_prefixes = (
    "RPL",
    "RPS",
    "MT-"
)

panel_res = panel_res[
    ~panel_res["names"].str.startswith(remove_prefixes)
]

# =========================================================
# ADD NEG LOG10
# =========================================================

panel_res["neglog10_padj"] = -np.log10(
    panel_res["pvals_adj"] + 1e-300
)

# =========================================================
# SIGNIFICANT ONLY
# =========================================================

sig = panel_res[
    (panel_res["pvals_adj"] < PADJ_THRESH) &
    (np.abs(panel_res["logfoldchanges"]) > LFC_THRESH)
].copy()

# =========================================================
# SORT
# =========================================================

sig = sig.sort_values(
    "logfoldchanges"
)

# =========================================================
# PRINT TABLE
# =========================================================

print(
    sig[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(5,5)
)

# ---------------------------------------------------------
# BACKGROUND
# ---------------------------------------------------------

ax.scatter(
    panel_res["logfoldchanges"],
    panel_res["neglog10_padj"],
    s=10,
    color="lightgray",
    alpha=0.10,
    linewidth=0,
    rasterized=True
)

# ---------------------------------------------------------
# COLORS
# ---------------------------------------------------------

colors = [
    "royalblue" if x < 0 else "red"
    for x in sig["logfoldchanges"]
]

# ---------------------------------------------------------
# SIGNIFICANT GENES
# ---------------------------------------------------------

ax.scatter(
    sig["logfoldchanges"],
    sig["neglog10_padj"],
    s=80,
    color=colors,
    edgecolor="black",
    linewidth=0.5,
    zorder=3
)

# ---------------------------------------------------------
# LABELS
# ---------------------------------------------------------

for _, row in sig.iterrows():

    x = row["logfoldchanges"]
    y = row["neglog10_padj"]

    ax.text(
        x + (0.06 if x > 0 else -0.06),
        y + 0.35,
        row["names"],
        fontsize=6,
        ha="left" if x > 0 else "right"
    )

# =========================================================
# THRESHOLD LINES
# =========================================================

ax.axvline(
    LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axvline(
    -LFC_THRESH,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.axhline(
    -np.log10(PADJ_THRESH),
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlim(-3,3)

ax.set_ylim(
    0,
    sig["neglog10_padj"].max() * 1.08
)

ax.set_xlabel(
    "Log fold change (T2D vs ND)"
)

ax.set_ylabel(
    "-log10 adjusted p-value"
)

ax.set_title(
    f"{GROUP1.upper()} Significant 300-Gene Panel Changes",
    fontsize=9,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(width=1)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_sig_only_300panel.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{GROUP1}_sig_only_300panel.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

sig.to_csv(
    f"/Users/kmeneses/Desktop/{GROUP1}_sig_only_300panel.csv",
    index=False
)

In [ ]:
# =========================================================
# REVIEWER-LEVEL PATHWAY ANALYSIS
#
# TRUE GSVA / ssGSEA STYLE ANALYSIS
# USING PSEUDOBULK FULL TRANSCRIPTOME
#
# FOR:
# - MSL1 ND vs T2D
# - MSL2 ND vs T2D
#
# OUTPUTS:
# 1. donor pseudobulk matrix
# 2. GSVA-like pathway scores
# 3. pathway statistics
# 4. ranked heatmap
# 5. export tables
#
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.preprocessing import StandardScaler

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "mesenchymal_leiden"
COND_COL = "disease_state"
DONOR_COL = "donor_id"

GROUPS = ["MSL1", "MSL2"]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# FULL TRANSCRIPTOME PROGRAMS
# =========================================================

gene_sets = {

    # -----------------------------------------------------
    # STRESS / IEG
    # -----------------------------------------------------

    "Stress_IEG": [
        "JUN", "FOS", "JUNB", "ATF3",
        "EGR1", "DUSP1", "HSPA1A",
        "HSPB1"
    ],

    # -----------------------------------------------------
    # CONTRACTILE / MURAL
    # -----------------------------------------------------

    "Contractile_Mural": [
        "MYL9", "CALD1", "TPM2",
        "TAGLN", "ACTA2", "CNN1",
        "DES", "MCAM"
    ],

    # -----------------------------------------------------
    # BASEMENT MEMBRANE
    # -----------------------------------------------------

    "Vascular_BM": [
        "COL4A1", "COL4A2",
        "LAMB1", "LAMB2",
        "LAMC1", "HSPG2",
        "NID1", "NID2"
    ],

    # -----------------------------------------------------
    # FIBRILLAR ECM
    # -----------------------------------------------------

    "Fibrillar_ECM": [
        "COL1A1", "COL1A2",
        "COL3A1", "COL5A1",
        "COL5A2", "COL6A1",
        "COL6A2", "COL6A3",
        "FN1", "DCN", "LUM"
    ],

    # -----------------------------------------------------
    # ECM REMODELING
    # -----------------------------------------------------

    "ECM_Remodeling": [
        "THBS1", "ADAM9",
        "MMP2", "SERPINF1",
        "MFGE8", "SPARCL1",
        "FBN1", "POSTN"
    ],

    # -----------------------------------------------------
    # TGFb
    # -----------------------------------------------------

    "TGFb_Response": [
        "TGFB1", "TGFBR2",
        "CTGF", "SERPINE1",
        "THBS1", "COL1A1"
    ],

    # -----------------------------------------------------
    # HYPOXIA
    # -----------------------------------------------------

    "Hypoxia": [
        "HIF1A", "VEGFA",
        "ENO1", "LDHA",
        "BNIP3", "SLC2A1"
    ],

    # -----------------------------------------------------
    # INFLAMMATORY
    # -----------------------------------------------------

    "Inflammatory": [
        "IL6", "CXCL8",
        "CCL2", "ICAM1",
        "TNFAIP3", "NFKBIA"
    ]
}

# =========================================================
# RESULTS STORAGE
# =========================================================

all_results = []

all_heatmaps = []

# =========================================================
# LOOP THROUGH GROUPS
# =========================================================

for GROUP in GROUPS:

    print("\n================================================")
    print(GROUP)
    print("================================================")

    # =====================================================
    # SUBSET
    # =====================================================

    ad = mesenchymalf[
        (
            mesenchymalf.obs[CELLTYPE_COL] == GROUP
        )
        &
        (
            mesenchymalf.obs[COND_COL].isin(
                ["Control", "T2D"]
            )
        )
    ].copy()

    # =====================================================
    # EXTRACT EXPRESSION
    # =====================================================

    X = ad.X

    if not isinstance(X, np.ndarray):
        X = X.toarray()

    expr = pd.DataFrame(
        X,
        index=ad.obs_names,
        columns=ad.var_names
    )

    # =====================================================
    # ADD METADATA
    # =====================================================

    meta = ad.obs[
        [DONOR_COL, COND_COL]
    ].copy()

    # =====================================================
    # BUILD PSEUDOBULK
    # donor averages
    # =====================================================

    expr["donor"] = meta[DONOR_COL].values
    expr["condition"] = meta[COND_COL].values

    pseudobulk = (
        expr.groupby(
            ["donor", "condition"]
        )
        .mean()
    )

    print("\nPSEUDOBULK MATRIX:")
    print(pseudobulk.shape)

    # =====================================================
    # GSVA-LIKE ENRICHMENT
    # mean z-score across genes
    # =====================================================

    pathway_scores = pd.DataFrame(
        index=pseudobulk.index
    )

    for pathway, genes in gene_sets.items():

        keep = [
            g for g in genes
            if g in pseudobulk.columns
        ]

        if len(keep) < 2:

            print(f"Skipping {pathway}")
            continue

        # -------------------------------------------------
        # z-score genes across donors
        # -------------------------------------------------

        zmat = pseudobulk[keep].copy()

        zmat = pd.DataFrame(
            StandardScaler().fit_transform(zmat),
            index=zmat.index,
            columns=zmat.columns
        )

        # -------------------------------------------------
        # pathway score
        # -------------------------------------------------

        pathway_scores[pathway] = (
            zmat.mean(axis=1)
        )

    # =====================================================
    # RESET INDEX
    # =====================================================

    pathway_scores = pathway_scores.reset_index()

    # =====================================================
    # TEST EACH PATHWAY
    # =====================================================

    for pathway in gene_sets.keys():

        if pathway not in pathway_scores.columns:
            continue

        nd = pathway_scores[
            pathway_scores["condition"] == "ND"
        ][pathway]

        t2d = pathway_scores[
            pathway_scores["condition"] == "T2D"
        ][pathway]

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        delta = t2d.mean() - nd.mean()

        all_results.append({

            "group": GROUP,
            "pathway": pathway,
            "ND_mean": nd.mean(),
            "T2D_mean": t2d.mean(),
            "delta": delta,
            "pval": p
        })

    # =====================================================
    # SAVE HEATMAP MATRIX
    # =====================================================

    hm = pathway_scores.copy()

    hm["group"] = GROUP

    all_heatmaps.append(hm)

    # =====================================================
    # EXPORT PSEUDOBULK
    # =====================================================

    pseudobulk.to_csv(
        f"/Users/kmeneses/Desktop/{GROUP}_pseudobulk_matrix.csv"
    )

    pathway_scores.to_csv(
        f"/Users/kmeneses/Desktop/{GROUP}_GSVA_scores.csv",
        index=False
    )

# =========================================================
# COMBINE RESULTS
# =========================================================

res = pd.DataFrame(all_results)

# =========================================================
# FDR
# =========================================================

res["padj"] = multipletests(
    res["pval"],
    method="fdr_bh"
)[1]

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "delta"
)

# =========================================================
# EXPORT
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/GSVA_Pathway_Results.csv",
    index=False
)

print("\n================================================")
print("PATHWAY RESULTS")
print("================================================")

print(res)

# =========================================================
# VISUALIZATION
# =========================================================

plot_df = res.copy()

plot_df["label"] = (
    plot_df["group"]
    + " | " +
    plot_df["pathway"]
)

colors = [
    "firebrick" if x > 0
    else "royalblue"
    for x in plot_df["delta"]
]

# =========================================================
# BARPLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(7,5)
)

ax.barh(
    plot_df["label"],
    plot_df["delta"],
    color=colors,
    edgecolor="black",
    linewidth=0.5
)

# =========================================================
# SIGNIFICANCE
# =========================================================

for i, row in enumerate(
    plot_df.itertuples()
):

    if row.padj < 0.05:

        ax.text(
            row.delta,
            i,
            " *",
            fontsize=10,
            va="center"
        )

# =========================================================
# ZERO LINE
# =========================================================

ax.axvline(
    0,
    linestyle="--",
    color="black"
)

# =========================================================
# LABELS
# =========================================================

ax.set_xlabel(
    "T2D − ND pathway enrichment"
)

ax.set_title(
    "GSVA-like Pathway Remodeling",
    fontsize=10,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/GSVA_Pathway_Remodeling.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/GSVA_Pathway_Remodeling.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# HEATMAP
# =========================================================

hm = pd.concat(
    all_heatmaps,
    axis=0
)

# remove metadata cols
hm_plot = hm.drop(
    columns=["donor", "condition", "group"],
    errors="ignore"
)

# row labels
# =========================================================
# FIX STRING TYPES
# =========================================================

hm["group"] = hm["group"].astype(str)
hm["condition"] = hm["condition"].astype(str)
hm["donor"] = hm["donor"].astype(str)

# =========================================================
# ROW LABELS
# =========================================================

row_labels = (
    hm["group"]
    + "_"
    + hm["condition"]
    + "_"
    + hm["donor"]
)

hm_plot.index = row_labels

# =========================================================
# CLUSTERMAP
# =========================================================

cg = sns.clustermap(
    hm_plot,
    cmap="coolwarm",
    center=0,
    figsize=(8,8),
    linewidths=0.5
)

plt.savefig(
    "/Users/kmeneses/Desktop/GSVA_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/GSVA_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

print("\n✔ COMPLETE")

In [ ]:
"""
msl_pathway_analysis.py
=======================
Comprehensive pathway analysis for MSL1 and MSL2 — T2D vs Control
Full transcriptome analysis using:
  1. Pseudobulk DEG (t-test on log-CPM)
  2. GSEA prerank (on pseudobulk t-statistics)
  3. Enrichr ORA (Hallmark + Reactome + KEGG)
  4. Module scoring (sc.tl.score_genes — ssGSEA-like)
  5. GSVA-equivalent (per-cell scoring via scanpy)

REQUIREMENTS:
  pip install gseapy scanpy pandas numpy scipy matplotlib seaborn requests

USAGE:
  Edit SETTINGS block below, then:
  python msl_pathway_analysis.py
  OR run cell-by-cell in Jupyter
"""

# ══════════════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import scanpy as sc
import gseapy as gp
import scipy.sparse as sp
import requests
from scipy import stats
from scipy.stats import false_discovery_control
from matplotlib.patches import Patch
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# SETTINGS — EDIT THESE
# ══════════════════════════════════════════════════════════════════════════════
ADATA        = "mesenchymalf"      # your AnnData object name (already in memory)
CELLTYPE_COL = "mesenchymal_leiden"
DISEASE_COL  = "disease_state"
DONOR_COL    = "donor_id"
CLUSTERS     = ["MSL1", "MSL2"]
CONDITION_T2D     = "T2D"
CONDITION_CONTROL = "Control"
OUT_DIR      = "/Users/kmeneses/Desktop/MSL_Analysis"
MIN_CELLS    = 10                  # min cells per donor for pseudobulk

# style
mpl_params = {
    "font.family": "Arial",
    "font.size": 8,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
}
plt.rcParams.update(mpl_params)
UP_COLOR   = "#c45b8a"
DOWN_COLOR = "#8b5e3c"

Path(OUT_DIR).mkdir(exist_ok=True)
print(f"Output directory: {OUT_DIR}")

# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def shorten(s, n=45):
    return s[:n] + "..." if len(s) > n else s

def save_fig(fig, name, out_dir=OUT_DIR):
    for ext in ["png", "pdf", "svg"]:
        fig.savefig(f"{out_dir}/{name}.{ext}",
                    dpi=300 if ext == "png" else None,
                    bbox_inches="tight")
    print(f"  Saved: {name}")

def plot_bar(ax, df, title, up_col=UP_COLOR, dn_col=DOWN_COLOR,
             hatch_col=None, n_show=10):
    """Generic horizontal bar plot for enrichment results."""
    df = df.copy()
    # take top/bottom n
    up = df[df["score"] > 0].nlargest(n_show, "score")
    dn = df[df["score"] < 0].nsmallest(n_show, "score")
    plot_df = pd.concat([dn, up]).sort_values("score")
    plot_df["Term"] = plot_df["Term"].apply(shorten)

    colors = [up_col if s > 0 else dn_col for s in plot_df["score"]]
    bars = ax.barh(plot_df["Term"], plot_df["score"],
                   color=colors, edgecolor="none", height=0.7)

    if hatch_col and "source" in plot_df.columns:
        for bar, (_, row) in zip(bars, plot_df.iterrows()):
            if row.get("source") == hatch_col:
                bar.set_hatch("////")
                bar.set_edgecolor("white")
                bar.set_linewidth(0.8)

    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=8, fontweight="bold", pad=5)
    ax.set_xlabel("Score", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.spines[["top", "right"]].set_visible(False)
    return plot_df

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — PSEUDOBULK DEG
# ══════════════════════════════════════════════════════════════════════════════

def make_pseudobulk(adata, donor_col, condition_col, min_cells=10):
    """Sum raw counts per donor → CPM → log1p."""
    donors = adata.obs[donor_col].unique()
    pb_counts, pb_meta = [], []

    for donor in donors:
        mask = (adata.obs[donor_col] == donor).values
        ncells = mask.sum()
        if ncells < min_cells:
            continue
        X = adata.X[mask]
        if sp.issparse(X):
            X = X.toarray()
        counts = X.sum(axis=0).astype(int)
        pb_counts.append(counts)
        cond = adata.obs.loc[adata.obs[donor_col] == donor,
                             condition_col].iloc[0]
        pb_meta.append({"donor": donor, "condition": cond,
                        "n_cells": int(ncells)})

    counts_df = pd.DataFrame(np.array(pb_counts),
                             columns=adata.var_names,
                             index=[m["donor"] for m in pb_meta])
    meta_df = pd.DataFrame(pb_meta).set_index("donor")
    return counts_df, meta_df


def run_pseudobulk_deg(counts_df, meta_df,
                       condition_t2d="T2D", condition_ctrl="Control"):
    """t-test on log-CPM pseudobulk."""
    cpm = counts_df.div(counts_df.sum(axis=1), axis=0) * 1e6
    log_cpm = np.log1p(cpm)

    nd_d  = meta_df[meta_df["condition"] == condition_ctrl].index.tolist()
    t2d_d = meta_df[meta_df["condition"] == condition_t2d].index.tolist()
    print(f"    Control donors: {len(nd_d)}, T2D donors: {len(t2d_d)}")

    results = []
    for gene in log_cpm.columns:
        nv = log_cpm.loc[nd_d,  gene].values
        tv = log_cpm.loc[t2d_d, gene].values
        if nv.std() == 0 and tv.std() == 0:
            continue
        stat, pval = stats.ttest_ind(tv, nv)
        results.append({"gene": gene,
                        "log2FC": tv.mean() - nv.mean(),
                        "pval": pval,
                        "t_stat": stat})

    res = pd.DataFrame(results)
    res["padj"] = false_discovery_control(res["pval"].fillna(1))
    return res.sort_values("t_stat", ascending=False).reset_index(drop=True)


# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — GENE SET DOWNLOAD
# ══════════════════════════════════════════════════════════════════════════════

def download_library(name):
    """Download gene set library from Enrichr."""
    url = (f"https://maayanlab.cloud/Enrichr/geneSetLibrary"
           f"?mode=text&libraryName={name}")
    r = requests.get(url, timeout=60)
    lib = {}
    for line in r.text.strip().split("\n"):
        parts = line.split("\t")
        term  = parts[0]
        genes = [g for g in parts[2:] if g]
        if genes:
            lib[term] = genes
    print(f"  {name}: {len(lib)} gene sets")
    return lib


print("\n[1/5] Downloading gene sets...")
hallmark  = download_library("MSigDB_Hallmark_2020")
reactome  = download_library("Reactome_Pathways_2024")
kegg      = download_library("KEGG_2021_Human")


# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — RUN ALL ANALYSES PER CLUSTER
# ══════════════════════════════════════════════════════════════════════════════

results_store = {}  # store all results

for CLUSTER in CLUSTERS:
    print(f"\n{'='*60}")
    print(f"  CLUSTER: {CLUSTER}")
    print(f"{'='*60}")

    # ── subset ────────────────────────────────────────────────────
    adata_sub = mesenchymalf[
        mesenchymalf.obs[CELLTYPE_COL] == CLUSTER
    ].copy()
    print(f"  Cells: {adata_sub.n_obs}")
    print(adata_sub.obs[DISEASE_COL].value_counts().to_string())

    cluster_results = {}

    # ── 3A: PSEUDOBULK DEG ────────────────────────────────────────
    print(f"\n  [A] Pseudobulk DEG...")
    counts_df, meta_df = make_pseudobulk(
        adata_sub, DONOR_COL, DISEASE_COL, MIN_CELLS)
    print(f"  Pseudobulk matrix: {counts_df.shape}")

    deg_pb = run_pseudobulk_deg(counts_df, meta_df,
                                CONDITION_T2D, CONDITION_CONTROL)

    # nominal significant genes for enrichment
    nom = deg_pb[(deg_pb["pval"] < 0.05) & (deg_pb["log2FC"].abs() > 0.25)]
    up_genes = nom[nom["log2FC"] > 0]["gene"].tolist()
    dn_genes = nom[nom["log2FC"] < 0]["gene"].tolist()
    print(f"  Nominal DEGs — Up: {len(up_genes)}, Down: {len(dn_genes)}")

    deg_pb.to_csv(f"{OUT_DIR}/{CLUSTER}_pseudobulk_DEG.csv", index=False)
    cluster_results["deg_pb"] = deg_pb
    cluster_results["up_genes"] = up_genes
    cluster_results["dn_genes"] = dn_genes

    # volcano
    fig, ax = plt.subplots(figsize=(5, 4))
    sig_up = (deg_pb["pval"] < 0.05) & (deg_pb["log2FC"] > 0.25)
    sig_dn = (deg_pb["pval"] < 0.05) & (deg_pb["log2FC"] < -0.25)
    ns     = ~sig_up & ~sig_dn
    ax.scatter(deg_pb.loc[ns, "log2FC"],
               -np.log10(deg_pb.loc[ns, "pval"] + 1e-300),
               c="lightgrey", s=3, alpha=0.5, rasterized=True)
    ax.scatter(deg_pb.loc[sig_up, "log2FC"],
               -np.log10(deg_pb.loc[sig_up, "pval"] + 1e-300),
               c=UP_COLOR, s=5, alpha=0.7, rasterized=True, label=f"Up ({sig_up.sum()})")
    ax.scatter(deg_pb.loc[sig_dn, "log2FC"],
               -np.log10(deg_pb.loc[sig_dn, "pval"] + 1e-300),
               c=DOWN_COLOR, s=5, alpha=0.7, rasterized=True, label=f"Down ({sig_dn.sum()})")
    for _, row in deg_pb.head(8).iterrows():
        ax.text(row["log2FC"], -np.log10(row["pval"] + 1e-300) + 0.1,
                row["gene"], fontsize=5)
    ax.axvline( 0.25, color="grey", linewidth=0.5, linestyle="--")
    ax.axvline(-0.25, color="grey", linewidth=0.5, linestyle="--")
    ax.axhline(-np.log10(0.05), color="grey", linewidth=0.5, linestyle="--")
    ax.set_xlabel("log2FC (T2D - Control)", fontsize=8)
    ax.set_ylabel("-log10(p-value)", fontsize=8)
    ax.set_title(f"{CLUSTER} Pseudobulk DEG", fontsize=9, fontweight="bold")
    ax.legend(fontsize=6, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    save_fig(fig, f"{CLUSTER}_volcano")
    plt.close()

    # ── 3B: GSEA PRERANK ──────────────────────────────────────────
    print(f"\n  [B] GSEA prerank (Hallmark + Reactome)...")
    rnk = (deg_pb[["gene", "t_stat"]]
           .drop_duplicates("gene")
           .set_index("gene"))
    rnk.index.name = None

    gsea_results = {}
    for gs_name, gs_lib in [("Hallmark",  hallmark),
                              ("Reactome",  reactome),
                              ("KEGG",      kegg)]:
        try:
            enr = gp.prerank(
                rnk=rnk, gene_sets=gs_lib,
                min_size=5, max_size=500,
                permutation_num=1000, seed=1,
                outdir=None, verbose=False
            )
            res = enr.res2d.copy().sort_values("NES", ascending=False)
            sig = res[res["FDR q-val"] < 0.25]
            print(f"    {gs_name}: {len(sig)} sig terms (FDR<0.25)")
            gsea_results[gs_name] = res
            res.to_csv(f"{OUT_DIR}/{CLUSTER}_GSEA_{gs_name}.csv", index=False)
        except Exception as e:
            print(f"    {gs_name} failed: {e}")

    cluster_results["gsea"] = gsea_results

    # GSEA plot — Hallmark
    if "Hallmark" in gsea_results:
        res = gsea_results["Hallmark"]
        sig = res[res["FDR q-val"] < 0.25]
        if len(sig) > 0:
            top_up = sig[sig["NES"] > 0].head(8)
            top_dn = sig[sig["NES"] < 0].tail(8)
            plot_df = pd.concat([top_dn, top_up]).copy()
            plot_df["Term"] = plot_df["Term"].apply(shorten)
            plot_df["score"] = plot_df["NES"]
            colors = [UP_COLOR if n > 0 else DOWN_COLOR for n in plot_df["NES"]]
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.barh(plot_df["Term"][::-1], plot_df["NES"][::-1],
                    color=colors[::-1], edgecolor="none", height=0.7)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_xlabel("NES", fontsize=8)
            ax.set_title(f"{CLUSTER} GSEA — Hallmark (FDR<0.25)",
                         fontsize=9, fontweight="bold")
            ax.spines[["top", "right"]].set_visible(False)
            ax.tick_params(labelsize=7)
            plt.tight_layout()
            save_fig(fig, f"{CLUSTER}_GSEA_Hallmark")
            plt.close()

    # ── 3C: ENRICHR ORA ───────────────────────────────────────────
    print(f"\n  [C] Enrichr ORA (Hallmark + Reactome + KEGG)...")
    ora_results = {}
    for gs_name in ["MSigDB_Hallmark_2020",
                    "Reactome_Pathways_2024",
                    "KEGG_2021_Human"]:
        short = gs_name.split("_")[0]
        try:
            eu = gp.enrichr(gene_list=up_genes, gene_sets=gs_name,
                            organism="Human", outdir=None)
            ed = gp.enrichr(gene_list=dn_genes, gene_sets=gs_name,
                            organism="Human", outdir=None)
            ru = eu.results[eu.results["Adjusted P-value"] < 0.05].copy()
            rd = ed.results[ed.results["Adjusted P-value"] < 0.05].copy()
            for df in [ru, rd]:
                df["overlap_n"] = df["Overlap"].apply(
                    lambda x: int(x.split("/")[0]))
            ru = ru[ru["overlap_n"] >= 3]
            rd = rd[rd["overlap_n"] >= 3]
            print(f"    {short}: {len(ru)} up, {len(rd)} down")
            ora_results[short] = {"up": ru, "down": rd}
            ru.to_csv(f"{OUT_DIR}/{CLUSTER}_ORA_{short}_UP.csv", index=False)
            rd.to_csv(f"{OUT_DIR}/{CLUSTER}_ORA_{short}_DN.csv", index=False)
        except Exception as e:
            print(f"    {short} failed: {e}")

    cluster_results["ora"] = ora_results

    # ORA plot — combined Hallmark + Reactome
    noise = ["SARS","Cobalamin","Dietary","L13a","Ribosomal Scanning",
             "Prostate","Salmonella","Hepatitis","Glomerulosclerosis"]
    panels = []
    for gs in ["Hallmark","Reactome"]:
        if gs not in ora_results:
            continue
        ru = ora_results[gs]["up"].head(6).copy()
        rd = ora_results[gs]["down"].head(6).copy()
        ru["source"] = gs; rd["source"] = gs
        ru["score"] =  -np.log10(ru["Adjusted P-value"])
        rd["score"] = -(-np.log10(rd["Adjusted P-value"]))
        panels.append(pd.concat([rd, ru]))

    if panels:
        df_comb = pd.concat(panels)
        df_comb["Term"] = df_comb["Term"].apply(shorten)
        df_comb = (df_comb
                   [~df_comb["Term"].str.contains("|".join(noise), case=False)]
                   .drop_duplicates("Term")
                   .sort_values("score"))

        fig, ax = plt.subplots(figsize=(9, 6))
        colors  = [UP_COLOR if s > 0 else DOWN_COLOR for s in df_comb["score"]]
        bars = ax.barh(df_comb["Term"], df_comb["score"],
                       color=colors, edgecolor="none", height=0.7)
        for bar, (_, row) in zip(bars, df_comb.iterrows()):
            if row["source"] == "Reactome":
                bar.set_hatch("////")
                bar.set_edgecolor("white")
                bar.set_linewidth(0.8)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlabel("-log10(adj p-value)", fontsize=8)
        ax.set_title(f"{CLUSTER}: T2D vs Control\nORA — Hallmark + Reactome",
                     fontsize=9, fontweight="bold")
        ax.tick_params(labelsize=7)
        ax.spines[["top", "right"]].set_visible(False)
        color_leg = [Patch(facecolor=UP_COLOR,   label="Up in T2D"),
                     Patch(facecolor=DOWN_COLOR, label="Down in T2D")]
        hatch_leg = [mpatches.Patch(facecolor="grey", label="Hallmark"),
                     mpatches.Patch(facecolor="grey", hatch="////",
                                    edgecolor="white", label="Reactome")]
        leg1 = ax.legend(handles=color_leg, fontsize=7, frameon=False,
                         loc="lower right")
        ax.add_artist(leg1)
        ax.legend(handles=hatch_leg, fontsize=7, frameon=False,
                  loc="upper right")
        plt.tight_layout()
        save_fig(fig, f"{CLUSTER}_ORA_Hallmark_Reactome")
        plt.close()

    # ── 3D: MODULE SCORING (ssGSEA-like) ──────────────────────────
    print(f"\n  [D] Module scoring (sc.tl.score_genes)...")

    all_genesets = {}
    all_genesets.update({f"H_{k}": v for k, v in hallmark.items()})
    all_genesets.update({f"R_{k}": v for k, v in reactome.items()
                         if any(g in adata_sub.var_names for g in v[:5])})

    scored = []
    for term, genes in all_genesets.items():
        overlap = [g for g in genes if g in adata_sub.var_names]
        if len(overlap) < 3:
            continue
        safe_name = term[:40].replace(" ", "_").replace("/", "_")
        try:
            sc.tl.score_genes(adata_sub, gene_list=overlap,
                              score_name=safe_name, use_raw=False)
            scored.append(safe_name)
        except Exception:
            pass

    print(f"  Scored {len(scored)} gene sets")

    # compare scores T2D vs Control
    score_df = adata_sub.obs[scored + [DISEASE_COL]].copy()
    means = score_df.groupby(DISEASE_COL)[scored].mean()
    if CONDITION_T2D in means.index and CONDITION_CONTROL in means.index:
        diff_scores = (means.loc[CONDITION_T2D] -
                       means.loc[CONDITION_CONTROL]).sort_values(ascending=False)

        # top 15 up + down
        plot_diff = pd.concat([diff_scores.tail(10), diff_scores.head(10)])
        plot_diff.index = [i.replace("H_","[H] ").replace("R_","[R] ")
                           for i in plot_diff.index]
        colors = [UP_COLOR if v > 0 else DOWN_COLOR for v in plot_diff]

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(plot_diff.index, plot_diff.values,
                color=colors, edgecolor="none", height=0.7)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlabel("Mean score (T2D - Control)", fontsize=8)
        ax.set_title(f"{CLUSTER}: Module Scoring\n[H]=Hallmark [R]=Reactome",
                     fontsize=9, fontweight="bold")
        ax.tick_params(labelsize=6.5)
        ax.spines[["top", "right"]].set_visible(False)
        plt.tight_layout()
        save_fig(fig, f"{CLUSTER}_module_scores")
        plt.close()

        diff_scores.to_csv(f"{OUT_DIR}/{CLUSTER}_module_scores.csv")
        cluster_results["module_scores"] = diff_scores

    results_store[CLUSTER] = cluster_results
    print(f"\n  ✔ {CLUSTER} complete")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — COMPARISON FIGURE: MSL1 vs MSL2
# ══════════════════════════════════════════════════════════════════════════════
print("\n[5/5] Generating comparison figures...")

if len(CLUSTERS) == 2:
    C1, C2 = CLUSTERS

    # ── ORA comparison ────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                             gridspec_kw={"wspace": 0.5})

    for ax, cl in zip(axes, [C1, C2]):
        ora = results_store[cl].get("ora", {})
        panels = []
        for gs in ["Hallmark", "Reactome"]:
            if gs not in ora:
                continue
            ru = ora[gs]["up"].head(6).copy()
            rd = ora[gs]["down"].head(6).copy()
            ru["source"] = gs; rd["source"] = gs
            ru["score"] =  -np.log10(ru["Adjusted P-value"])
            rd["score"] = -(-np.log10(rd["Adjusted P-value"]))
            panels.append(pd.concat([rd, ru]))

        if not panels:
            continue

        df_c = pd.concat(panels)
        df_c["Term"] = df_c["Term"].apply(shorten)
        df_c = (df_c
                [~df_c["Term"].str.contains(
                    "|".join(noise), case=False)]
                .drop_duplicates("Term")
                .sort_values("score"))

        colors  = [UP_COLOR if s > 0 else DOWN_COLOR for s in df_c["score"]]
        bars = ax.barh(df_c["Term"], df_c["score"],
                       color=colors, edgecolor="none", height=0.7)
        for bar, (_, row) in zip(bars, df_c.iterrows()):
            if row["source"] == "Reactome":
                bar.set_hatch("////")
                bar.set_edgecolor("white")
                bar.set_linewidth(0.8)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(f"{cl}: T2D vs Control", fontsize=9, fontweight="bold")
        ax.set_xlabel("-log10(adj p-value)", fontsize=7)
        ax.tick_params(labelsize=6.5)
        ax.spines[["top", "right"]].set_visible(False)

    color_leg = [Patch(facecolor=UP_COLOR,   label="Up in T2D"),
                 Patch(facecolor=DOWN_COLOR, label="Down in T2D")]
    hatch_leg = [mpatches.Patch(facecolor="grey", label="Hallmark"),
                 mpatches.Patch(facecolor="grey", hatch="////",
                                edgecolor="white", label="Reactome")]
    leg1 = fig.legend(handles=color_leg, fontsize=7, frameon=False,
                      loc="lower left",  bbox_to_anchor=(0.01, -0.04), ncol=2)
    fig.legend(handles=hatch_leg, fontsize=7, frameon=False,
               loc="lower right", bbox_to_anchor=(0.99, -0.04), ncol=2)
    fig.add_artist(leg1)
    fig.suptitle("Pathway Enrichment: MSL1 vs MSL2\n"
                 "MSigDB Hallmark (solid) · Reactome (hatched)",
                 fontsize=10, fontweight="bold")
    save_fig(fig, "MSL1_MSL2_ORA_comparison")
    plt.close()

    # ── Module score comparison heatmap ───────────────────────────
    if all("module_scores" in results_store[c] for c in CLUSTERS):
        sc1 = results_store[C1]["module_scores"].rename(C1)
        sc2 = results_store[C2]["module_scores"].rename(C2)
        combined = pd.concat([sc1, sc2], axis=1).dropna()

        # top 20 most variable across clusters
        combined["range"] = combined.max(axis=1) - combined.min(axis=1)
        top20 = combined.nlargest(20, "range").drop(columns="range")
        top20.index = [i.replace("H_","[H] ").replace("R_","[R] ")
                       for i in top20.index]

        fig, ax = plt.subplots(figsize=(5, 8))
        sns.heatmap(top20, cmap="RdBu_r", center=0,
                    linewidths=0.5, linecolor="white",
                    ax=ax, cbar_kws={"label": "Mean score (T2D-Control)",
                                     "shrink": 0.5})
        ax.set_title("Module Score: T2D vs Control\nMSL1 vs MSL2",
                     fontsize=9, fontweight="bold")
        ax.tick_params(axis="x", labelsize=8)
        ax.tick_params(axis="y", labelsize=6.5, rotation=0)
        plt.tight_layout()
        save_fig(fig, "MSL1_MSL2_module_heatmap")
        plt.close()

print(f"\n{'='*60}")
print(f"✔ ALL ANALYSES COMPLETE")
print(f"  Results saved to: {OUT_DIR}")
print(f"{'='*60}")
print("\nFiles generated:")
for f in sorted(Path(OUT_DIR).glob("*")):
    print(f"  {f.name}")


In [ ]:
# fix for pyarrow index out of bounds
# replace the GSEA section in the script with this:

print(f"\n  [B] GSEA prerank (Hallmark + Reactome + KEGG)...")

# reset index to avoid pyarrow bug
rnk = (deg_pb[["gene", "t_stat"]]
       .drop_duplicates("gene")
       .sort_values("t_stat", ascending=False)
       .reset_index(drop=True)          # ← key fix
       .set_index("gene"))
rnk.index.name = None

# convert to plain dict for gseapy — bypasses pyarrow entirely  
rnk_dict = rnk["t_stat"].to_dict()
rnk_df   = pd.DataFrame(list(rnk_dict.items()),
                         columns=["gene","score"])

gsea_results = {}
for gs_name, gs_lib in [("Hallmark", hallmark),
                          ("Reactome", reactome),
                          ("KEGG",     kegg)]:
    try:
        enr = gp.prerank(
            rnk=rnk_df,               # plain DataFrame, no index
            gene_sets=gs_lib,
            min_size=5,
            max_size=500,
            permutation_num=1000,
            seed=1,
            outdir=None,
            verbose=False
        )
        res = enr.res2d.copy().sort_values("NES", ascending=False)
        sig = res[res["FDR q-val"] < 0.25]
        print(f"    {gs_name}: {len(sig)} sig terms (FDR<0.25)")
        print(sig[["Term","NES","FDR q-val"]].head(5).to_string())
        gsea_results[gs_name] = res
        res.to_csv(f"{OUT_DIR}/{CLUSTER}_GSEA_{gs_name}.csv", index=False)
    except Exception as e:
        print(f"    {gs_name} failed: {e}")

cluster_results["gsea"] = gsea_results

In [ ]:
# replace the comparison heatmap section with this
if all("module_scores" in results_store[c] for c in CLUSTERS):
    sc1 = results_store[C1]["module_scores"].rename(C1)
    sc2 = results_store[C2]["module_scores"].rename(C2)

    # fix duplicate index
    sc1 = sc1[~sc1.index.duplicated(keep="first")]
    sc2 = sc2[~sc2.index.duplicated(keep="first")]

    combined = pd.concat([sc1, sc2], axis=1).dropna()
    print(f"Combined module scores: {combined.shape}")

    # top 20 most variable
    combined["range"] = combined.max(axis=1) - combined.min(axis=1)
    top20 = combined.nlargest(20, "range").drop(columns="range")
    top20.index = [i.replace("H_","[H] ").replace("R_","[R] ")
                   for i in top20.index]

    fig, ax = plt.subplots(figsize=(5, 8))
    sns.heatmap(top20, cmap="RdBu_r", center=0,
                linewidths=0.5, linecolor="white",
                ax=ax,
                cbar_kws={"label": "Mean score (T2D-Control)",
                          "shrink": 0.5})
    ax.set_title("Module Score: T2D vs Control\nMSL1 vs MSL2",
                 fontsize=9, fontweight="bold")
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=6.5, rotation=0)
    plt.tight_layout()
    save_fig(fig, "MSL1_MSL2_module_heatmap")
    print("✔ Heatmap saved")

In [ ]:
if all("module_scores" in results_store[c] for c in CLUSTERS):
    sc1 = results_store[C1]["module_scores"].rename(C1)
    sc2 = results_store[C2]["module_scores"].rename(C2)
    sc1 = sc1[~sc1.index.duplicated(keep="first")]
    sc2 = sc2[~sc2.index.duplicated(keep="first")]
    combined = pd.concat([sc1, sc2], axis=1).dropna()

    # ── keep only biologically relevant terms ──────────────────────
    keep_keywords = [
        "EMT", "Epithelial_Mesenchymal",
        "Collagen", "Extracellular_Matrix", "ECM", "Fibril",
        "MMP", "Matrix_Metalloprotein",
        "Hypoxia", "Angiogenesis",
        "TGF", "TGFB", "TNF", "NF.kB", "NFkB",
        "mTORC1", "MTORC1", "PI3K", "AKT",
        "Myc_Target", "G2M", "E2F", "Cell_Cycle",
        "Apoptosis", "p53",
        "Oxidative_Phosphorylation", "OxPhos",
        "Reactive_Oxygen", "ROS",
        "Inflammatory", "Interferon",
        "Adipogenesis", "Fatty_Acid",
        "Unfolded_Protein", "UPR",
        "PDGF", "VEGF", "FGF",
        "Wnt", "Notch", "Hedgehog",
        "Pancreas", "Beta_Cell", "Insulin",
        "Translation", "Ribosom",
        "Autophagy", "Mitochondri",
        "Hallmark"  # catches all [H] terms
    ]

    pattern = "|".join(keep_keywords)
    filtered = combined[
        combined.index.str.contains(pattern, case=False, regex=True)
    ]
    print(f"Filtered to {len(filtered)} relevant pathways")

    # sort by MSL2 score (your main cluster of interest)
    filtered = filtered.sort_values(C2, ascending=True)

    # clean index labels
    filtered.index = (filtered.index
        .str.replace("H_", "[H] ")
        .str.replace("R_", "[R] ")
        .str.replace("_", " "))

    fig, ax = plt.subplots(figsize=(5, max(6, len(filtered)*0.35)))
    sns.heatmap(filtered, cmap="RdBu_r", center=0,
                linewidths=0.5, linecolor="white",
                ax=ax,
                cbar_kws={"label": "Mean score (T2D-Control)",
                          "shrink": 0.5})
    ax.set_title("Module Score: T2D vs Control\nMSL1 vs MSL2",
                 fontsize=9, fontweight="bold")
    ax.tick_params(axis="x", labelsize=8, rotation=0)
    ax.tick_params(axis="y", labelsize=6.5, rotation=0)
    plt.tight_layout()
    save_fig(fig, "MSL1_MSL2_module_heatmap_filtered")
    print("✔ Filtered heatmap saved")

In [ ]:
# ── find pathways with biggest MSL1 vs MSL2 difference ────────────
combined_clean = combined.copy()
combined_clean.index = (combined_clean.index
    .str.replace("H_", "[H] ")
    .str.replace("R_", "[R] ")
    .str.replace("_", " "))

# score = absolute difference between clusters
combined_clean["diff"]     = combined_clean[C2] - combined_clean[C1]
combined_clean["abs_diff"] = combined_clean["diff"].abs()

# also require at least one cluster shows meaningful signal
combined_clean["max_abs"]  = combined_clean[[C1,C2]].abs().max(axis=1)

# filter: big difference AND meaningful signal
curated = combined_clean[
    (combined_clean["abs_diff"] > 0.003) &
    (combined_clean["max_abs"]  > 0.003)
].copy()

print(f"Pathways with clear MSL1 vs MSL2 difference: {len(curated)}")

# ── also filter by keywords to keep relevant biology ───────────────
keep = [
    "EMT", "Epithelial Mesenchymal",
    "Collagen", "Extracellular Matrix", "ECM", "Fibril", "MMP",
    "Hypoxia", "Angiogenesis", "VEGF",
    "TGF", "TNF", "NF", "Inflammatory", "Interferon", "Complement",
    "mTORC1", "MTORC1", "PI3K", "AKT", "KRAS",
    "Myc", "G2M", "E2F", "Cell Cycle", "Mitotic", "Checkpoint",
    "Apoptosis", "p53", "DNA Repair",
    "Oxidative Phosphorylation", "Reactive Oxygen", "ROS",
    "Fatty Acid", "Adipogenesis", "Metabolism", "Glycolysis",
    "Unfolded Protein", "UPR", "Autophagy",
    "PDGF", "FGF", "Notch", "Wnt", "Hedgehog",
    "Pancreas", "Beta Cell", "Insulin",
    "Translation", "Ribosom", "Mitochondri",
    "Coagulation", "Complement", "Protein Secretion",
    "Hallmark"
]
pattern = "|".join(keep)
curated = curated[curated.index.str.contains(pattern, case=False)]
print(f"After keyword filter: {len(curated)}")

# ── sort: MSL2-high at top, MSL1-high at bottom ────────────────────
curated = curated.sort_values("diff", ascending=False)

# top 15 higher in MSL2, top 15 higher in MSL1
top_msl2 = curated[curated["diff"] > 0].head(15)
top_msl1 = curated[curated["diff"] < 0].tail(15)
plot_df  = pd.concat([top_msl1, top_msl2])[[C1, C2]].copy()

print("\nHigher in MSL2 vs MSL1:")
print(top_msl2[[C1, C2, "diff"]].to_string())
print("\nHigher in MSL1 vs MSL2:")
print(top_msl1[[C1, C2, "diff"]].to_string())

# ── heatmap ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(4, len(plot_df) * 0.38 + 1.5))

# add divider between MSL1-enriched and MSL2-enriched
divider = len(top_msl1)

sns.heatmap(plot_df, cmap="RdBu_r", center=0,
            linewidths=0.5, linecolor="white",
            ax=ax,
            cbar_kws={"label": "Mean score\n(T2D - Control)",
                      "shrink": 0.5})

# divider line
ax.axhline(divider, color="black", linewidth=1.5, linestyle="--")

# section labels
ax.text(-0.02, divider/2,        "MSL1\nenriched",
        transform=ax.get_yaxis_transform(),
        ha="right", va="center", fontsize=6,
        color=DOWN_COLOR, fontweight="bold")
ax.text(-0.02, divider + len(top_msl2)/2, "MSL2\nenriched",
        transform=ax.get_yaxis_transform(),
        ha="right", va="center", fontsize=6,
        color=UP_COLOR, fontweight="bold")

ax.set_title("Module Scores: MSL1 vs MSL2\nT2D vs Control",
             fontsize=9, fontweight="bold")
ax.tick_params(axis="x", labelsize=8, rotation=0)
ax.tick_params(axis="y", labelsize=6.5, rotation=0)
ax.set_ylabel("")

plt.tight_layout()
save_fig(fig, "MSL1_MSL2_module_heatmap_curated")
print("✔ Curated heatmap saved")

In [ ]:
# match pathway_order against combined_clean index
matched = []
for term in pathway_order:
    hits = [idx for idx in combined_clean.index
            if term.lower() in idx.lower()]
    if hits:
        matched.append(hits[0])
        print(f"✔ {term} → {hits[0]}")
    else:
        print(f"✗ NOT FOUND: {term}")

plot_df = combined_clean.loc[matched, [C1, C2]].copy()

# clean labels
plot_df.index = [i.replace("[H] ","").replace("[R] ","")
                 for i in plot_df.index]

# ── plot ───────────────────────────────────────────────────────────
vabs = np.abs(plot_df.values).max()

fig, ax = plt.subplots(figsize=(3.5, len(plot_df) * 0.38 + 1.5))
sns.heatmap(plot_df,
            cmap="RdBu_r",
            center=0,
            vmin=-vabs, vmax=vabs,
            linewidths=0.5,
            linecolor="white",
            ax=ax,
            cbar_kws={"label": "Mean score\n(T2D - Control)",
                      "shrink": 0.45,
                      "aspect": 12})

ax.set_title("Module Scores: MSL1 vs MSL2\nT2D vs Control",
             fontsize=9, fontweight="bold", pad=8)
ax.tick_params(axis="x", labelsize=8, rotation=0)
ax.tick_params(axis="y", labelsize=7,  rotation=0)
ax.set_ylabel("")
ax.set_xlabel("")

plt.tight_layout()
save_fig(fig, "MSL1_MSL2_final_curated")
plt.show()
print("✔ Done")

In [ ]:
import seaborn as sns
import numpy as np

pathway_order = [
    "Collagen Formation",
    "Metabolism of Angiotensinogen to Angio",
    "Collagen Chain Trimerization",
    "Assembly of Collagen Fibrils and Other",
    "Crosslinking of Collagen Fibrils",
    "Anchoring Fibril Formation",
    "Collagen Degradation",
    "ECM Proteoglycans",
    "Regulation of Apoptosis",
    "FGFR1b Ligand Binding and Activation",
    "Non-integrin membrane-ECM Interactions",
    "TGFBR3 PTM Regulation",
    "Epithelial Mesenchymal Transition"
]

C1, C2 = "MSL1", "MSL2"

matched = []
for term in pathway_order:
    hits = [idx for idx in combined_clean.index
            if term.lower() in idx.lower()]
    if hits:
        matched.append(hits[0])
        print(f"✔ {term}")
    else:
        print(f"✗ NOT FOUND: {term}")

plot_df = combined_clean.loc[matched, [C1, C2]].copy()
plot_df.index = [i.replace("[H] ","").replace("[R] ","")
                 for i in plot_df.index]

vabs = np.abs(plot_df.values).max()

fig, ax = plt.subplots(figsize=(3.5, len(plot_df)*0.38 + 1.5))
sns.heatmap(plot_df,
            cmap="RdBu_r", center=0,
            vmin=-vabs, vmax=vabs,
            linewidths=0.5, linecolor="white",
            ax=ax,
            cbar_kws={"label": "Mean score\n(T2D - Control)",
                      "shrink": 0.45, "aspect": 12})
ax.set_title("Module Scores: MSL1 vs MSL2\nT2D vs Control",
             fontsize=9, fontweight="bold", pad=8)
ax.tick_params(axis="x", labelsize=8, rotation=0)
ax.tick_params(axis="y", labelsize=7,  rotation=0)
ax.set_ylabel(""); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/MSL1_MSL2_final_curated.png", dpi=300, bbox_inches="tight")
plt.savefig(f"{OUT_DIR}/MSL1_MSL2_final_curated.pdf", bbox_inches="tight")
plt.show()
print("✔ Done")

In [ ]:
# =========================================================
# COMPARE 4 VISUALIZATION STYLES
# SAME ECM PATHWAYS
# =========================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import pandas as pd
import numpy as np

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# PATHWAYS
# =========================================================

pathway_order = [
    "Collagen Formation",
    "Collagen Biosynthesis and Modifying Enzymes",
    "Collagen Chain Trimerization",
    "Assembly of Collagen Fibrils and Other",
    "Crosslinking of Collagen Fibrils",
    "Anchoring Fibril Formation",
    "Collagen Degradation",
    "ECM Proteoglycans",
    "Regulation of Apoptosis",
    "FGFR1b Ligand Binding and Activation",
    "Non-integrin membrane-ECM Interactions",
    "TGFBR3 PTM Regulation",
    "Epithelial Mesenchymal Transition"
]

C1, C2 = "MSL1", "MSL2"

# =========================================================
# MATCH PATHWAYS
# =========================================================

matched = []

for term in pathway_order:

    hits = [
        idx for idx in combined_clean.index
        if term.lower() in idx.lower()
    ]

    if hits:
        matched.append(hits[0])
        print(f"✔ {term}")

    else:
        print(f"✗ NOT FOUND: {term}")

# =========================================================
# BUILD MATRIX
# =========================================================

plot_df = combined_clean.loc[
    matched,
    [C1, C2]
].copy()

plot_df.index = [
    i.replace("[H] ","").replace("[R] ","")
    for i in plot_df.index
]

vabs = np.abs(plot_df.values).max()

OUT_DIR = "/Users/kmeneses/Desktop"

# =========================================================
# 1. HEATMAP
# =========================================================

fig, ax = plt.subplots(
    figsize=(3.5, len(plot_df)*0.38 + 1.5)
)

sns.heatmap(
    plot_df,
    cmap="RdBu_r",
    center=0,
    vmin=-vabs,
    vmax=vabs,
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    cbar_kws={
        "label": "Mean score\n(T2D - Control)",
        "shrink": 0.45,
        "aspect": 12
    }
)

ax.set_title(
    "Heatmap",
    fontsize=10,
    fontweight="bold"
)

ax.tick_params(
    axis="x",
    rotation=0
)

ax.tick_params(
    axis="y",
    labelsize=7
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()

plt.savefig(
    f"{OUT_DIR}/COMPARE_heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# 2. DOTPLOT
# =========================================================

dot_df = plot_df.reset_index().melt(
    id_vars="index",
    var_name="Group",
    value_name="Score"
)

dot_df.columns = [
    "Pathway",
    "Group",
    "Score"
]

fig, ax = plt.subplots(
    figsize=(3.5, len(plot_df)*0.38 + 1.5)
)

sc = ax.scatter(
    x=dot_df["Group"],
    y=dot_df["Pathway"],
    c=dot_df["Score"],
    s=np.abs(dot_df["Score"]) * 5000 + 50,
    cmap="RdBu_r",
    vmin=-vabs,
    vmax=vabs,
    edgecolor="black",
    linewidth=0.5
)

cbar = plt.colorbar(
    sc,
    ax=ax,
    shrink=0.5
)

cbar.set_label(
    "Mean score\n(T2D - Control)"
)

ax.set_title(
    "Dotplot",
    fontsize=10,
    fontweight="bold"
)

ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()

plt.savefig(
    f"{OUT_DIR}/COMPARE_dotplot.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# 3. SLOPE PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(4.5, len(plot_df)*0.38 + 1)
)

ypos = np.arange(len(plot_df))

for i, pathway in enumerate(plot_df.index):

    x1 = plot_df.loc[pathway, C1]
    x2 = plot_df.loc[pathway, C2]

    ax.plot(
        [x1, x2],
        [i, i],
        color="gray",
        linewidth=1.5,
        zorder=1
    )

    ax.scatter(
        x1,
        i,
        color="royalblue",
        s=45,
        edgecolor="black",
        linewidth=0.4,
        zorder=3
    )

    ax.scatter(
        x2,
        i,
        color="red",
        s=45,
        edgecolor="black",
        linewidth=0.4,
        zorder=3
    )

ax.axvline(
    0,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.set_yticks(ypos)

ax.set_yticklabels(
    plot_df.index,
    fontsize=7
)

ax.set_xlabel(
    "Mean score (T2D - Control)"
)

ax.set_title(
    "Slope Plot",
    fontsize=10,
    fontweight="bold"
)

ax.text(
    plot_df[C1].min() - 0.01,
    len(plot_df)+0.3,
    "MSL1",
    color="royalblue",
    fontsize=9,
    fontweight="bold"
)

ax.text(
    plot_df[C2].max() - 0.005,
    len(plot_df)+0.3,
    "MSL2",
    color="red",
    fontsize=9,
    fontweight="bold"
)

sns.despine()

plt.tight_layout()

plt.savefig(
    f"{OUT_DIR}/COMPARE_slopeplot.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# 4. DIVERGING BARPLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(4.5, len(plot_df)*0.38 + 1)
)

y = np.arange(len(plot_df))

ax.barh(
    y,
    plot_df[C1],
    color="royalblue",
    label="MSL1"
)

ax.barh(
    y,
    plot_df[C2],
    color="red",
    label="MSL2"
)

ax.axvline(
    0,
    linestyle="--",
    color="black",
    linewidth=1
)

ax.set_yticks(y)

ax.set_yticklabels(
    plot_df.index,
    fontsize=7
)

ax.set_xlabel(
    "Mean score (T2D - Control)"
)

ax.set_title(
    "Diverging Barplot",
    fontsize=10,
    fontweight="bold"
)

ax.legend(
    frameon=False
)

sns.despine()

plt.tight_layout()

plt.savefig(
    f"{OUT_DIR}/COMPARE_barplot.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("✔ ALL 4 PLOT TYPES COMPLETE")

# SECTION 15 — LOAD + MERGE SPATIAL OBJECTS
 Loads ND MERSCOPE and T2D MERSCOPE AnnData objects
 Tags each with Condition label (ND / T2D)
 Concatenates into adata_all for cross-disease analysis


In [ ]:
adata_ND=sc.read_h5ad('/Volumes/Samsung_4TB/051720206/ND_merscopeROIinfo.h5ad')

In [ ]:
for sample in adata_all.obs["Sample"].unique():
    
    print(f"\n── {sample} ──────────────────────────")
    
    mask     = adata_all.obs["Sample"] == sample
    cell_ids = adata_all.obs_names[mask]
    
    for i, cid in enumerate(cell_ids, 1):
        print(f"  {i:>4}  {cid}")
    
    print(f"  total: {mask.sum()} cells")

In [ ]:
for sample in adata_all.obs["Sample"].unique():
    mask = adata_all.obs["Sample"] == sample
    print(f"{sample}  →  {mask.sum()} cells")
    print(adata_all.obs_names[mask].tolist())

In [ ]:
adata_ND

In [ ]:
nd_adata = adata_ND.copy()
t2d_adata = adata.copy()

nd_adata.obs["Condition"] = "ND"
t2d_adata.obs["Condition"] = "T2D"

# =========================================
# COMBINE
# =========================================
adata_all = sc.concat([nd_adata, t2d_adata])


In [ ]:
adata_all

In [ ]:
# =====================================================
# SUBSET VASCULAR CELLS
# =====================================================

VASC_TYPES = [
    "Pericytes",
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

vasc_adata = adata_all[
    adata_all.obs["cell_type_final"]
    .isin(VASC_TYPES)
].copy()

# =====================================================
# CHECK
# =====================================================

print(vasc_adata)

print(
    vasc_adata.obs["cell_type_final"]
    .value_counts()
)

print(
    vasc_adata.obs["Condition"]
    .value_counts()
)

In [ ]:
# check your merscope data
# what's the object called? common names:
print(vasc_adata)  # or adata_spatial, spatial_adata etc

# check structure
print(vasc_adata.obs.columns.tolist())  # sample, disease, cell type cols
print(vasc_adata.obsm.keys())           # should have 'spatial'
print(vasc_adata.obs["Sample"].value_counts())  # 5 samples

In [ ]:
vasc_adata.write('/Users/kmeneses/Desktop/New_U/allvasc_cells.h5ad')

In [ ]:
# Set device to cpu or to gpu (if your torch has been set up correctly to use GPU), for mac you can use either torch.device('mps') or torch.device('cpu')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Initialize Concord with an AnnData object, skip input_feature to use all features, set preload_dense=False if your data is very large
# Provide 'domain_key' if integrating across batches, see below
cur_ccd = ccd.Concord(adata=vasc_adata, input_feature=None, domain_key='Sample', device=device, preload_dense=True) 

# Encode data, saving the latent embedding in adata.obsm['Concord']
cur_ccd.fit_transform(output_key='Concord')

In [ ]:
ccd.ul.run_umap(vasc_adata, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine')

# Plot the UMAP embeddings
color_by = ['Sample', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    vasc_adata, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
marker_genes_dict = {
    "B-cell": ["CD79A", "CD19"],
    "Macrophages": ["CCR3", "CXCR3", "C1QC"],
    "Cadherins": ["CDH1", "CDH5"],
    "Beta Cells": ["IAPP", "GCK", "PDX1", "MAFA", 'UCN3', 'GLP1R', 'MAFB', 'NEUROD1', 'NKX6-1', 'PAX4', 'SOX1', 'SDC4'],
    "Pericytes": ["CSPG4", "PDGFRB", "SYP", 'ACE2', 'RGS5'],
    "Endothelial": ['PLVAP', 'PECAM1', 'VWF'],
    "Ductal Cells": ["CFTR", "KRT19", "DCDC2", 'PROM1', 'SOX9', 'SPP1', 'TUBB3', 'VAT1L'], 
    "Endocrine Markers": ['CHGA', 'UCHL1', 'ISL1', 'CCDC186', 'SIMC1', 'NOS1', 'ERLIN1'], 
    "Alpha Cells": ['GCG', 'BEND4', 'IRX1', 'IRX2', 'LOXL4', 'RPL14'], 
    "Delta Cells": ['SST', 'SSTR2', 'HHEX', 'BMI1', 'VIP', 'RSPH1'], 
    "Epsilon Cells": ['GHR', 'LGALS3BP', 'POGK', 'SERGEF', 'VTN'], 
    "PPY Cells": ['PPY', 'PAX6', 'SERTM1'], 
    "Nerve Cells": ['CBX2', 'CEACAM1', 'MNX1', 'NCAM1', 'NPY']
}

In [ ]:
sc.pp.neighbors(vasc_adata, use_rep='Concord')
sc.tl.leiden(vasc_adata, resolution=1.0, key_added='concord_leiden')

In [ ]:
sc.pl.embedding(
    vasc_adata,
    basis='Concord_UMAP',
    color=['concord_leiden', 'RGS5', 'Health_Status', 'Location' ],
    frameon=False,
    ncols=2,
    vmax=5
)

In [ ]:
vasc_adata.obs['concord_leiden'].value_counts()

In [ ]:
sc.tl.rank_genes_groups(vasc_adata, "concord_leiden", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(vasc_adata, n_genes=20)

In [ ]:
sc.pl.dotplot(vasc_adata, marker_genes_dict, groupby='concord_leiden', vmax=10)

In [ ]:


def merge_clusters(adata, cluster_column, merge_dict, new_column_name="merged_clusters"):
    # Create a new column with original cluster labels
    adata.obs[new_column_name] = adata.obs[cluster_column].astype(str)

    # Apply merging rules
    for old_clusters, new_label in merge_dict.items():
        adata.obs.loc[adata.obs[cluster_column].isin(old_clusters), new_column_name] = new_label

    return adata


In [ ]:
import scanpy as sc

def rename_cell_categories(adata, column, rename_dict):
    """
    Rename categories in a specified column of an AnnData object.

    Parameters:
    - adata: AnnData object.
    - column: str, the column in adata.obs that contains categorical labels.
    - rename_dict: dict, mapping of old category names to new names.

    Returns:
    - Updated AnnData object with renamed categories.
    """
    if column in adata.obs:
        adata.obs[column] = adata.obs[column].astype("category")  # Ensure categorical dtype
        adata.obs[column] = adata.obs[column].cat.rename_categories(rename_dict)
    else:
        print(f"Column '{column}' not found in adata.obs.")
    
    return adata


In [ ]:
endo_peri_refined_markers = {
"ECs": ["PECAM1", "PLVAP", "VWF"],
"Pericytes": ["CSPG4", "PDGFRB", 'RGS5', 'DCN', "LUM"]
}

In [ ]:
vasc_adataf = vasc_adata[
    ~vasc_adata.obs["concord_leiden"].isin(["7", "13", "3", "15", "8"])
].copy()

In [ ]:
sc.pl.dotplot(vasc_adataf, endo_peri_refined_markers, groupby='concord_leiden', vmax=10)

In [ ]:
sc.tl.rank_genes_groups(vasc_adataf, "concord_leiden", use_raw=False, method="wilcoxon")
sc.pl.rank_genes_groups(vasc_adataf, n_genes=20)

In [ ]:
# Define clusters to merge
merge_map = {
    ('1', '2', '4', '11', '12'): 'Endothelial Cells',
    ('5', '9', '14'): 'Pericytes',
    ('0', '6'): 'Fibroblasts'
}

# Merge clusters
vasc_adataf = merge_clusters(
    vasc_adataf,
    cluster_column="concord_leiden",
    merge_dict=merge_map,
    new_column_name="merged_cell_type"
)

# Check result
print(vasc_adataf.obs["merged_cell_type"].value_counts())

In [ ]:


# Example usage:
rename_dict = {
    "10": "Islet-associated Fibroblasts"
}

vasc_adataf = rename_cell_categories(vasc_adataf, column="merged_cell_type", rename_dict=rename_dict)
# Check renamed categories
print(vasc_adataf.obs["merged_cell_type"].cat.categories)

In [ ]:
sc.pl.embedding(
    vasc_adataf,
    basis='Concord_UMAP',
    color=['Health_Status', 'Sample', 'Location', 'merged_cell_type' ],
    frameon=False,
    ncols=2,
)

In [ ]:
ECM = ['SFN', 'COL4A3', 'COL18A1', 'COL15A1', 'COL5A3', 'COL6A3', 'COL6A2', 'COL5A2', 'COL1A1', 'COL1A2', 'COL6A1', 'COL3A1', 
       'LGALS4', 'LOXL4', 'SFRP2', 'SPARCL1', 'IGFBP7', 'AGPS', 'AGRN', 'CLASRP', 'MGP', 'AMBP', 'HSPG2', 
       'F13A1', 'ASPN', 'LAMC3', 'TINAGL1', 'VWA1', 'PCOLCE', 'MFGE8', 'FBLN2', 
       'LAMB2', 'FBN1', 'LAMA2', 'THBS1', 'VTN', 'FN1', 'FGG', 'FGB', 'FGA', 'LAMA5', 'COL16A1', 'COL11A1', 
       'COL4A2', 'COL4A1', 'COL2A1', 'OGN', 'EMILIN1', 'LAMA4', 'POSTN', 'FBN2', 'COL12A1', 'COL14A1', 'COL5A1', 'LAMC1', 'LAMC2',  'LAMB3', 'FBLN1',
        'LAMB1', 'LAMB4', 'LAMA3', 'SDC1']

In [ ]:
ccd.ul.run_umap(vasc_adataf, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine')

# Plot the UMAP embeddings
color_by = ['Sample', 'Location'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    vasc_adataf, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
import pandas as pd
import numpy as np

# ── Disable Arrow string inference BEFORE anything ───────
pd.options.future.infer_string = False

def strip_arrow(df):
    data = {}
    for col in df.columns:
        s = df[col]
        if hasattr(s, "cat"):
            codes = s.cat.codes.to_numpy()
            cats  = np.array(s.cat.categories.tolist(), dtype=object)
            data[col] = pd.Categorical.from_codes(
                codes, categories=cats, ordered=s.cat.ordered
            )
        else:
            arr = s.to_numpy()
            # Force to plain object array if string-like
            if arr.dtype.kind in ("U", "S") or arr.dtype == object:
                arr = np.array(arr.tolist(), dtype=object)
            data[col] = arr

    new_index = np.array(df.index.tolist(), dtype=object)
    return pd.DataFrame(data, index=new_index)

vasc_adataf.obs = strip_arrow(vasc_adataf.obs)
vasc_adataf.var = strip_arrow(vasc_adataf.var)

# Verify — should see no ArrowStringArray
print("Remaining Arrow columns:")
for col in vasc_adataf.obs.columns:
    s = vasc_adataf.obs[col]
    vals = s.cat.categories._values if hasattr(s, "cat") else s._values
    if "Arrow" in type(vals).__name__:
        print(" STILL ARROW:", col, type(vals))
print("obs index type:", type(vasc_adataf.obs.index._values))

vasc_adataf.write_h5ad(
    "/Users/kmeneses/Desktop/New_U/allvasc_cells.h5ad"
)
print("✔ saved")

In [ ]:
import pandas as pd
import numpy as np
import anndata as ad

pd.options.future.infer_string = False

def strip_arrow(df):
    data = {}
    for col in df.columns:
        s = df[col]
        if hasattr(s, "cat"):
            codes = s.cat.codes.to_numpy()
            cats  = np.array(s.cat.categories.tolist(), dtype=object)
            data[col] = pd.Categorical.from_codes(
                codes, categories=cats, ordered=s.cat.ordered
            )
        else:
            arr = s.to_numpy()
            if arr.dtype.kind in ("U", "S") or arr.dtype == object:
                arr = np.array(arr.tolist(), dtype=object)
            data[col] = arr
    new_index = np.array(df.index.tolist(), dtype=object)
    return pd.DataFrame(data, index=new_index)

# ── Fix obs / var ─────────────────────────────────────────
adata_all.obs = strip_arrow(adata_all.obs)
adata_all.var = strip_arrow(adata_all.var)

# ── Fix raw slot ──────────────────────────────────────────
if adata_all.raw is not None:
    clean_raw_var = strip_arrow(adata_all.raw.var)
    tmp = ad.AnnData(
        X=adata_all.raw.X,
        var=clean_raw_var,
        obs=adata_all.obs,   # obs must match shape
    )
    adata_all.raw = tmp

# ── Verify nothing left ───────────────────────────────────
for slot_name, df in [("obs", adata_all.obs), ("var", adata_all.var)]:
    for col in df.columns:
        s = df[col]
        vals = s.cat.categories._values if hasattr(s, "cat") else s._values
        if "Arrow" in type(vals).__name__:
            print(f"STILL ARROW in {slot_name}: {col}")

if adata_all.raw is not None:
    for col in adata_all.raw.var.columns:
        s = adata_all.raw.var[col]
        vals = s.cat.categories._values if hasattr(s, "cat") else s._values
        if "Arrow" in type(vals).__name__:
            print(f"STILL ARROW in raw.var: {col}")
    idx_type = type(adata_all.raw.var.index._values).__name__
    if "Arrow" in idx_type:
        print(f"STILL ARROW in raw.var index: {idx_type}")

print("Verification done — writing...")
adata_all.write_h5ad("/Users/kmeneses/Desktop/New_U/alladata_all_cells.h5ad")
print("✔ saved")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (4, 4)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    vasc_adataf,
    basis="Concord_UMAP",
    color=['Sample', 'Location', 'Health_Status', 'merged_cell_type'],
    ncols=2,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/vascall_umap"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

plt.rcParams["figure.figsize"] = (4, 4)

# =========================
# CUSTOM CELL TYPE COLORS
# =========================

vasc_adataf.uns["merged_cell_type_colors"] = [

    "#FFA500",  # orange
    "#FF69B4",  # pink
    "#32CD32",  # green
    "#00FFFF"   # cyan
]

# IMPORTANT:
# order follows:
print(
    vasc_adataf.obs["merged_cell_type"]
    .cat.categories
)

# =========================
# PLOT
# =========================

fig = sc.pl.embedding(
    vasc_adataf,
    basis="Concord_UMAP",
    color=[
        'Sample',
        'Location',
        'Health_Status',
        'merged_cell_type'
    ],
    ncols=2,
    size=10,
    wspace=0.1,
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True
)

# =========================
# RASTERIZE POINTS
# =========================

for ax in fig.axes:

    for coll in ax.collections:

        coll.set_rasterized(True)

# =========================
# CLEAN AXES
# =========================

for ax in fig.axes:

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.title.set_fontweight("normal")

    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")

    for spine in ax.spines.values():

        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================

out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/vascall_umap"

fig.savefig(
    f"{out_base}.pdf",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal",
    "axes.titleweight": "normal",
    "axes.labelweight": "normal",
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

plt.rcParams["figure.figsize"] = (4, 4)

# =========================
# SET CATEGORY ORDER
# IMPORTANT
# =========================

print("\nmerged_cell_type order:")
print(
    vasc_adataf.obs["merged_cell_type"]
    .cat.categories
)

print("\nLocation order:")
print(
    vasc_adataf.obs["Location"]
    .cat.categories
)

print("\nHealth_Status order:")
print(
    vasc_adataf.obs["Health_Status"]
    .cat.categories
)

# =========================
# CUSTOM COLORS
# =========================

# ---------------------------------
# merged_cell_type
# order MUST match categories
# ---------------------------------

vasc_adataf.uns["merged_cell_type_colors"] = [

    "#FFA500",  # orange
    "#32CD32",  # pink
    "#00FFFF",  # green
    "#FF69B4"   # cyan
]

# ---------------------------------
# Location
# example:
# Exocrine = gray
# Islet = red
# ---------------------------------

vasc_adataf.uns["Location_colors"] = [

    "#F6F615",  # Exocrine
    "#0E6D74"   # Islet
]

# ---------------------------------
# Health_Status
# example:
# ND = blue
# T2D = dark red
# ---------------------------------

vasc_adataf.uns["Health_Status_colors"] = [
    "#6B5327",      # dark purple
"#E378EB"  
]

# =========================
# PLOT
# =========================

fig = sc.pl.embedding(
    vasc_adataf,
    basis="Concord_UMAP",
    color=[
        "Sample",
        "Location",
        "Health_Status",
        "merged_cell_type"
    ],
    ncols=2,
    size=10,
    wspace=0.1,
    frameon=True,
    legend_loc="right",
    show=False,
    return_fig=True
)

# =========================
# RASTERIZE
# =========================

for ax in fig.axes:

    for coll in ax.collections:

        coll.set_rasterized(True)

# =========================
# CLEAN AXES
# =========================

for ax in fig.axes:

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.title.set_fontweight("normal")

    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")

    for spine in ax.spines.values():

        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================

out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/vascall_umap"

fig.savefig(
    f"{out_base}.pdf",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"{out_base}.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# DGE:
# ND ISLET PERICYTES vs T2D ISLET PERICYTES
# CLEAN VERSION
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

CELLTYPE = "Pericytes"

GROUP1 = "T2D"
GROUP2 = "ND"

# =========================================================
# SUBSET
# =========================================================

peri = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[LOC_COL] == "Islet") &
    (vasc_adataf.obs[COND_COL].isin([GROUP1, GROUP2]))
].copy()

print(peri)
print()
print(
    peri.obs[COND_COL].value_counts()
)

# =========================================================
# OPTIONAL:
# REMOVE UNUSED CATEGORIES
# =========================================================

for col in peri.obs.columns:

    if str(peri.obs[col].dtype) == "category":

        peri.obs[col] = (
            peri.obs[col]
            .cat.remove_unused_categories()
        )

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    peri,
    groupby=COND_COL,
    groups=[GROUP1],
    reference=GROUP2,
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    peri,
    group=GROUP1
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

res = res[
    np.isfinite(
        res["logfoldchanges"]
    )
]

# =========================================================
# ADD NEG LOG10
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "pvals_adj"
)

# =========================================================
# PRINT TOP
# =========================================================

print("\nTOP UP IN T2D:")
print(
    res.head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

print("\nTOP DOWN IN T2D:")
print(
    res.sort_values(
        "logfoldchanges"
    ).head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================================================
# SAVE CSV
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv",
    index=False
)

print("\n✔ saved:")
print("/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv")

In [ ]:
# =========================================================
# DGE:
# ND ISLET PERICYTES vs T2D ISLET PERICYTES
# CLEAN VERSION
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

CELLTYPE = "Pericytes"

GROUP1 = "T2D"
GROUP2 = "ND"

# =========================================================
# SUBSET
# =========================================================

peri = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[LOC_COL] == "Exocrine") &
    (vasc_adataf.obs[COND_COL].isin([GROUP1, GROUP2]))
].copy()

print(peri)
print()
print(
    peri.obs[COND_COL].value_counts()
)

# =========================================================
# OPTIONAL:
# REMOVE UNUSED CATEGORIES
# =========================================================

for col in peri.obs.columns:

    if str(peri.obs[col].dtype) == "category":

        peri.obs[col] = (
            peri.obs[col]
            .cat.remove_unused_categories()
        )

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    peri,
    groupby=COND_COL,
    groups=[GROUP1],
    reference=GROUP2,
    method="wilcoxon",
    use_raw=False,
    layer='counts'
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    peri,
    group=GROUP1
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

res = res[
    np.isfinite(
        res["logfoldchanges"]
    )
]

# =========================================================
# ADD NEG LOG10
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "pvals_adj"
)

# =========================================================
# PRINT TOP
# =========================================================

print("\nTOP UP IN T2D:")
print(
    res.head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

print("\nTOP DOWN IN T2D:")
print(
    res.sort_values(
        "logfoldchanges"
    ).head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================================================
# SAVE CSV
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv",
    index=False
)

print("\n✔ saved:")
print("/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv")

In [ ]:
vasc_adataf


# SECTION 21 — VASCULAR COMPOSITION ANALYSIS
 Donor-level cell type fractions computed per compartment
 Mean fraction across donors plotted as stacked bar
   (ND vs T2D, Islet and Exocrine separately)
 Biologically impossible combinations excluded:
   Islet-assoc Fibroblasts removed from Exocrine
    Prism-ready CSVs exported per compartment


In [ ]:
# =========================================================
# FIXED VERSION
# (bottom array must be writable)
# =========================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

vascular_cells = [
    "Pericytes",
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL].isin(vascular_cells)
].copy()

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL, CELLTYPE_COL]
    )
    .size()
    .reset_index(name="n_cells")
)

# =========================================================
# COMPUTE FRACTIONS
# =========================================================

counts["fraction"] = (
    counts["n_cells"]
    /
    counts.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["n_cells"].transform("sum")
)

# =========================================================
# PLOT FUNCTION
# =========================================================

def composition_plot(location):

    d = counts[
        counts[LOC_COL] == location
    ].copy()

    # -----------------------------------------------------
    # pivot
    # -----------------------------------------------------

    pivot = d.pivot_table(
        index=[DONOR_COL, COND_COL],
        columns=CELLTYPE_COL,
        values="fraction",
        fill_value=0
    )

    # keep desired order
    pivot = pivot[
        [c for c in vascular_cells if c in pivot.columns]
    ]

    plot_df = pivot.reset_index()

    # sort by condition
    plot_df = plot_df.sort_values(
        COND_COL
    )

    # -----------------------------------------------------
    # plotting
    # -----------------------------------------------------

    fig, ax = plt.subplots(figsize=(5,3))

    x = np.arange(len(plot_df))

    # IMPORTANT FIX
    bottom = np.zeros(len(plot_df))

    for celltype in vascular_cells:

        if celltype not in plot_df.columns:
            continue

        vals = plot_df[celltype].values.astype(float)

        ax.bar(
            x,
            vals,
            bottom=bottom,
            label=celltype
        )

        # FIX
        bottom = bottom + vals

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    xticks = [
        f"{r[DONOR_COL]}\n{r[COND_COL]}"
        for _, r in plot_df.iterrows()
    ]

    ax.set_xticks(x)

    ax.set_xticklabels(
        xticks,
        rotation=45,
        ha="right"
    )

    ax.set_ylabel("Fraction")
    ax.set_xlabel("")

    ax.set_ylim(0,1)

    ax.set_title(
        f"Vascular Composition — {location}",
        fontsize=9
    )

    ax.legend(
        bbox_to_anchor=(1.02,1),
        loc="upper left",
        frameon=False
    )

    sns.despine()

    plt.tight_layout()

    # -----------------------------------------------------
    # SAVE
    # -----------------------------------------------------

    plt.savefig(
        f"/Users/kmeneses/Desktop/VascComposition_{location}.svg",
        bbox_inches="tight"
    )

    plt.savefig(
        f"/Users/kmeneses/Desktop/VascComposition_{location}.pdf",
        bbox_inches="tight"
    )

    plt.show()

# =========================================================
# RUN
# =========================================================

composition_plot("Islet")

composition_plot("Exocrine")

# =========================================================
# EXPORT
# =========================================================

counts.to_csv(
    "/Users/kmeneses/Desktop/Vascular_Composition_ByDonor.csv",
    index=False
)

print("✔ exported")

In [ ]:
# =========================================================
# MERGED ND vs T2D COMPOSITION
# (one bar for ND and one for T2D)
# separately for Islet and Exocrine
# =========================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

vascular_cells = [
    "Pericytes",
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL].isin(vascular_cells)
].copy()

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL, CELLTYPE_COL]
    )
    .size()
    .reset_index(name="n_cells")
)

# =========================================================
# DONOR-LEVEL FRACTIONS
# =========================================================

counts["fraction"] = (
    counts["n_cells"]
    /
    counts.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["n_cells"].transform("sum")
)

# =========================================================
# AVERAGE ACROSS DONORS
# =========================================================

mean_frac = (
    counts.groupby(
        [COND_COL, LOC_COL, CELLTYPE_COL]
    )["fraction"]
    .mean()
    .reset_index()
)

# =========================================================
# PLOT FUNCTION
# =========================================================

def composition_plot(location):

    d = mean_frac[
        mean_frac[LOC_COL] == location
    ].copy()

    # -----------------------------------------------------
    # pivot
    # -----------------------------------------------------

    pivot = d.pivot_table(
        index=COND_COL,
        columns=CELLTYPE_COL,
        values="fraction",
        fill_value=0
    )

    # keep order
    pivot = pivot[
        [c for c in vascular_cells if c in pivot.columns]
    ]

    # ND first
    pivot = pivot.reindex(["ND", "T2D"])

    # -----------------------------------------------------
    # plotting
    # -----------------------------------------------------

    fig, ax = plt.subplots(figsize=(2.2,3))

    x = np.arange(len(pivot))

    bottom = np.zeros(len(pivot))

    for celltype in vascular_cells:

        if celltype not in pivot.columns:
            continue

        vals = pivot[celltype].values.astype(float)

        ax.bar(
            x,
            vals,
            bottom=bottom,
            label=celltype
        )

        bottom += vals

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    ax.set_xticks(x)

    ax.set_xticklabels([
        "ND",
        "T2D"
    ])

    ax.set_ylabel("Mean fraction")

    ax.set_ylim(0,1)

    ax.set_title(
        f"{location}",
        fontsize=9,
        weight="bold"
    )

    sns.despine()

    plt.tight_layout()

    # -----------------------------------------------------
    # save
    # -----------------------------------------------------

    plt.savefig(
        f"/Users/kmeneses/Desktop/{location}_MergedComposition.svg",
        bbox_inches="tight"
    )

    plt.savefig(
        f"/Users/kmeneses/Desktop/{location}_MergedComposition.pdf",
        bbox_inches="tight"
    )

    plt.show()

# =========================================================
# RUN
# =========================================================

composition_plot("Islet")

composition_plot("Exocrine")

# =========================================================
# EXPORT FOR PRISM
# =========================================================

mean_frac.to_csv(
    "/Users/kmeneses/Desktop/VascularComposition_Merged.csv",
    index=False
)

print("✔ exported")

In [ ]:
sc.pl.dotplot(
    vasc_adataf,
    [
        "RGS5",
        "CSPG4",
        "PDGFRB",

        "COL1A1",
        "COL3A1",
        "DCN",
        "LUM",
        "FAP",
        "POSTN",
        "GCG", 
        "CHGA"
    ],
    groupby=["merged_cell_type", "Health_Status"],
    use_raw=False,
    standard_scale="var"
)

In [ ]:
sc.pl.dotplot(
    vasc_adataf,
    [
        "RGS5",
        "CSPG4",
        "PDGFRB",

        "COL1A1",
        "COL3A1",
        "DCN",
        "LUM",
        "FAP",
        "POSTN",
        "GCG", 
        "CHGA"
    ],
    groupby=["merged_cell_type"],
    use_raw=False,
    standard_scale="var"
)

In [ ]:
# =========================================================
# DGE:
# T2D ISLET FIBROBLASTS
# vs
# T2D EXOCRINE FIBROBLASTS
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

CELLTYPE = "Fibroblasts"

GROUP1 = "Islet"
GROUP2 = "Exocrine"

# =========================================================
# SUBSET
# =========================================================

fib = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[COND_COL] == "T2D") &
    (vasc_adataf.obs[LOC_COL].isin([GROUP1, GROUP2]))
].copy()

print(fib)

print("\nLocation counts:")
print(
    fib.obs[LOC_COL].value_counts()
)

# =========================================================
# REMOVE UNUSED CATEGORIES
# =========================================================

for col in fib.obs.columns:

    if str(fib.obs[col].dtype) == "category":

        fib.obs[col] = (
            fib.obs[col]
            .cat.remove_unused_categories()
        )

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    fib,
    groupby=LOC_COL,
    groups=[GROUP1],
    reference=GROUP2,
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    fib,
    group=GROUP1
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

res = res[
    np.isfinite(
        res["logfoldchanges"]
    )
]

# =========================================================
# ADD NEG LOG10
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "pvals_adj"
)

# =========================================================
# PRINT TOP ISLET-ENRICHED
# positive = higher in ISLET fibroblasts
# =========================================================

print("\nTOP ISLET FIBROBLAST GENES:")
print(
    res[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ].head(40)
)

# =========================================================
# PRINT TOP EXOCRINE-ENRICHED
# negative = higher in EXOCRINE fibroblasts
# =========================================================

print("\nTOP EXOCRINE FIBROBLAST GENES:")
print(
    res.sort_values(
        "logfoldchanges"
    )[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ].head(40)
)

# =========================================================
# SAVE
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_Islet_vs_Exocrine_Fibroblasts.csv",
    index=False
)

print("\n✔ saved:")
print("/Users/kmeneses/Desktop/T2D_Islet_vs_Exocrine_Fibroblasts.csv")

In [ ]:
# =========================================================
# DGE:
# ND ISLET FIBROBLASTS
# vs
# T2D ISLET FIBROBLASTS
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

CELLTYPE = "Fibroblasts"

GROUP1 = "T2D"
GROUP2 = "ND"

# =========================================================
# SUBSET
# =========================================================

fib = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[LOC_COL] == "Islet") &
    (vasc_adataf.obs[COND_COL].isin([GROUP1, GROUP2]))
].copy()

print(fib)

print("\nCondition counts:")
print(
    fib.obs[COND_COL].value_counts()
)

# =========================================================
# CLEAN CATEGORIES
# =========================================================

for col in fib.obs.columns:

    if str(fib.obs[col].dtype) == "category":

        fib.obs[col] = (
            fib.obs[col]
            .cat.remove_unused_categories()
        )

# =========================================================
# DGE
# positive logFC = HIGHER IN T2D
# =========================================================

sc.tl.rank_genes_groups(
    fib,
    groupby=COND_COL,
    groups=[GROUP1],
    reference=GROUP2,
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    fib,
    group=GROUP1
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

res = res[
    np.isfinite(
        res["logfoldchanges"]
    )
]

# =========================================================
# ADD NEGLOG10
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "pvals_adj"
)

# =========================================================
# TOP UP IN T2D
# =========================================================

print("\nTOP UP IN T2D:")
print(
    res[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ].head(40)
)

# =========================================================
# TOP DOWN IN T2D
# =========================================================

print("\nTOP DOWN IN T2D:")
print(
    res.sort_values(
        "logfoldchanges"
    )[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ].head(40)
)

# =========================================================
# SAVE
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/ND_vs_T2D_Islet_Fibroblasts.csv",
    index=False
)

print("\n✔ saved:")
print("/Users/kmeneses/Desktop/ND_vs_T2D_Islet_Fibroblasts.csv")

In [ ]:
# =========================================================
# TEST FOR GENES UP IN T2D
# =========================================================

# positive logFC = UP in T2D
up_t2d = res[
    res["logfoldchanges"] > 0
].copy()

# sort strongest positive first
up_t2d = up_t2d.sort_values(
    "logfoldchanges",
    ascending=False
)

# =========================================================
# SIGNIFICANT ONLY
# =========================================================

sig_up_t2d = up_t2d[
    up_t2d["pvals_adj"] < 0.05
].copy()

# =========================================================
# PRINT
# =========================================================

print("\nSIGNIFICANT GENES UP IN T2D:")
print(
    sig_up_t2d[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ].head(50)
)

# =========================================================
# SAVE
# =========================================================

sig_up_t2d.to_csv(
    "/Users/kmeneses/Desktop/T2D_UP_genes.csv",
    index=False
)

print("\n✔ saved:")
print("/Users/kmeneses/Desktop/T2D_UP_genes.csv")

In [ ]:
# =========================================================
# DGE:
# ND ISLET PERICYTES vs T2D ISLET PERICYTES
# CLEAN VERSION
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

CELLTYPE = "Islet-associated Fibroblasts"

GROUP1 = "T2D"
GROUP2 = "ND"

# =========================================================
# SUBSET
# =========================================================

peri = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[LOC_COL] == "Islet") &
    (vasc_adataf.obs[COND_COL].isin([GROUP1, GROUP2]))
].copy()

print(peri)
print()
print(
    peri.obs[COND_COL].value_counts()
)

# =========================================================
# OPTIONAL:
# REMOVE UNUSED CATEGORIES
# =========================================================

for col in peri.obs.columns:

    if str(peri.obs[col].dtype) == "category":

        peri.obs[col] = (
            peri.obs[col]
            .cat.remove_unused_categories()
        )

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    peri,
    groupby=COND_COL,
    groups=[GROUP1],
    reference=GROUP2,
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    peri,
    group=GROUP1
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

res = res[
    np.isfinite(
        res["logfoldchanges"]
    )
]

# =========================================================
# ADD NEG LOG10
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "pvals_adj"
)

# =========================================================
# PRINT TOP
# =========================================================

print("\nTOP UP IN T2D:")
print(
    res.head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

print("\nTOP DOWN IN T2D:")
print(
    res.sort_values(
        "logfoldchanges"
    ).head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================================================
# SAVE CSV
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv",
    index=False
)

print("\n✔ saved:")
print("/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv")

In [ ]:
# =========================================================
# DGE:
# ND ISLET PERICYTES vs T2D ISLET PERICYTES
# CLEAN VERSION
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

CELLTYPE = "Endothelial Cells"

GROUP1 = "T2D"
GROUP2 = "ND"

# =========================================================
# SUBSET
# =========================================================

peri = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[LOC_COL] == "Islet") &
    (vasc_adataf.obs[COND_COL].isin([GROUP1, GROUP2]))
].copy()

print(peri)
print()
print(
    peri.obs[COND_COL].value_counts()
)

# =========================================================
# OPTIONAL:
# REMOVE UNUSED CATEGORIES
# =========================================================

for col in peri.obs.columns:

    if str(peri.obs[col].dtype) == "category":

        peri.obs[col] = (
            peri.obs[col]
            .cat.remove_unused_categories()
        )

# =========================================================
# DGE
# =========================================================

sc.tl.rank_genes_groups(
    peri,
    groupby=COND_COL,
    groups=[GROUP1],
    reference=GROUP2,
    method="wilcoxon",
    use_raw=False
)

# =========================================================
# EXTRACT RESULTS
# =========================================================

res = sc.get.rank_genes_groups_df(
    peri,
    group=GROUP1
)

# =========================================================
# CLEAN
# =========================================================

res = res.dropna(
    subset=[
        "logfoldchanges",
        "pvals_adj"
    ]
)

res = res[
    np.isfinite(
        res["logfoldchanges"]
    )
]

# =========================================================
# ADD NEG LOG10
# =========================================================

res["neglog10_padj"] = -np.log10(
    res["pvals_adj"] + 1e-300
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "pvals_adj"
)

# =========================================================
# PRINT TOP
# =========================================================

print("\nTOP UP IN T2D:")
print(
    res.head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

print("\nTOP DOWN IN T2D:")
print(
    res.sort_values(
        "logfoldchanges"
    ).head(30)[
        [
            "names",
            "logfoldchanges",
            "pvals_adj"
        ]
    ]
)

# =========================================================
# SAVE CSV
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv",
    index=False
)

print("\n✔ saved:")
print("/Users/kmeneses/Desktop/T2D_vs_ND_Islet_Pericytes_DGE.csv")

In [ ]:
# =========================================================
# MERGE:
# islet fibroblasts -> Islet-associated Fibroblasts
# =========================================================

CELLTYPE_COL = "merged_cell_type"
LOC_COL = "Location"

# =========================================================
# COPY ORIGINAL LABELS
# =========================================================

vasc_adataf.obs["merged_cell_type_old"] = (
    vasc_adataf.obs[CELLTYPE_COL].copy()
)

# =========================================================
# MERGE LABELS
# =========================================================

mask = (
    (vasc_adataf.obs[CELLTYPE_COL] == "Fibroblasts") &
    (vasc_adataf.obs[LOC_COL] == "Islet")
)

vasc_adataf.obs.loc[
    mask,
    CELLTYPE_COL
] = "Islet-associated Fibroblasts"

# =========================================================
# CLEAN CATEGORY
# =========================================================

if str(vasc_adataf.obs[CELLTYPE_COL].dtype) == "category":

    vasc_adataf.obs[CELLTYPE_COL] = (
        vasc_adataf.obs[CELLTYPE_COL]
        .cat.remove_unused_categories()
    )

# =========================================================
# CHECK COUNTS
# =========================================================

print(
    vasc_adataf.obs
    .groupby([LOC_COL, CELLTYPE_COL])
    .size()
)

print("\n✔ merged islet fibroblasts into Islet-associated Fibroblasts")

In [ ]:
# =========================================================
# MERGED ND vs T2D COMPOSITION
# (one bar for ND and one for T2D)
# separately for Islet and Exocrine
# =========================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

vascular_cells = [
    "Pericytes",
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL].isin(vascular_cells)
].copy()

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL, CELLTYPE_COL]
    )
    .size()
    .reset_index(name="n_cells")
)

# =========================================================
# DONOR-LEVEL FRACTIONS
# =========================================================

counts["fraction"] = (
    counts["n_cells"]
    /
    counts.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["n_cells"].transform("sum")
)

# =========================================================
# AVERAGE ACROSS DONORS
# =========================================================

mean_frac = (
    counts.groupby(
        [COND_COL, LOC_COL, CELLTYPE_COL]
    )["fraction"]
    .mean()
    .reset_index()
)

# =========================================================
# PLOT FUNCTION
# =========================================================

def composition_plot(location):

    d = mean_frac[
        mean_frac[LOC_COL] == location
    ].copy()

    # -----------------------------------------------------
    # pivot
    # -----------------------------------------------------

    pivot = d.pivot_table(
        index=COND_COL,
        columns=CELLTYPE_COL,
        values="fraction",
        fill_value=0
    )

    # keep order
    pivot = pivot[
        [c for c in vascular_cells if c in pivot.columns]
    ]

    # ND first
    pivot = pivot.reindex(["ND", "T2D"])

    # -----------------------------------------------------
    # plotting
    # -----------------------------------------------------

    fig, ax = plt.subplots(figsize=(2.2,3))

    x = np.arange(len(pivot))

    bottom = np.zeros(len(pivot))

    for celltype in vascular_cells:

        if celltype not in pivot.columns:
            continue

        vals = pivot[celltype].values.astype(float)

        ax.bar(
            x,
            vals,
            bottom=bottom,
            label=celltype
        )

        bottom += vals

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    ax.set_xticks(x)

    ax.set_xticklabels([
        "ND",
        "T2D"
    ])

    ax.set_ylabel("Mean fraction")

    ax.set_ylim(0,1)

    ax.set_title(
        f"{location}",
        fontsize=9,
        weight="bold"
    )

    sns.despine()

    plt.tight_layout()

    # -----------------------------------------------------
    # save
    # -----------------------------------------------------

    plt.savefig(
        f"/Users/kmeneses/Desktop/{location}_MergedComposition.svg",
        bbox_inches="tight"
    )

    plt.savefig(
        f"/Users/kmeneses/Desktop/{location}_MergedComposition.pdf",
        bbox_inches="tight"
    )

    plt.show()

# =========================================================
# RUN
# =========================================================

composition_plot("Islet")

composition_plot("Exocrine")

# =========================================================
# EXPORT FOR PRISM
# =========================================================

mean_frac.to_csv(
    "/Users/kmeneses/Desktop/VascularComposition_Merged.csv",
    index=False
)

print("✔ exported")

In [ ]:
# =========================================================
# MERGED ND vs T2D COMPOSITION
#
# SPATIAL DATA
#
# ISLET:
#   includes:
#   - Pericytes
#   - Endothelial Cells
#   - Islet-associated Fibroblasts
#   - Fibroblasts
#
# EXOCRINE:
#   removes Islet-associated Fibroblasts
#
# EXPORTS:
# - Illustrator-ready plots
# - Prism-friendly CSVs
# =========================================================

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# CELL TYPES
# =========================================================

islet_cells = [
    "Pericytes",
    "Endothelial Cells",
    "Islet-associated Fibroblasts",
    "Fibroblasts"
]

exocrine_cells = [
    "Pericytes",
    "Endothelial Cells",
    "Fibroblasts"
]

# =========================================================
# COLORS
# =========================================================

colors = {
    "Pericytes": "#4C72B0",
    "Endothelial Cells": "#55A868",
    "Islet-associated Fibroblasts": "#C44E52",
    "Fibroblasts": "#8172B2"
}

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf.copy()

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL, CELLTYPE_COL]
    )
    .size()
    .reset_index(name="n_cells")
)

# =========================================================
# DONOR-LEVEL FRACTIONS
# =========================================================

counts["fraction"] = (
    counts["n_cells"]
    /
    counts.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["n_cells"].transform("sum")
)

# =========================================================
# AVERAGE ACROSS DONORS
# =========================================================

mean_frac = (
    counts.groupby(
        [COND_COL, LOC_COL, CELLTYPE_COL]
    )["fraction"]
    .mean()
    .reset_index()
)

# =========================================================
# FUNCTION
# =========================================================

def composition_plot(location):

    # -----------------------------------------------------
    # choose cell types
    # -----------------------------------------------------

    if location == "Islet":
        keep_cells = islet_cells

    else:
        keep_cells = exocrine_cells

    # -----------------------------------------------------
    # subset
    # -----------------------------------------------------

    d = mean_frac[
        (mean_frac[LOC_COL] == location) &
        (mean_frac[CELLTYPE_COL].isin(keep_cells))
    ].copy()

    # -----------------------------------------------------
    # pivot
    # -----------------------------------------------------

    pivot = d.pivot_table(
        index=COND_COL,
        columns=CELLTYPE_COL,
        values="fraction",
        fill_value=0
    )

    # keep desired order
    pivot = pivot[
        [c for c in keep_cells if c in pivot.columns]
    ]

    # ND first
    pivot = pivot.reindex(["ND", "T2D"])

    # =====================================================
    # PLOT
    # =====================================================

    fig, ax = plt.subplots(figsize=(2.5,3))

    x = np.arange(len(pivot))

    bottom = np.zeros(len(pivot))

    for celltype in keep_cells:

        # IMPORTANT FIX
        if celltype not in pivot.columns:
            continue

        vals = pivot[celltype].values.astype(float)

        ax.bar(
            x,
            vals,
            bottom=bottom,
            label=celltype,
            color=colors[celltype],
            edgecolor="black",
            linewidth=0.5
        )

        bottom += vals

    # =====================================================
    # LABELS
    # =====================================================

    ax.set_xticks(x)

    ax.set_xticklabels([
        "ND",
        "T2D"
    ])

    ax.set_ylabel("Mean fraction")

    ax.set_ylim(0,1)

    ax.set_title(
        location,
        fontsize=9,
        weight="bold"
    )

    # =====================================================
    # LEGEND
    # =====================================================

    ax.legend(
        frameon=False,
        bbox_to_anchor=(1.02,1),
        loc="upper left",
        fontsize=7
    )

    sns.despine()

    plt.tight_layout()

    # =====================================================
    # SAVE
    # =====================================================

    plt.savefig(
        f"{OUT}/{location}_MergedComposition.svg",
        bbox_inches="tight"
    )

    plt.savefig(
        f"{OUT}/{location}_MergedComposition.pdf",
        bbox_inches="tight"
    )

    plt.show()

    # =====================================================
    # PRISM EXPORT
    # =====================================================

    prism = pivot.copy().T

    prism.to_csv(
        f"{OUT}/{location}_Composition_PRISM.csv"
    )

# =========================================================
# RUN
# =========================================================

composition_plot("Islet")

composition_plot("Exocrine")

# =========================================================
# EXPORT FULL TABLE
# =========================================================

mean_frac.to_csv(
    f"{OUT}/VascularComposition_Merged.csv",
    index=False
)

print("✔ COMPLETE")

In [ ]:
# =========================================================
# ENDO:PERI RATIO ANALYSIS
#
# donor level
#
# ratio =
# Endothelial Cells / Pericytes
#
# compares:
# ND vs T2D
#
# separately for:
# - Islet
# - Exocrine
# =========================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# KEEP ONLY ENDO + PERI
# =========================================================

keep = [
    "Pericytes",
    "Endothelial Cells"
]

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL].isin(keep)
].copy()

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    ad.obs.groupby(
        [
            DONOR_COL,
            COND_COL,
            LOC_COL,
            CELLTYPE_COL
        ]
    )
    .size()
    .reset_index(name="n_cells")
)

# =========================================================
# PIVOT
# =========================================================

pivot = counts.pivot_table(
    index=[
        DONOR_COL,
        COND_COL,
        LOC_COL
    ],
    columns=CELLTYPE_COL,
    values="n_cells",
    fill_value=0
).reset_index()

# =========================================================
# COMPUTE RATIO
# =========================================================

pivot["Endo_Peri_Ratio"] = (
    pivot["Endothelial Cells"]
    /
    pivot["Pericytes"]
)

# optional safety
pivot = pivot.replace(
    [np.inf, -np.inf],
    np.nan
)

# =========================================================
# EXPORT RAW
# =========================================================

pivot.to_csv(
    f"{OUT}/EndoPeri_Ratios_ByDonor.csv",
    index=False
)

print("✔ donor ratios exported")

# =========================================================
# FUNCTION
# =========================================================

def ratio_analysis(location):

    # -----------------------------------------------------
    # subset
    # -----------------------------------------------------

    d = pivot[
        pivot[LOC_COL] == location
    ].copy()

    # -----------------------------------------------------
    # groups
    # -----------------------------------------------------

    nd = d.loc[
        d[COND_COL] == "ND",
        "Endo_Peri_Ratio"
    ]

    t2d = d.loc[
        d[COND_COL] == "T2D",
        "Endo_Peri_Ratio"
    ]

    # -----------------------------------------------------
    # stats
    # -----------------------------------------------------

    stat, p = mannwhitneyu(
        nd,
        t2d,
        alternative="two-sided"
    )

    print("\n====================================")
    print(location)
    print("====================================")

    print("\nND median:")
    print(np.median(nd))

    print("\nT2D median:")
    print(np.median(t2d))

    print("\nMann-Whitney p-value:")
    print(p)

    # -----------------------------------------------------
    # plot
    # -----------------------------------------------------

    fig, ax = plt.subplots(figsize=(2.3,3))

    sns.boxplot(
        data=d,
        x=COND_COL,
        y="Endo_Peri_Ratio",
        showcaps=True,
        showfliers=False,
        width=0.6,
        ax=ax
    )

    sns.stripplot(
        data=d,
        x=COND_COL,
        y="Endo_Peri_Ratio",
        color="black",
        size=4,
        alpha=0.8,
        ax=ax
    )

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    ax.set_title(
        location,
        fontsize=9,
        weight="bold"
    )

    ax.set_ylabel(
        "Endothelial : Pericyte ratio"
    )

    ax.set_xlabel("")

    # -----------------------------------------------------
    # p-value
    # -----------------------------------------------------

    ymax = d["Endo_Peri_Ratio"].max()

    ax.text(
        0.5,
        ymax * 1.05,
        f"p = {p:.3f}",
        ha="center",
        fontsize=8
    )

    sns.despine()

    plt.tight_layout()

    # -----------------------------------------------------
    # SAVE
    # -----------------------------------------------------

    plt.savefig(
        f"{OUT}/{location}_EndoPeriRatio.svg",
        bbox_inches="tight"
    )

    plt.savefig(
        f"{OUT}/{location}_EndoPeriRatio.pdf",
        bbox_inches="tight"
    )

    plt.show()

    # -----------------------------------------------------
    # PRISM EXPORT
    # -----------------------------------------------------

    prism = d.pivot_table(
        index=DONOR_COL,
        columns=COND_COL,
        values="Endo_Peri_Ratio"
    )

    prism.to_csv(
        f"{OUT}/{location}_EndoPeriRatio_PRISM.csv"
    )

# =========================================================
# RUN
# =========================================================

ratio_analysis("Islet")

ratio_analysis("Exocrine")

print("✔ COMPLETE")

In [ ]:
# =========================================================
# ISLET-ASSOCIATED FIBROBLAST : PERICYTE
#
# donor level
#
# ratio =
# Islet-associated Fibroblasts / Pericytes
#
# compares:
# ND vs T2D
#
# ISLET ONLY
# =========================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# KEEP ONLY RELEVANT CELLS
# =========================================================

keep = [
    "Pericytes",
    "Islet-associated Fibroblasts"
]

ad = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL].isin(keep)) &
    (vasc_adataf.obs[LOC_COL] == "Islet")
].copy()

print(ad)

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    ad.obs.groupby(
        [
            DONOR_COL,
            COND_COL,
            CELLTYPE_COL
        ]
    )
    .size()
    .reset_index(name="n_cells")
)

# =========================================================
# PIVOT
# =========================================================

pivot = counts.pivot_table(
    index=[
        DONOR_COL,
        COND_COL
    ],
    columns=CELLTYPE_COL,
    values="n_cells",
    fill_value=0
).reset_index()

# =========================================================
# ENSURE COLUMNS EXIST
# =========================================================

for col in keep:

    if col not in pivot.columns:

        pivot[col] = 0

# =========================================================
# COMPUTE RATIO
# =========================================================

pivot["IsletFib_Peri_Ratio"] = (
    pivot["Islet-associated Fibroblasts"]
    /
    (pivot["Pericytes"] + 1e-9)
)

# remove inf
pivot = pivot.replace(
    [np.inf, -np.inf],
    np.nan
)

# =========================================================
# EXPORT RAW
# =========================================================

pivot.to_csv(
    f"{OUT}/IsletFib_Peri_Ratios_ByDonor.csv",
    index=False
)

print("✔ donor ratios exported")

# =========================================================
# GROUPS
# =========================================================

nd = pivot.loc[
    pivot[COND_COL] == "ND",
    "IsletFib_Peri_Ratio"
]

t2d = pivot.loc[
    pivot[COND_COL] == "T2D",
    "IsletFib_Peri_Ratio"
]

# =========================================================
# STATS
# =========================================================

stat, p = mannwhitneyu(
    nd,
    t2d,
    alternative="two-sided"
)

print("\n====================================")
print("ISLET FIBROBLAST : PERICYTE")
print("====================================")

print("\nND median:")
print(np.median(nd))

print("\nT2D median:")
print(np.median(t2d))

print("\nMWU p-value:")
print(p)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(2.3,3))

sns.boxplot(
    data=pivot,
    x=COND_COL,
    y="IsletFib_Peri_Ratio",
    order=["ND", "T2D"],
    showcaps=True,
    showfliers=False,
    width=0.6,
    ax=ax
)

sns.stripplot(
    data=pivot,
    x=COND_COL,
    y="IsletFib_Peri_Ratio",
    order=["ND", "T2D"],
    color="black",
    size=4,
    alpha=0.8,
    ax=ax
)

# =========================================================
# LABELS
# =========================================================

ax.set_title(
    "Islet",
    fontsize=9,
    fontweight="bold"
)

ax.set_ylabel(
    "Islet-associated fibroblast : Pericyte"
)

ax.set_xlabel("")

# =========================================================
# P-VALUE
# =========================================================

ymax = pivot["IsletFib_Peri_Ratio"].max()

ax.text(
    0.5,
    ymax * 1.05,
    f"MWU p = {p:.3f}",
    ha="center",
    fontsize=8
)

sns.despine()

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/IsletFib_Peri_Ratio.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/IsletFib_Peri_Ratio.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# PRISM EXPORT
# =========================================================

prism = pivot.pivot_table(
    index=DONOR_COL,
    columns=COND_COL,
    values="IsletFib_Peri_Ratio"
)

prism.to_csv(
    f"{OUT}/IsletFib_Peri_Ratio_PRISM.csv"
)

print("✔ COMPLETE")

In [ ]:
adata_all.obs['merged_cell_type'].value_counts()

In [ ]:
# =========================================================
# TRUE VASCULAR : ENDOCRINE RATIO
#
# VASCULAR CELLS:
# from vasc_adataf
#
# ENDOCRINE CELLS:
# from adata_all
#
# avoids needing transferred labels
# =========================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu

# =========================================================
# SETTINGS
# =========================================================

COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# CELL DEFINITIONS
# =========================================================

vascular_cells = [
    "Pericytes",
    "Endothelial Cells"
]

endocrine_cells = [
    "Beta Cells",
    "Alpha Cells",
    "Delta Cells",
    "PPY Cells"
]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# VASCULAR COUNTS
# FROM vasc_adataf
# =========================================================

vasc = vasc_adataf[
    vasc_adataf.obs["merged_cell_type"].isin(
        vascular_cells
    )
].copy()

vasc_counts = (
    vasc.obs.groupby(
        [
            DONOR_COL,
            COND_COL,
            LOC_COL
        ]
    )
    .size()
    .reset_index(name="vascular_n")
)

print("\nVASCULAR COUNTS")
print(vasc_counts.head())

# =========================================================
# ENDOCRINE COUNTS
# FROM adata_all
# =========================================================

endo = adata_all[
    adata_all.obs["merged_cell_type"].isin(
        endocrine_cells
    )
].copy()

endo_counts = (
    endo.obs.groupby(
        [
            DONOR_COL,
            COND_COL,
            LOC_COL
        ]
    )
    .size()
    .reset_index(name="endocrine_n")
)

print("\nENDOCRINE COUNTS")
print(endo_counts.head())

# =========================================================
# MERGE
# =========================================================

merged = pd.merge(
    vasc_counts,
    endo_counts,
    on=[
        DONOR_COL,
        COND_COL,
        LOC_COL
    ],
    how="outer"
)

# fill missing
merged["vascular_n"] = (
    merged["vascular_n"]
    .fillna(0)
)

merged["endocrine_n"] = (
    merged["endocrine_n"]
    .fillna(0)
)

# =========================================================
# COMPUTE RATIO
# =========================================================

merged["Vasc_Endocrine_Ratio"] = (
    merged["vascular_n"]
    /
    (merged["endocrine_n"] + 1e-9)
)

print("\nMERGED:")
print(merged.head())

# =========================================================
# EXPORT RAW
# =========================================================

merged.to_csv(
    f"{OUT}/TRUE_Vasc_Endocrine_Ratio_ByDonor.csv",
    index=False
)

# =========================================================
# FUNCTION
# =========================================================

def ratio_analysis(location):

    d = merged[
        merged[LOC_COL] == location
    ].copy()

    # -----------------------------------------------------
    # groups
    # -----------------------------------------------------

    nd = d.loc[
        d[COND_COL] == "ND",
        "Vasc_Endocrine_Ratio"
    ]

    t2d = d.loc[
        d[COND_COL] == "T2D",
        "Vasc_Endocrine_Ratio"
    ]

    # -----------------------------------------------------
    # stats
    # -----------------------------------------------------

    stat, p = mannwhitneyu(
        nd,
        t2d,
        alternative="two-sided"
    )

    print("\n====================================")
    print(location)
    print("====================================")

    print("\nND median:")
    print(np.median(nd))

    print("\nT2D median:")
    print(np.median(t2d))

    print("\nMWU p-value:")
    print(p)

    # -----------------------------------------------------
    # plot
    # -----------------------------------------------------

    fig, ax = plt.subplots(figsize=(2.5,3))

    sns.boxplot(
        data=d,
        x=COND_COL,
        y="Vasc_Endocrine_Ratio",
        order=["ND", "T2D"],
        width=0.6,
        showfliers=False,
        ax=ax
    )

    sns.stripplot(
        data=d,
        x=COND_COL,
        y="Vasc_Endocrine_Ratio",
        order=["ND", "T2D"],
        color="black",
        size=4,
        alpha=0.8,
        ax=ax
    )

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    ax.set_title(
        location,
        fontsize=9,
        fontweight="bold"
    )

    ax.set_xlabel("")

    ax.set_ylabel(
        "(Pericytes + Endothelium)\n/ Endocrine cells"
    )

    ymax = d["Vasc_Endocrine_Ratio"].max()

    ax.text(
        0.5,
        ymax * 1.05,
        f"MWU p = {p:.3f}",
        ha="center",
        fontsize=8
    )

    sns.despine()

    plt.tight_layout()

    # -----------------------------------------------------
    # save
    # -----------------------------------------------------

    plt.savefig(
        f"{OUT}/{location}_TRUE_VascEndocrineRatio.svg",
        bbox_inches="tight"
    )

    plt.savefig(
        f"{OUT}/{location}_TRUE_VascEndocrineRatio.pdf",
        bbox_inches="tight"
    )

    plt.show()

    # -----------------------------------------------------
    # PRISM EXPORT
    # -----------------------------------------------------

    prism = d.pivot_table(
        index=DONOR_COL,
        columns=COND_COL,
        values="Vasc_Endocrine_Ratio"
    )

    prism.to_csv(
        f"{OUT}/{location}_TRUE_VascEndocrineRatio_PRISM.csv"
    )

# =========================================================
# RUN
# =========================================================

ratio_analysis("Islet")

ratio_analysis("Exocrine")

print("\n✔ COMPLETE")

In [ ]:
adata_all

In [ ]:
adata_all.obs['merged_cell_type']

In [ ]:
# =========================================================
# EXOCRINE FIBROBLAST : EXOCRINE PERICYTE RATIO
#
# compares:
# ND vs T2D
#
# donor-level analysis
# =========================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET EXOCRINE ONLY
# =========================================================

ad = vasc_adataf[
    (vasc_adataf.obs[LOC_COL] == "Exocrine") &
    (
        vasc_adataf.obs[CELLTYPE_COL].isin([
            "Fibroblasts",
            "Pericytes"
        ])
    )
].copy()

print(ad)

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    ad.obs.groupby(
        [
            DONOR_COL,
            COND_COL,
            CELLTYPE_COL
        ]
    )
    .size()
    .reset_index(name="n_cells")
)

# =========================================================
# PIVOT
# =========================================================

pivot = counts.pivot_table(
    index=[
        DONOR_COL,
        COND_COL
    ],
    columns=CELLTYPE_COL,
    values="n_cells",
    fill_value=0
).reset_index()

# =========================================================
# RATIO
# =========================================================

pivot["Fibro_Pericyte_Ratio"] = (
    pivot.get("Fibroblasts", 0)
    /
    pivot.get("Pericytes", 1)
)

# remove inf
pivot = pivot.replace(
    [np.inf, -np.inf],
    np.nan
)

print("\n====================================")
print("DONOR RATIOS")
print("====================================")

print(pivot)

# =========================================================
# STATS
# =========================================================

nd = pivot.loc[
    pivot[COND_COL] == "ND",
    "Fibro_Pericyte_Ratio"
]

t2d = pivot.loc[
    pivot[COND_COL] == "T2D",
    "Fibro_Pericyte_Ratio"
]

stat, p = mannwhitneyu(
    nd,
    t2d,
    alternative="two-sided"
)

print("\n====================================")
print("EXOCRINE FIBRO : PERICYTE")
print("====================================")

print("\nND median:")
print(np.median(nd))

print("\nT2D median:")
print(np.median(t2d))

print("\nMWU p-value:")
print(p)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(2.5,3))

sns.boxplot(
    data=pivot,
    x=COND_COL,
    y="Fibro_Pericyte_Ratio",
    order=["ND", "T2D"],
    showcaps=True,
    showfliers=False,
    width=0.6,
    ax=ax
)

sns.stripplot(
    data=pivot,
    x=COND_COL,
    y="Fibro_Pericyte_Ratio",
    order=["ND", "T2D"],
    color="black",
    size=4,
    alpha=0.8,
    ax=ax
)

# =========================================================
# LABELS
# =========================================================

ax.set_title(
    "Exocrine",
    fontsize=9,
    weight="bold"
)

ax.set_ylabel(
    "Fibroblasts : Pericytes"
)

ax.set_xlabel("")

ymax = pivot["Fibro_Pericyte_Ratio"].max()

ax.text(
    0.5,
    ymax * 1.05,
    f"MWU p = {p:.3f}",
    ha="center",
    fontsize=8
)

sns.despine()

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/Exocrine_Fibro_Pericyte_Ratio.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/Exocrine_Fibro_Pericyte_Ratio.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

pivot.to_csv(
    f"{OUT}/Exocrine_Fibro_Pericyte_Ratio.csv",
    index=False
)

print("✔ COMPLETE")

In [ ]:
# =========================================================
# TEST:
# PERICYTE : ENDOTHELIAL RATIO
# ND vs T2D
# =========================================================

import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

PERI = "Pericytes"
ENDO = "Endothelial Cells"

# =========================================================
# SUBSET
# =========================================================

df = vasc_adataf.obs.copy()

df = df[
    df[CELLTYPE_COL].isin([
        PERI,
        ENDO
    ])
].copy()

# =========================================================
# COUNT CELLS
# =========================================================

counts = (
    df.groupby(
        [DONOR_COL, COND_COL, LOC_COL, CELLTYPE_COL]
    )
    .size()
    .unstack(fill_value=0)
)

# =========================================================
# ENSURE COLUMNS
# =========================================================

for c in [PERI, ENDO]:

    if c not in counts.columns:

        counts[c] = 0

# =========================================================
# COMPUTE RATIO
# =========================================================

counts["Peri_Endo_Ratio"] = (
    counts[PERI]
    /
    (counts[ENDO] + 1e-9)
)

# =========================================================
# RESET INDEX
# =========================================================

counts = counts.reset_index()

# =========================================================
# TEST FUNCTION
# =========================================================

def run_test(location):

    d = counts[
        counts[LOC_COL] == location
    ].copy()

    nd = d[
        d[COND_COL] == "ND"
    ]["Peri_Endo_Ratio"]

    t2d = d[
        d[COND_COL] == "T2D"
    ]["Peri_Endo_Ratio"]

    stat, p = mannwhitneyu(
        nd,
        t2d,
        alternative="two-sided"
    )

    print("\n===================================")
    print(location.upper())
    print("===================================")

    print("\nND ratios:")
    print(nd.values)

    print("\nT2D ratios:")
    print(t2d.values)

    print(f"\nMWU p-value = {p:.5f}")

    print("\nMedian ND:")
    print(np.median(nd))

    print("\nMedian T2D:")
    print(np.median(t2d))

# =========================================================
# RUN
# =========================================================

run_test("Islet")

run_test("Exocrine")

# =========================================================
# EXPORT
# =========================================================

counts.to_csv(
    "/Users/kmeneses/Desktop/Peri_Endo_Ratio.csv",
    index=False
)

print("\n✔ exported:")
print("/Users/kmeneses/Desktop/Peri_Endo_Ratio.csv")

In [ ]:
# =========================================================
# CELL-LEVEL ANALYSIS
# PERICYTE : ENDOTHELIAL ASSOCIATION
# =========================================================
#
# Instead of donor-level ratios,
# this tests:
#
# Is a randomly selected vascular cell
# more likely to be a PERICYTE in ND vs T2D?
#
# This gives MUCH higher power.
#
# =========================================================

import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from scipy.stats import chi2_contingency

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

PERI = "Pericytes"
ENDO = "Endothelial Cells"

# =========================================================
# SUBSET
# =========================================================

df = vasc_adataf.obs.copy()

df = df[
    df[CELLTYPE_COL].isin([
        PERI,
        ENDO
    ])
].copy()

# =========================================================
# CREATE BINARY LABEL
# =========================================================

df["is_pericyte"] = (
    df[CELLTYPE_COL] == PERI
).astype(int)

# =========================================================
# FUNCTION
# =========================================================

def run_cell_level(location):

    d = df[
        df[LOC_COL] == location
    ].copy()

    # -----------------------------------------------------
    # contingency table
    # -----------------------------------------------------

    tab = pd.crosstab(
        d[COND_COL],
        d["is_pericyte"]
    )

    # columns:
    # 0 = endothelial
    # 1 = pericyte

    print("\n====================================")
    print(location.upper())
    print("====================================")

    print("\nContingency table:")
    print(tab)

    # -----------------------------------------------------
    # fisher exact
    # -----------------------------------------------------

    oddsratio, p_fisher = fisher_exact(tab)

    # -----------------------------------------------------
    # chi-square
    # -----------------------------------------------------

    chi2, p_chi2, dof, exp = chi2_contingency(tab)

    # -----------------------------------------------------
    # proportions
    # -----------------------------------------------------

    prop = (
        tab[1]
        /
        (tab[0] + tab[1])
    )

    print("\nPericyte fraction:")
    print(prop)

    print("\nOdds ratio:")
    print(oddsratio)

    print("\nFisher exact p-value:")
    print(p_fisher)

    print("\nChi-square p-value:")
    print(p_chi2)

# =========================================================
# RUN
# =========================================================

run_cell_level("Islet")

run_cell_level("Exocrine")

In [ ]:
# =========================================================
# SPATIAL VASCULAR NICHE ANALYSIS
#
# Computes:
#
# 1. BM continuity
# 2. Vascular coverage
# 3. Local ECM intensity
# 4. Islet edge organization
#
# Requires:
# - vasc_adataf
# - spatial coordinates in obsm["spatial"]
# - cell labels
# - BM / ECM genes in panel
#
# =========================================================

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
LOC_COL = "Location"
COND_COL = "Health_Status"
DONOR_COL = "Sample"

SPATIAL_KEY = "spatial"

PERI = "Pericytes"
ENDO = "Endothelial Cells"
ISLET_FIB = "Islet-associated Fibroblasts"

# =========================================================
# GENE SETS
# =========================================================

BM_GENES = [
    "COL4A1",
    "COL4A2",
    "LAMB1",
    "LAMB2",
    "LAMC1",
    "LAMA4",
    "LAMA5",
    "HSPG2"
]

ECM_GENES = [
    "COL1A1",
    "COL1A2",
    "COL3A1",
    "COL5A1",
    "COL5A2",
    "COL6A1",
    "COL6A2",
    "COL6A3",
    "FN1",
    "VCAN",
    "POSTN",
    "THBS1",
    "SPARCL1"
]

BM_GENES = [
    g for g in BM_GENES
    if g in vasc_adataf.var_names
]

ECM_GENES = [
    g for g in ECM_GENES
    if g in vasc_adataf.var_names
]

# =========================================================
# SUBSET VASCULAR CELLS
# =========================================================

vascular_cells = [
    PERI,
    ENDO,
    ISLET_FIB
]

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL].isin(
        vascular_cells
    )
].copy()

# =========================================================
# SCORE BM / ECM PROGRAMS
# =========================================================

sc.tl.score_genes(
    ad,
    gene_list=BM_GENES,
    score_name="BM_Score",
    use_raw=False
)

sc.tl.score_genes(
    ad,
    gene_list=ECM_GENES,
    score_name="ECM_Score",
    use_raw=False
)

# =========================================================
# SPATIAL COORDS
# =========================================================

coords = ad.obsm[SPATIAL_KEY]

ad.obs["x"] = coords[:,0]
ad.obs["y"] = coords[:,1]

# =========================================================
# 1. BM CONTINUITY
#
# nearest endothelial distance
# lower = tighter vascular wrapping
# =========================================================

endo = ad.obs[
    ad.obs[CELLTYPE_COL] == ENDO
]

peri = ad.obs[
    ad.obs[CELLTYPE_COL] == PERI
]

tree = cKDTree(
    endo[["x","y"]].values
)

dist, idx = tree.query(
    peri[["x","y"]].values,
    k=1
)

peri = peri.copy()

peri["nearest_endo_dist"] = dist

# =========================================================
# donor summary
# =========================================================

bm_continuity = (
    peri.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["nearest_endo_dist"]
    .median()
    .reset_index()
)

bm_continuity.columns = [
    DONOR_COL,
    COND_COL,
    LOC_COL,
    "BM_Continuity"
]

# =========================================================
# 2. VASCULAR COVERAGE
#
# fraction of endothelial cells
# within X microns of a pericyte
# =========================================================

tree_peri = cKDTree(
    peri[["x","y"]].values
)

dist2, idx2 = tree_peri.query(
    endo[["x","y"]].values,
    k=1
)

endo = endo.copy()

endo["nearest_peri_dist"] = dist2

# threshold in microns
THRESH = 20

endo["covered"] = (
    endo["nearest_peri_dist"] < THRESH
).astype(int)

vascular_coverage = (
    endo.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["covered"]
    .mean()
    .reset_index()
)

vascular_coverage.columns = [
    DONOR_COL,
    COND_COL,
    LOC_COL,
    "Vascular_Coverage"
]

# =========================================================
# 3. LOCAL ECM INTENSITY
#
# mean ECM score around islet edge
# =========================================================

edge_cells = ad.obs[
    np.abs(
        ad.obs["dist_edge_islet_um"]
    ) < 20
].copy()

local_ecm = (
    edge_cells.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["ECM_Score"]
    .median()
    .reset_index()
)

local_ecm.columns = [
    DONOR_COL,
    COND_COL,
    LOC_COL,
    "Local_ECM_Intensity"
]

# =========================================================
# 4. ISLET EDGE ORGANIZATION
#
# variance of distance to edge
# lower variance = organized niche
# =========================================================

edge_org = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )["dist_edge_islet_um"]
    .std()
    .reset_index()
)

edge_org.columns = [
    DONOR_COL,
    COND_COL,
    LOC_COL,
    "Islet_Edge_Organization"
]

# =========================================================
# MERGE ALL METRICS
# =========================================================

metrics = bm_continuity.merge(
    vascular_coverage,
    on=[DONOR_COL, COND_COL, LOC_COL]
)

metrics = metrics.merge(
    local_ecm,
    on=[DONOR_COL, COND_COL, LOC_COL]
)

metrics = metrics.merge(
    edge_org,
    on=[DONOR_COL, COND_COL, LOC_COL]
)

# =========================================================
# EXPORT
# =========================================================

metrics.to_csv(
    "/Users/kmeneses/Desktop/Vascular_Niche_Metrics.csv",
    index=False
)

print(metrics.head())

print("\n✔ exported:")
print("/Users/kmeneses/Desktop/Vascular_Niche_Metrics.csv")

# =========================================================
# STATS
# =========================================================

for metric in [
    "BM_Continuity",
    "Vascular_Coverage",
    "Local_ECM_Intensity",
    "Islet_Edge_Organization"
]:

    print("\n====================================")
    print(metric)
    print("====================================")

    for loc in ["Islet", "Exocrine"]:

        d = metrics[
            metrics[LOC_COL] == loc
        ]

        nd = d[
            d[COND_COL] == "ND"
        ][metric]

        t2d = d[
            d[COND_COL] == "T2D"
        ][metric]

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        print(f"\n{loc}")
        print(f"p = {p:.5f}")

        print("ND median:")
        print(np.median(nd))

        print("T2D median:")
        print(np.median(t2d))

# SECTION 26 — CUSTOM PROGRAM SCORING
 6-7 biologically curated gene programs scored per cell
  using sc.tl.score_genes (mean vs random background genes)
 Programs filtered to genes present in var_names
  Minimum 2 genes required to score a program
 Scores added to obs columns as {program}_Score


In [ ]:
# =========================================================
# FILTER EMPTY PROGRAMS
# =========================================================

filtered_programs = {}

for prog, genes in programs.items():

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    # keep only programs with >= 2 genes
    if len(keep) >= 2:

        filtered_programs[prog] = keep

        print(f"\n{prog}")
        print(keep)

    else:

        print(f"\nSKIPPING {prog}")
        print("not enough genes in panel")

# replace
programs = filtered_programs

In [ ]:
# =========================================================
# SCORE PROGRAMS
# =========================================================

score_cols = []

for prog, genes in programs.items():

    score_name = f"{prog}_Score"

    sc.tl.score_genes(
        ad,
        gene_list=genes,
        score_name=score_name,
        use_raw=False
    )

    score_cols.append(score_name)

In [ ]:
# =========================================================
# PROGRAM ENRICHMENT ANALYSIS
# FIND RELATIVE PROGRAMS ENRICHED IN T2D
#
# WORKING FULL VERSION
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE = "Pericytes"

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

GROUPS = ["ND", "T2D"]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[COND_COL].isin(GROUPS))
].copy()

print(ad)

# =========================================================
# PROGRAM DEFINITIONS
# ONLY USE GENES IN YOUR PANEL
# =========================================================

programs = {

    # -----------------------------------------------------
    # STRESS / IMMEDIATE EARLY
    # -----------------------------------------------------

    "Stress_ImmediateEarly": [
        "JUN",
        "FOS",
        "ATF3"
    ],

    # -----------------------------------------------------
    # CONTRACTILE / MURAL
    # -----------------------------------------------------

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2",
        "TAGLN",
        "ACTA2"
    ],

    # -----------------------------------------------------
    # ECM REMODELING
    # -----------------------------------------------------

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    # -----------------------------------------------------
    # VASCULAR BASEMENT MEMBRANE
    # -----------------------------------------------------

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    # -----------------------------------------------------
    # FIBRILLAR ECM
    # -----------------------------------------------------

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    # -----------------------------------------------------
    # PERICYTE IDENTITY
    # -----------------------------------------------------

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4",
        "MCAM",
        "NOTCH3"
    ]
}

# =========================================================
# FILTER TO PRESENT GENES
# =========================================================

filtered_programs = {}

for prog, genes in programs.items():

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(keep) >= 2:

        filtered_programs[prog] = keep

        print("\n====================================")
        print(prog)
        print("====================================")
        print(keep)

    else:

        print(f"\nSKIPPING {prog}")
        print("not enough genes in panel")

programs = filtered_programs

# =========================================================
# SCORE PROGRAMS
# =========================================================

score_cols = []

for prog, genes in programs.items():

    score_name = f"{prog}_Score"

    sc.tl.score_genes(
        ad,
        gene_list=genes,
        score_name=score_name,
        use_raw=False
    )

    score_cols.append(score_name)

# =========================================================
# BUILD DONOR-LEVEL PSEUDOBULK
# =========================================================

pb = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )[score_cols]
    .mean()
    .reset_index()
)

print("\nPSEUDOBULK:")
print(pb.head())

# =========================================================
# TEST PROGRAMS
# =========================================================

results = []

for score in score_cols:

    for loc in ["Islet", "Exocrine"]:

        d = pb[
            pb[LOC_COL] == loc
        ]

        nd = d[
            d[COND_COL] == "ND"
        ][score]

        t2d = d[
            d[COND_COL] == "T2D"
        ][score]

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        results.append({

            "program": score,

            "location": loc,

            "ND_mean":
                nd.mean(),

            "T2D_mean":
                t2d.mean(),

            "delta_T2D_minus_ND":
                t2d.mean() - nd.mean(),

            "pval":
                p
        })

# =========================================================
# RESULTS TABLE
# =========================================================

res = pd.DataFrame(results)

# =========================================================
# FDR CORRECTION
# =========================================================

res["padj"] = multipletests(
    res["pval"],
    method="fdr_bh"
)[1]

# =========================================================
# CLEAN LABELS
# =========================================================

res["program_clean"] = (
    res["program"]
    .str.replace("_Score", "", regex=False)
    .str.replace("_", " ", regex=False)
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "delta_T2D_minus_ND",
    ascending=False
)

# =========================================================
# PRINT RESULTS
# =========================================================

print("\n====================================")
print("PROGRAM ENRICHMENT RESULTS")
print("====================================")

print(
    res[
        [
            "program_clean",
            "location",
            "ND_mean",
            "T2D_mean",
            "delta_T2D_minus_ND",
            "pval",
            "padj"
        ]
    ]
)

# =========================================================
# EXPORT RESULTS
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_Program_Enrichment.csv",
    index=False
)

print("\n✔ exported:")
print("/Users/kmeneses/Desktop/T2D_Program_Enrichment.csv")

# =========================================================
# VISUALIZATION
# =========================================================

colors = []

for x in res["delta_T2D_minus_ND"]:

    if x > 0:
        colors.append("firebrick")
    else:
        colors.append("royalblue")

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(6,4.8)
)

bars = ax.barh(
    y=np.arange(len(res)),
    width=res["delta_T2D_minus_ND"],
    color=colors,
    edgecolor="black",
    linewidth=0.5
)

# =========================================================
# LABELS
# =========================================================

labels = [
    f"{p} ({l})"
    for p, l in zip(
        res["program_clean"],
        res["location"]
    )
]

ax.set_yticks(
    np.arange(len(res))
)

ax.set_yticklabels(labels)

# =========================================================
# SIGNIFICANCE STARS
# =========================================================

for i, row in enumerate(res.itertuples()):

    if row.padj < 0.05:

        ax.text(
            row.delta_T2D_minus_ND,
            i,
            " *",
            fontsize=10,
            va="center"
        )

# =========================================================
# ZERO LINE
# =========================================================

ax.axvline(
    0,
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlabel(
    "T2D − ND program score"
)

ax.set_title(
    f"{CELLTYPE} Program Remodeling in T2D",
    fontsize=10,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# REVIEWER-PROOF DONOR-LEVEL PROGRAM ANALYSIS
#
# KEY FIX:
# donors are independent biological replicates
#
# NOT:
# individual cells
#
# This avoids pseudoreplication.
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE = "Pericytes"

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

GROUPS = ["ND", "T2D"]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[COND_COL].isin(GROUPS))
].copy()

print(ad)

# =========================================================
# ENSURE ONE CONDITION PER DONOR
# =========================================================

donor_check = (
    ad.obs.groupby(DONOR_COL)[COND_COL]
    .nunique()
)

bad = donor_check[donor_check > 1]

if len(bad) > 0:

    print("\nERROR:")
    print("These donors contain multiple conditions:")
    print(bad)

else:

    print("\n✔ donor-condition mapping valid")

# =========================================================
# PROGRAM DEFINITIONS
# =========================================================

programs = {


    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2",
        "TAGLN",
        "ACTA2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4",
        "MCAM",
        "NOTCH3"
    ]
}

# =========================================================
# FILTER PROGRAMS
# =========================================================

filtered_programs = {}

for prog, genes in programs.items():

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(keep) >= 2:

        filtered_programs[prog] = keep

        print("\n====================================")
        print(prog)
        print("====================================")
        print(keep)

programs = filtered_programs

# =========================================================
# SCORE PROGRAMS
# =========================================================

score_cols = []

for prog, genes in programs.items():

    score_name = f"{prog}_Score"

    sc.tl.score_genes(
        ad,
        gene_list=genes,
        score_name=score_name,
        use_raw=False
    )

    score_cols.append(score_name)

# =========================================================
# DONOR-LEVEL PSEUDOBULK
# =========================================================

pb = (
    ad.obs.groupby(
        [
            DONOR_COL,
            COND_COL,
            LOC_COL
        ]
    )[score_cols]
    .mean()
    .reset_index()
)

print("\nPSEUDOBULK:")
print(pb.head())

# =========================================================
# STATS
# =========================================================

results = []

for score in score_cols:

    for loc in ["Islet", "Exocrine"]:

        d = pb[
            pb[LOC_COL] == loc
        ].copy()

        nd = d[
            d[COND_COL] == "ND"
        ][score]

        t2d = d[
            d[COND_COL] == "T2D"
        ][score]

        # ---------------------------------------------
        # skip if not enough donors
        # ---------------------------------------------

        if (
            len(nd) < 2
            or
            len(t2d) < 2
        ):

            continue

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        results.append({

            "program":
                score.replace("_Score", ""),

            "location":
                loc,

            "ND_n":
                len(nd),

            "T2D_n":
                len(t2d),

            "ND_mean":
                nd.mean(),

            "T2D_mean":
                t2d.mean(),

            "delta_T2D_minus_ND":
                t2d.mean() - nd.mean(),

            "effect_size":
                (
                    t2d.mean() - nd.mean()
                ) / np.std(
                    pd.concat([nd, t2d])
                ),

            "pval":
                p
        })

# =========================================================
# RESULTS TABLE
# =========================================================

res = pd.DataFrame(results)

# =========================================================
# FDR
# =========================================================

res["padj"] = multipletests(
    res["pval"],
    method="fdr_bh"
)[1]

# =========================================================
# CLEAN LABELS
# =========================================================

res["program_clean"] = (
    res["program"]
    .str.replace("_", " ")
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    [
        "location",
        "delta_T2D_minus_ND"
    ],
    ascending=[True, False]
)

# =========================================================
# PRINT
# =========================================================

print("\n====================================")
print("PROGRAM ENRICHMENT RESULTS")
print("====================================")

print(
    res[
        [
            "program_clean",
            "location",
            "ND_n",
            "T2D_n",
            "ND_mean",
            "T2D_mean",
            "delta_T2D_minus_ND",
            "effect_size",
            "pval",
            "padj"
        ]
    ]
)

# =========================================================
# EXPORT
# =========================================================

res.to_csv(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Enrichment_DonorLevel.csv",
    index=False
)

print("\n✔ exported")

# =========================================================
# PLOT
# =========================================================

plot_res = res.copy()

colors = [
    "firebrick" if x > 0 else "royalblue"
    for x in plot_res["delta_T2D_minus_ND"]
]

fig, ax = plt.subplots(
    figsize=(6,5)
)

bars = ax.barh(
    y=np.arange(len(plot_res)),
    width=plot_res["delta_T2D_minus_ND"],
    color=colors,
    edgecolor="black",
    linewidth=0.5
)

# =========================================================
# LABELS
# =========================================================

labels = [

    f"{p} ({l})"

    for p, l in zip(
        plot_res["program_clean"],
        plot_res["location"]
    )
]

ax.set_yticks(
    np.arange(len(plot_res))
)

ax.set_yticklabels(
    labels
)

# =========================================================
# STARS
# =========================================================

for i, row in enumerate(
    plot_res.itertuples()
):

    if row.padj < 0.05:

        ax.text(
            row.delta_T2D_minus_ND,
            i,
            " *",
            fontsize=10,
            va="center"
        )

# =========================================================
# ZERO LINE
# =========================================================

ax.axvline(
    0,
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlabel(
    "Mean donor score difference\n(T2D − ND)"
)

ax.set_title(
    f"{CELLTYPE} Program Remodeling",
    fontsize=10,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling_DonorLevel.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling_DonorLevel.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# DONOR-LEVEL BARPLOTS
# WITH SIGNIFICANCE TESTING
#
# ND vs T2D
# FOR EACH PROGRAM
# =========================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import numpy as np
from scipy.stats import mannwhitneyu

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# PROGRAMS TO PLOT
# =========================================================

programs_to_plot = [

    "Stress_ImmediateEarly_Score",

    "Contractile_Mural_Score",

    "ECM_Remodeling_Score",

    "Vascular_BM_Score",

    "Fibrillar_ECM_Score",

    "Pericyte_Identity_Score"
]

# =========================================================
# FUNCTION
# =========================================================

def donor_barplot(score_col, location):

    # -----------------------------------------------------
    # subset
    # -----------------------------------------------------

    d = pb[
        pb["Location"] == location
    ].copy()

    # -----------------------------------------------------
    # stats
    # -----------------------------------------------------

    nd = d[
        d["Health_Status"] == "ND"
    ][score_col]

    t2d = d[
        d["Health_Status"] == "T2D"
    ][score_col]

    stat, p = mannwhitneyu(
        nd,
        t2d,
        alternative="two-sided"
    )

    # -----------------------------------------------------
    # means
    # -----------------------------------------------------

    means = (
        d.groupby("Health_Status")[score_col]
        .mean()
    )

    sems = (
        d.groupby("Health_Status")[score_col]
        .sem()
    )

    # -----------------------------------------------------
    # plot
    # -----------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(2.3,3)
    )

    x = np.arange(2)

    # bars
    ax.bar(
        x,
        means.values,
        yerr=sems.values,
        capsize=3,
        width=0.6,
        edgecolor="black",
        linewidth=1
    )

    # donor points
    sns.stripplot(
        data=d,
        x="Health_Status",
        y=score_col,
        order=["ND", "T2D"],
        color="black",
        size=5,
        jitter=0.08,
        ax=ax
    )

    # -----------------------------------------------------
    # p-value
    # -----------------------------------------------------

    ymax = d[score_col].max()

    ax.text(
        0.5,
        ymax * 1.08,
        f"MWU p = {p:.3f}",
        ha="center",
        fontsize=7
    )

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    clean = (
        score_col
        .replace("_Score", "")
        .replace("_", " ")
    )

    ax.set_title(
        f"{clean}\n{location}",
        fontsize=9,
        weight="bold"
    )

    ax.set_xlabel("")
    ax.set_ylabel("Mean donor score")

    # -----------------------------------------------------
    # clean
    # -----------------------------------------------------

    sns.despine()

    plt.tight_layout()

    # -----------------------------------------------------
    # save
    # -----------------------------------------------------

    save_name = (
        f"/Users/kmeneses/Desktop/"
        f"{clean}_{location}_donorBarplot"
    )

    plt.savefig(
        save_name + ".svg",
        bbox_inches="tight"
    )

    plt.savefig(
        save_name + ".pdf",
        bbox_inches="tight"
    )

    plt.show()

# =========================================================
# RUN ALL
# =========================================================

for prog in programs_to_plot:

    donor_barplot(
        prog,
        "Islet"
    )

    donor_barplot(
        prog,
        "Exocrine"
    )

print("✔ all donor-level barplots complete")

In [ ]:
# =========================================================
# FULL CORRECT WORKFLOW
#
# 1. RUN PERICYTE PROGRAM ANALYSIS
# 2. SAVE AS peri_res
#
# 3. RUN ISLET-ASSOCIATED FIBROBLAST ANALYSIS
# 4. SAVE AS fibro_res
#
# 5. COMBINE INTO ONE HEATMAP
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# PROGRAMS
# =========================================================

programs = {

    "Stress_ImmediateEarly": [
        "JUN",
        "FOS",
        "ATF3"
    ],

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4"
    ]
}

# =========================================================
# FUNCTION
# =========================================================

def run_program_analysis(celltype):

    print("\n====================================")
    print(celltype)
    print("====================================")

    # -----------------------------------------------------
    # subset
    # -----------------------------------------------------

    ad = vasc_adataf[
        (vasc_adataf.obs[CELLTYPE_COL] == celltype)
    ].copy()

    # -----------------------------------------------------
    # filter genes
    # -----------------------------------------------------

    filtered_programs = {}

    for prog, genes in programs.items():

        keep = [
            g for g in genes
            if g in ad.var_names
        ]

        if len(keep) >= 2:

            filtered_programs[prog] = keep

    # -----------------------------------------------------
    # score
    # -----------------------------------------------------

    score_cols = []

    for prog, genes in filtered_programs.items():

        score_name = f"{prog}_Score"

        sc.tl.score_genes(
            ad,
            gene_list=genes,
            score_name=score_name,
            use_raw=False
        )

        score_cols.append(score_name)

    # -----------------------------------------------------
    # donor pseudobulk
    # -----------------------------------------------------

    pb = (
        ad.obs.groupby(
            [
                DONOR_COL,
                COND_COL,
                LOC_COL
            ]
        )[score_cols]
        .mean()
        .reset_index()
    )

    # -----------------------------------------------------
    # stats
    # -----------------------------------------------------

    results = []

    for score in score_cols:

        for loc in ["Islet", "Exocrine"]:

            d = pb[
                pb[LOC_COL] == loc
            ]

            nd = d[
                d[COND_COL] == "ND"
            ][score]

            t2d = d[
                d[COND_COL] == "T2D"
            ][score]

            if (
                len(nd) < 2
                or
                len(t2d) < 2
            ):
                continue

            stat, p = mannwhitneyu(
                nd,
                t2d,
                alternative="two-sided"
            )

            results.append({

                "program":
                    score.replace("_Score", ""),

                "location":
                    loc,

                "ND_mean":
                    nd.mean(),

                "T2D_mean":
                    t2d.mean(),

                "delta_T2D_minus_ND":
                    t2d.mean() - nd.mean(),

                "pval":
                    p
            })

    # -----------------------------------------------------
    # dataframe
    # -----------------------------------------------------

    res = pd.DataFrame(results)

    res["padj"] = multipletests(
        res["pval"],
        method="fdr_bh"
    )[1]

    res["program_clean"] = (
        res["program"]
        .str.replace("_", " ")
    )

    return res

# =========================================================
# RUN BOTH
# =========================================================

peri_res = run_program_analysis(
    "Pericytes"
)

fibro_res = run_program_analysis(
    "Islet-associated Fibroblasts"
)

# =========================================================
# ADD LABELS
# =========================================================

peri_res["group"] = "Pericytes"

fibro_res["group"] = (
    "Islet-associated\nFibroblasts"
)

# =========================================================
# COMBINE
# =========================================================

combined = pd.concat([
    peri_res,
    fibro_res
])

# =========================================================
# KEEP ISLET ONLY
# =========================================================

combined = combined[
    combined["location"] == "Islet"
].copy()

# =========================================================
# PIVOT
# =========================================================

heat = combined.pivot_table(
    index="program_clean",
    columns="group",
    values="delta_T2D_minus_ND"
)

# =========================================================
# ORDER
# =========================================================

program_order = [

    "Stress ImmediateEarly",

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# SCALE
# =========================================================

vabs = np.abs(
    heat.values
).max()

# =========================================================
# HEATMAP
# =========================================================

fig, ax = plt.subplots(
    figsize=(3.5,3)
)

sns.heatmap(
    heat,
    cmap="RdBu_r",
    center=0,
    vmin=-vabs,
    vmax=vabs,
    linewidths=0.6,
    linecolor="white",
    cbar_kws={
        "label":
        "Mean donor score\n(T2D − ND)",
        "shrink": 0.6
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat.shape[0]):

    for j in range(heat.shape[1]):

        val = heat.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.02f}",
            ha="center",
            va="center",
            fontsize=6
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Divergent Islet Stromal Remodeling",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=8
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/Combined_Stromal_Programs_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/Combined_Stromal_Programs_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

combined.to_csv(
    f"{OUT}/Combined_Stromal_Programs.csv",
    index=False
)

print("✔ COMPLETE")

In [ ]:
# =========================================================
# FULL MULTI-COMPARTMENT STROMAL REMODELING
#
# COMPARES:
#
# 1. Islet Pericytes
# 2. Exocrine Pericytes
# 3. Islet-associated Fibroblasts
# 4. Exocrine Fibroblasts
#
# T2D vs ND
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# PROGRAMS
# =========================================================

programs = {

    "Stress_ImmediateEarly": [
        "JUN",
        "FOS",
        "ATF3"
    ],

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4"
    ]
}

# =========================================================
# FUNCTION
# =========================================================

def run_program_analysis(celltype, location):

    print("\n====================================")
    print(celltype)
    print(location)
    print("====================================")

    # -----------------------------------------------------
    # subset
    # -----------------------------------------------------

    ad = vasc_adataf[
        (vasc_adataf.obs[CELLTYPE_COL] == celltype) &
        (vasc_adataf.obs[LOC_COL] == location)
    ].copy()

    # -----------------------------------------------------
    # filter programs
    # -----------------------------------------------------

    filtered_programs = {}

    for prog, genes in programs.items():

        keep = [
            g for g in genes
            if g in ad.var_names
        ]

        if len(keep) >= 2:

            filtered_programs[prog] = keep

    # -----------------------------------------------------
    # score
    # -----------------------------------------------------

    score_cols = []

    for prog, genes in filtered_programs.items():

        score_name = f"{prog}_Score"

        sc.tl.score_genes(
            ad,
            gene_list=genes,
            score_name=score_name,
            use_raw=False
        )

        score_cols.append(score_name)

    # -----------------------------------------------------
    # donor pseudobulk
    # -----------------------------------------------------

    pb = (
        ad.obs.groupby(
            [
                DONOR_COL,
                COND_COL
            ]
        )[score_cols]
        .mean()
        .reset_index()
    )

    # -----------------------------------------------------
    # stats
    # -----------------------------------------------------

    results = []

    for score in score_cols:

        nd = pb[
            pb[COND_COL] == "ND"
        ][score]

        t2d = pb[
            pb[COND_COL] == "T2D"
        ][score]

        if (
            len(nd) < 2
            or
            len(t2d) < 2
        ):
            continue

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        results.append({

            "program":
                score.replace("_Score", ""),

            "group":
                f"{location}\n{celltype}",

            "ND_mean":
                nd.mean(),

            "T2D_mean":
                t2d.mean(),

            "delta_T2D_minus_ND":
                t2d.mean() - nd.mean(),

            "pval":
                p
        })

    # -----------------------------------------------------
    # dataframe
    # -----------------------------------------------------

    res = pd.DataFrame(results)

    res["padj"] = multipletests(
        res["pval"],
        method="fdr_bh"
    )[1]

    res["program_clean"] = (
        res["program"]
        .str.replace("_", " ")
    )

    return res

# =========================================================
# RUN ALL 4 GROUPS
# =========================================================

groups = [

    ("Pericytes", "Islet"),

    ("Pericytes", "Exocrine"),

    ("Islet-associated Fibroblasts", "Islet"),

    ("Fibroblasts", "Exocrine")
]

all_results = []

for celltype, location in groups:

    r = run_program_analysis(
        celltype,
        location
    )

    all_results.append(r)

# =========================================================
# COMBINE
# =========================================================

combined = pd.concat(
    all_results
)

# =========================================================
# PIVOT
# =========================================================

heat = combined.pivot_table(
    index="program_clean",
    columns="group",
    values="delta_T2D_minus_ND"
)

# =========================================================
# ORDER
# =========================================================

program_order = [

    "Stress ImmediateEarly",

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# COLUMN ORDER
# =========================================================

desired_cols = [

    "Islet\nPericytes",

    "Exocrine\nPericytes",

    "Islet\nIslet-associated Fibroblasts",

    "Exocrine\nFibroblasts"
]

heat = heat[
    [c for c in desired_cols if c in heat.columns]
]

# =========================================================
# SCALE
# =========================================================

vabs = np.abs(
    heat.values
).max()

# =========================================================
# HEATMAP
# =========================================================

fig, ax = plt.subplots(
    figsize=(4.5,3.2)
)

sns.heatmap(
    heat,
    cmap="RdBu_r",
    center=0,
    vmin=-vabs,
    vmax=vabs,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label":
        "Mean donor score\n(T2D − ND)",
        "shrink": 0.7
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat.shape[0]):

    for j in range(heat.shape[1]):

        val = heat.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.02f}",
            ha="center",
            va="center",
            fontsize=6
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Compartment-Specific Stromal Remodeling in T2D",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=7
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/MultiCompartment_Stromal_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/MultiCompartment_Stromal_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# EXPORT
# =========================================================

combined.to_csv(
    f"{OUT}/MultiCompartment_Stromal_Programs.csv",
    index=False
)

print("✔ COMPLETE")

In [ ]:
# =========================================================
# MULTI-COMPARTMENT HEATMAP
# USING MEAN CELL SCORES
#
# instead of:
# delta(T2D - ND)
#
# columns become:
#
# ND Islet Peri
# T2D Islet Peri
# ND Exo Peri
# T2D Exo Peri
# etc
# =========================================================

# =========================================================
# BUILD MATRIX OF MEAN SCORES
# =========================================================

heat_rows = []

for celltype, location in groups:

    # ---------------------------------------------
    # subset
    # ---------------------------------------------

    ad = vasc_adataf[
        (vasc_adataf.obs[CELLTYPE_COL] == celltype) &
        (vasc_adataf.obs[LOC_COL] == location)
    ].copy()

    # ---------------------------------------------
    # score programs
    # ---------------------------------------------

    for prog, genes in programs.items():

        keep = [
            g for g in genes
            if g in ad.var_names
        ]

        if len(keep) < 2:
            continue

        score_name = f"{prog}_Score"

        # only compute if not already present
        if score_name not in ad.obs.columns:

            sc.tl.score_genes(
                ad,
                gene_list=keep,
                score_name=score_name,
                use_raw=False
            )

        # -----------------------------------------
        # means
        # -----------------------------------------

        for cond in ["ND", "T2D"]:

            vals = ad.obs.loc[
                ad.obs[COND_COL] == cond,
                score_name
            ]

            heat_rows.append({

                "Program":
                    prog.replace("_", " "),

                "Group":
                    f"{cond}\n{location}\n{celltype}",

                "MeanScore":
                    vals.mean()
            })

# =========================================================
# DATAFRAME
# =========================================================

heat_df = pd.DataFrame(
    heat_rows
)

# =========================================================
# PIVOT
# =========================================================

heat = heat_df.pivot_table(
    index="Program",
    columns="Group",
    values="MeanScore"
)

# =========================================================
# ORDER PROGRAMS
# =========================================================

program_order = [

    "Stress ImmediateEarly",

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# ORDER COLUMNS
# =========================================================

desired_cols = [

    "ND\nIslet\nPericytes",
    "T2D\nIslet\nPericytes",

    "ND\nExocrine\nPericytes",
    "T2D\nExocrine\nPericytes",

    "ND\nIslet\nIslet-associated Fibroblasts",
    "T2D\nIslet\nIslet-associated Fibroblasts",

    "ND\nExocrine\nFibroblasts",
    "T2D\nExocrine\nFibroblasts"
]

heat = heat[
    [c for c in desired_cols if c in heat.columns]
]

# =========================================================
# SCALE
# =========================================================

vabs = np.abs(
    heat.values
).max()

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(7,3.5)
)

sns.heatmap(
    heat,
    cmap="RdBu_r",
    center=0,
    vmin=-vabs,
    vmax=vabs,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label":
        "Mean cell score",
        "shrink": 0.7
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat.shape[0]):

    for j in range(heat.shape[1]):

        val = heat.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.02f}",
            ha="center",
            va="center",
            fontsize=5.5
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Compartment-Specific Stromal Programs",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=6
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/MeanCellScore_Stromal_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/MeanCellScore_Stromal_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ mean-score heatmap complete")

In [ ]:
# =========================================================
# PERICYTES ONLY
# Z-SCORE HEATMAP
#
# compares:
#
# ND Islet
# T2D Islet
# ND Exocrine
# T2D Exocrine
#
# each PROGRAM is z-scored across conditions
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import zscore

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# PROGRAMS
# =========================================================

programs = {

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4"
    ]
}

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL] == "Pericytes"
].copy()

print(ad)

# =========================================================
# BUILD MATRIX
# =========================================================

heat_rows = []

for prog, genes in programs.items():

    # -----------------------------------------------------
    # keep genes present
    # -----------------------------------------------------

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(keep) < 2:
        continue

    # -----------------------------------------------------
    # score
    # -----------------------------------------------------

    score_name = f"{prog}_Score"

    if score_name not in ad.obs.columns:

        sc.tl.score_genes(
            ad,
            gene_list=keep,
            score_name=score_name,
            use_raw=False
        )

    # -----------------------------------------------------
    # means
    # -----------------------------------------------------

    for loc in ["Islet", "Exocrine"]:

        for cond in ["ND", "T2D"]:

            vals = ad.obs.loc[
                (ad.obs[LOC_COL] == loc) &
                (ad.obs[COND_COL] == cond),
                score_name
            ]

            heat_rows.append({

                "Program":
                    prog.replace("_", " "),

                "Group":
                    f"{cond}\n{loc}",

                "MeanScore":
                    vals.mean()
            })

# =========================================================
# DATAFRAME
# =========================================================

heat_df = pd.DataFrame(
    heat_rows
)

# =========================================================
# PIVOT
# =========================================================

heat = heat_df.pivot_table(
    index="Program",
    columns="Group",
    values="MeanScore"
)

# =========================================================
# ORDER PROGRAMS
# =========================================================

program_order = [

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# ORDER COLUMNS
# =========================================================

desired_cols = [

    "ND\nIslet",
    "T2D\nIslet",

    "ND\nExocrine",
    "T2D\nExocrine"
]

heat = heat[
    [c for c in desired_cols if c in heat.columns]
]

# =========================================================
# Z-SCORE BY PROGRAM
# =========================================================

heat_z = heat.copy()

heat_z[:] = zscore(
    heat,
    axis=1,
    nan_policy="omit"
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(4,3)
)

sns.heatmap(
    heat_z,
    cmap="RdBu_r",
    center=0,
    vmin=-2,
    vmax=2,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label":
        "Program z-score",
        "shrink": 0.7
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat_z.shape[0]):

    for j in range(heat_z.shape[1]):

        val = heat_z.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.1f}",
            ha="center",
            va="center",
            fontsize=6
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Pericyte Program States",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=7
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/Pericyte_Zscore_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/Pericyte_Zscore_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ pericyte z-score heatmap complete")

In [ ]:
# =========================================================
# FIBROBLAST PROGRAM STATES
# Z-SCORE HEATMAP
#
# compares:
#
# ND Islet-associated Fibroblasts
# T2D Islet-associated Fibroblasts
#
# ND Exocrine Fibroblasts
# T2D Exocrine Fibroblasts
#
# each PROGRAM is z-scored across conditions
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import zscore

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# PROGRAMS
# =========================================================

programs = {


    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL5A3",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Fibroblast_Identity": [

    "LUM",
    "DCN",
    "FBLN1",
    "PRELP",
    "SFRP2"

]
}

# =========================================================
# BUILD MATRIX
# =========================================================

heat_rows = []

# ---------------------------------------------------------
# GROUP DEFINITIONS
# ---------------------------------------------------------

group_defs = [

    (
        "Islet-associated Fibroblasts",
        "Islet"
    ),

    (
        "Fibroblasts",
        "Exocrine"
    )
]

# =========================================================
# LOOP
# =========================================================

for celltype, location in group_defs:

    ad = vasc_adataf[
        (vasc_adataf.obs[CELLTYPE_COL] == celltype) &
        (vasc_adataf.obs[LOC_COL] == location)
    ].copy()

    print("\n====================================")
    print(celltype)
    print(location)
    print(ad)

    # -----------------------------------------------------
    # programs
    # -----------------------------------------------------

    for prog, genes in programs.items():

        keep = [
            g for g in genes
            if g in ad.var_names
        ]

        if len(keep) < 2:
            continue

        score_name = f"{prog}_Score"

        # -------------------------------------------------
        # compute scores
        # -------------------------------------------------

        if score_name not in ad.obs.columns:

            sc.tl.score_genes(
                ad,
                gene_list=keep,
                score_name=score_name,
                use_raw=False
            )

        # -------------------------------------------------
        # means
        # -------------------------------------------------

        for cond in ["ND", "T2D"]:

            vals = ad.obs.loc[
                ad.obs[COND_COL] == cond,
                score_name
            ]

            heat_rows.append({

                "Program":
                    prog.replace("_", " "),

                "Group":
                    f"{cond}\n{location}",

                "MeanScore":
                    vals.mean()
            })

# =========================================================
# DATAFRAME
# =========================================================

heat_df = pd.DataFrame(
    heat_rows
)

# =========================================================
# PIVOT
# =========================================================

heat = heat_df.pivot_table(
    index="Program",
    columns="Group",
    values="MeanScore"
)

# =========================================================
# ORDER PROGRAMS
# =========================================================

program_order = [

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Fibroblast Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# ORDER COLUMNS
# =========================================================

desired_cols = [

    "ND\nIslet",
    "T2D\nIslet",

    "ND\nExocrine",
    "T2D\nExocrine"
]

heat = heat[
    [c for c in desired_cols if c in heat.columns]
]

# =========================================================
# Z-SCORE
# =========================================================

heat_z = heat.copy()

heat_z[:] = zscore(
    heat,
    axis=1,
    nan_policy="omit"
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(4,3)
)

sns.heatmap(
    heat_z,
    cmap="RdBu_r",
    center=0,
    vmin=-2,
    vmax=2,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label":
        "Program z-score",
        "shrink": 0.7
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat_z.shape[0]):

    for j in range(heat_z.shape[1]):

        val = heat_z.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.1f}",
            ha="center",
            va="center",
            fontsize=6
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Fibroblast Program States",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=7
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/Fibroblast_Zscore_Heatmap.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/Fibroblast_Zscore_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ fibroblast z-score heatmap complete")

In [ ]:
# =========================================================
# PERICYTES ONLY
# Z-SCORE HEATMAP
#
# compares:
#
# ND Islet
# T2D Islet
# ND Exocrine
# T2D Exocrine
#
# each PROGRAM is z-scored across conditions
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import zscore

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# PROGRAMS
# =========================================================

programs = {

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL5A3",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4"
    ]
}

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL] == "Pericytes"
].copy()

print(ad)

# =========================================================
# BUILD MATRIX
# =========================================================

heat_rows = []

for prog, genes in programs.items():

    # -----------------------------------------------------
    # keep genes present
    # -----------------------------------------------------

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(keep) < 2:
        continue

    # -----------------------------------------------------
    # score
    # -----------------------------------------------------

    score_name = f"{prog}_Score"

    if score_name not in ad.obs.columns:

        sc.tl.score_genes(
            ad,
            gene_list=keep,
            score_name=score_name,
            use_raw=False
        )

    # -----------------------------------------------------
    # means
    # -----------------------------------------------------

    for loc in ["Islet", "Exocrine"]:

        for cond in ["ND", "T2D"]:

            vals = ad.obs.loc[
                (ad.obs[LOC_COL] == loc) &
                (ad.obs[COND_COL] == cond),
                score_name
            ]

            heat_rows.append({

                "Program":
                    prog.replace("_", " "),

                "Group":
                    f"{cond}\n{loc}",

                "MeanScore":
                    vals.mean()
            })

# =========================================================
# DATAFRAME
# =========================================================

heat_df = pd.DataFrame(
    heat_rows
)

# =========================================================
# PIVOT
# =========================================================

heat = heat_df.pivot_table(
    index="Program",
    columns="Group",
    values="MeanScore"
)

# =========================================================
# ORDER PROGRAMS
# =========================================================

program_order = [

    "Stress ImmediateEarly",

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# ORDER COLUMNS
# =========================================================

desired_cols = [

    "ND\nIslet",
    "T2D\nIslet",

    "ND\nExocrine",
    "T2D\nExocrine"
]

heat = heat[
    [c for c in desired_cols if c in heat.columns]
]

# =========================================================
# Z-SCORE BY PROGRAM
# =========================================================

heat_z = heat.copy()

heat_z[:] = zscore(
    heat,
    axis=1,
    nan_policy="omit"
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(4,3)
)

sns.heatmap(
    heat_z,
    cmap="RdBu_r",
    center=0,
    vmin=-2,
    vmax=2,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label":
        "Program z-score",
        "shrink": 0.7
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat_z.shape[0]):

    for j in range(heat_z.shape[1]):

        val = heat_z.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.1f}",
            ha="center",
            va="center",
            fontsize=6
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Pericyte Program States",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=7
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/Pericyte_Zscore_Heatmap.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/Pericyte_Zscore_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ pericyte z-score heatmap complete")

In [ ]:
# =========================================================
# MULTI-COMPARTMENT HEATMAP
# USING CELL Z-SCORES
#
# Instead of raw mean module scores,
# each program is z-scored across groups
#
# Better for:
# ✔ comparing relative enrichment
# ✔ visual balance
# ✔ publication figures
# =========================================================

import scipy.stats as stats

# =========================================================
# BUILD MATRIX OF MEAN SCORES
# =========================================================

heat_rows = []

for celltype, location in groups:

    # ---------------------------------------------
    # subset
    # ---------------------------------------------

    ad = vasc_adataf[
        (vasc_adataf.obs[CELLTYPE_COL] == celltype) &
        (vasc_adataf.obs[LOC_COL] == location)
    ].copy()

    # ---------------------------------------------
    # score programs
    # ---------------------------------------------

    for prog, genes in programs.items():

        keep = [
            g for g in genes
            if g in ad.var_names
        ]

        if len(keep) < 2:
            continue

        score_name = f"{prog}_Score"

        # -----------------------------------------
        # compute score
        # -----------------------------------------

        if score_name not in ad.obs.columns:

            sc.tl.score_genes(
                ad,
                gene_list=keep,
                score_name=score_name,
                use_raw=False
            )

        # -----------------------------------------
        # mean score
        # -----------------------------------------

        for cond in ["ND", "T2D"]:

            vals = ad.obs.loc[
                ad.obs[COND_COL] == cond,
                score_name
            ]

            heat_rows.append({

                "Program":
                    prog.replace("_", " "),

                "Group":
                    f"{cond}\n{location}\n{celltype}",

                "MeanScore":
                    vals.mean()
            })

# =========================================================
# DATAFRAME
# =========================================================

heat_df = pd.DataFrame(
    heat_rows
)

# =========================================================
# PIVOT
# =========================================================

heat = heat_df.pivot_table(
    index="Program",
    columns="Group",
    values="MeanScore"
)

# =========================================================
# ORDER PROGRAMS
# =========================================================

program_order = [

    "Stress ImmediateEarly",

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# ORDER COLUMNS
# =========================================================

desired_cols = [

    "ND\nIslet\nPericytes",
    "T2D\nIslet\nPericytes",

    "ND\nExocrine\nPericytes",
    "T2D\nExocrine\nPericytes",

    "ND\nIslet\nIslet-associated Fibroblasts",
    "T2D\nIslet\nIslet-associated Fibroblasts",

    "ND\nExocrine\nFibroblasts",
    "T2D\nExocrine\nFibroblasts"
]

heat = heat[
    [c for c in desired_cols if c in heat.columns]
]

# =========================================================
# Z-SCORE EACH PROGRAM (ROW)
# =========================================================

heat_z = heat.copy()

heat_z[:] = stats.zscore(
    heat,
    axis=1,
    nan_policy="omit"
)

# =========================================================
# SCALE
# =========================================================

vabs = np.abs(
    heat_z.values
).max()

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(7,3.5)
)

sns.heatmap(
    heat_z,
    cmap="RdBu_r",
    center=0,
    vmin=-2,
    vmax=2,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label":
        "Program z-score",
        "shrink": 0.7
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat_z.shape[0]):

    for j in range(heat_z.shape[1]):

        val = heat_z.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.1f}",
            ha="center",
            va="center",
            fontsize=5.5
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Compartment-Specific Stromal Programs",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=6
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/Zscore_Stromal_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/Zscore_Stromal_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ z-score heatmap complete")

In [ ]:
# =========================================================
# PERICYTES ONLY
# MEAN CELL SCORE HEATMAP
#
# compares:
#
# ND Islet Pericytes
# T2D Islet Pericytes
# ND Exocrine Pericytes
# T2D Exocrine Pericytes
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# PROGRAMS
# =========================================================

programs = {

    "Stress_ImmediateEarly": [
        "JUN",
        "FOS",
        "ATF3"
    ],

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4"
    ]
}

# =========================================================
# SUBSET PERICYTES
# =========================================================

ad = vasc_adataf[
    vasc_adataf.obs[CELLTYPE_COL] == "Pericytes"
].copy()

print(ad)

# =========================================================
# BUILD MATRIX
# =========================================================

heat_rows = []

for prog, genes in programs.items():

    # -----------------------------------------------------
    # keep valid genes
    # -----------------------------------------------------

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(keep) < 2:
        continue

    # -----------------------------------------------------
    # score
    # -----------------------------------------------------

    score_name = f"{prog}_Score"

    if score_name not in ad.obs.columns:

        sc.tl.score_genes(
            ad,
            gene_list=keep,
            score_name=score_name,
            use_raw=False
        )

    # -----------------------------------------------------
    # means
    # -----------------------------------------------------

    for loc in ["Islet", "Exocrine"]:

        for cond in ["ND", "T2D"]:

            vals = ad.obs.loc[
                (ad.obs[LOC_COL] == loc) &
                (ad.obs[COND_COL] == cond),
                score_name
            ]

            heat_rows.append({

                "Program":
                    prog.replace("_", " "),

                "Group":
                    f"{cond}\n{loc}",

                "MeanScore":
                    vals.mean()
            })

# =========================================================
# DATAFRAME
# =========================================================

heat_df = pd.DataFrame(
    heat_rows
)

# =========================================================
# PIVOT
# =========================================================

heat = heat_df.pivot_table(
    index="Program",
    columns="Group",
    values="MeanScore"
)

# =========================================================
# ORDER PROGRAMS
# =========================================================

program_order = [

    "Stress ImmediateEarly",

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# ORDER COLUMNS
# =========================================================

desired_cols = [

    "ND\nIslet",
    "T2D\nIslet",

    "ND\nExocrine",
    "T2D\nExocrine"
]

heat = heat[
    [c for c in desired_cols if c in heat.columns]
]

# =========================================================
# SCALE
# =========================================================

vabs = np.abs(
    heat.values
).max()

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(4,3)
)

sns.heatmap(
    heat,
    cmap="RdBu_r",
    center=0,
    vmin=-vabs,
    vmax=vabs,
    linewidths=0.7,
    linecolor="white",
    cbar_kws={
        "label":
        "Mean cell score",
        "shrink": 0.7
    },
    ax=ax
)

# =========================================================
# LABEL VALUES
# =========================================================

for i in range(heat.shape[0]):

    for j in range(heat.shape[1]):

        val = heat.iloc[i, j]

        ax.text(
            j + 0.5,
            i + 0.5,
            f"{val:.02f}",
            ha="center",
            va="center",
            fontsize=6
        )

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Pericyte Program States",
    fontsize=10,
    fontweight="bold",
    pad=10
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=7
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/Pericyte_MeanScore_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/Pericyte_MeanScore_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ pericyte heatmap complete")

In [ ]:
# =========================================================
# PROGRAM ENRICHMENT ANALYSIS
# FIND RELATIVE PROGRAMS ENRICHED IN T2D
#
# WORKING FULL VERSION
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE = "Islet-associated Fibroblasts"

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

GROUPS = ["ND", "T2D"]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[COND_COL].isin(GROUPS))
].copy()

print(ad)

# =========================================================
# PROGRAM DEFINITIONS
# ONLY USE GENES IN YOUR PANEL
# =========================================================

programs = {

    # -----------------------------------------------------
    # STRESS / IMMEDIATE EARLY
    # -----------------------------------------------------

    "Stress_ImmediateEarly": [
        "JUN",
        "FOS",
        "ATF3"
    ],

    # -----------------------------------------------------
    # CONTRACTILE / MURAL
    # -----------------------------------------------------

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2",
        "TAGLN",
        "ACTA2"
    ],

    # -----------------------------------------------------
    # ECM REMODELING
    # -----------------------------------------------------

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    # -----------------------------------------------------
    # VASCULAR BASEMENT MEMBRANE
    # -----------------------------------------------------

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    # -----------------------------------------------------
    # FIBRILLAR ECM
    # -----------------------------------------------------

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    # -----------------------------------------------------
    # PERICYTE IDENTITY
    # -----------------------------------------------------

}

# =========================================================
# FILTER TO PRESENT GENES
# =========================================================

filtered_programs = {}

for prog, genes in programs.items():

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(keep) >= 2:

        filtered_programs[prog] = keep

        print("\n====================================")
        print(prog)
        print("====================================")
        print(keep)

    else:

        print(f"\nSKIPPING {prog}")
        print("not enough genes in panel")

programs = filtered_programs

# =========================================================
# SCORE PROGRAMS
# =========================================================

score_cols = []

for prog, genes in programs.items():

    score_name = f"{prog}_Score"

    sc.tl.score_genes(
        ad,
        gene_list=genes,
        score_name=score_name,
        use_raw=False
    )

    score_cols.append(score_name)

# =========================================================
# BUILD DONOR-LEVEL PSEUDOBULK
# =========================================================

pb = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )[score_cols]
    .mean()
    .reset_index()
)

print("\nPSEUDOBULK:")
print(pb.head())

# =========================================================
# TEST PROGRAMS
# =========================================================

results = []

for score in score_cols:

    for loc in ["Islet", "Exocrine"]:

        d = pb[
            pb[LOC_COL] == loc
        ]

        nd = d[
            d[COND_COL] == "ND"
        ][score]

        t2d = d[
            d[COND_COL] == "T2D"
        ][score]

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        results.append({

            "program": score,

            "location": loc,

            "ND_mean":
                nd.mean(),

            "T2D_mean":
                t2d.mean(),

            "delta_T2D_minus_ND":
                t2d.mean() - nd.mean(),

            "pval":
                p
        })

# =========================================================
# RESULTS TABLE
# =========================================================

res = pd.DataFrame(results)

# =========================================================
# FDR CORRECTION
# =========================================================

res["padj"] = multipletests(
    res["pval"],
    method="fdr_bh"
)[1]

# =========================================================
# CLEAN LABELS
# =========================================================

res["program_clean"] = (
    res["program"]
    .str.replace("_Score", "", regex=False)
    .str.replace("_", " ", regex=False)
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "delta_T2D_minus_ND",
    ascending=False
)

# =========================================================
# PRINT RESULTS
# =========================================================

print("\n====================================")
print("PROGRAM ENRICHMENT RESULTS")
print("====================================")

print(
    res[
        [
            "program_clean",
            "location",
            "ND_mean",
            "T2D_mean",
            "delta_T2D_minus_ND",
            "pval",
            "padj"
        ]
    ]
)

# =========================================================
# EXPORT RESULTS
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_Program_Enrichment.csv",
    index=False
)

print("\n✔ exported:")
print("/Users/kmeneses/Desktop/T2D_Program_Enrichment.csv")

# =========================================================
# VISUALIZATION
# =========================================================

colors = []

for x in res["delta_T2D_minus_ND"]:

    if x > 0:
        colors.append("firebrick")
    else:
        colors.append("royalblue")

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(6,4.8)
)

bars = ax.barh(
    y=np.arange(len(res)),
    width=res["delta_T2D_minus_ND"],
    color=colors,
    edgecolor="black",
    linewidth=0.5
)

# =========================================================
# LABELS
# =========================================================

labels = [
    f"{p} ({l})"
    for p, l in zip(
        res["program_clean"],
        res["location"]
    )
]

ax.set_yticks(
    np.arange(len(res))
)

ax.set_yticklabels(labels)

# =========================================================
# SIGNIFICANCE STARS
# =========================================================

for i, row in enumerate(res.itertuples()):

    if row.padj < 0.05:

        ax.text(
            row.delta_T2D_minus_ND,
            i,
            " *",
            fontsize=10,
            va="center"
        )

# =========================================================
# ZERO LINE
# =========================================================

ax.axvline(
    0,
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlabel(
    "T2D − ND program score"
)

ax.set_title(
    f"{CELLTYPE} Program Remodeling in T2D",
    fontsize=10,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# PROGRAM ENRICHMENT ANALYSIS
# FIND RELATIVE PROGRAMS ENRICHED IN T2D
#
# WORKING FULL VERSION
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE = "Endothelial Cells"

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

GROUPS = ["ND", "T2D"]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    (vasc_adataf.obs[CELLTYPE_COL] == CELLTYPE) &
    (vasc_adataf.obs[COND_COL].isin(GROUPS))
].copy()

print(ad)

# =========================================================
# PROGRAM DEFINITIONS
# ONLY USE GENES IN YOUR PANEL
# =========================================================

programs = {

    # -----------------------------------------------------
    # STRESS / IMMEDIATE EARLY
    # -----------------------------------------------------

    "Stress_ImmediateEarly": [
        "JUN",
        "FOS",
        "ATF3"
    ],

    # -----------------------------------------------------
    # CONTRACTILE / MURAL
    # -----------------------------------------------------

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2",
        "TAGLN",
        "ACTA2"
    ],

    # -----------------------------------------------------
    # ECM REMODELING
    # -----------------------------------------------------

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    # -----------------------------------------------------
    # VASCULAR BASEMENT MEMBRANE
    # -----------------------------------------------------

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    # -----------------------------------------------------
    # FIBRILLAR ECM
    # -----------------------------------------------------

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    # -----------------------------------------------------
    # PERICYTE IDENTITY
    # -----------------------------------------------------

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4",
        "MCAM",
        "NOTCH3"
    ]
}

# =========================================================
# FILTER TO PRESENT GENES
# =========================================================

filtered_programs = {}

for prog, genes in programs.items():

    keep = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(keep) >= 2:

        filtered_programs[prog] = keep

        print("\n====================================")
        print(prog)
        print("====================================")
        print(keep)

    else:

        print(f"\nSKIPPING {prog}")
        print("not enough genes in panel")

programs = filtered_programs

# =========================================================
# SCORE PROGRAMS
# =========================================================

score_cols = []

for prog, genes in programs.items():

    score_name = f"{prog}_Score"

    sc.tl.score_genes(
        ad,
        gene_list=genes,
        score_name=score_name,
        use_raw=False
    )

    score_cols.append(score_name)

# =========================================================
# BUILD DONOR-LEVEL PSEUDOBULK
# =========================================================

pb = (
    ad.obs.groupby(
        [DONOR_COL, COND_COL, LOC_COL]
    )[score_cols]
    .mean()
    .reset_index()
)

print("\nPSEUDOBULK:")
print(pb.head())

# =========================================================
# TEST PROGRAMS
# =========================================================

results = []

for score in score_cols:

    for loc in ["Islet", "Exocrine"]:

        d = pb[
            pb[LOC_COL] == loc
        ]

        nd = d[
            d[COND_COL] == "ND"
        ][score]

        t2d = d[
            d[COND_COL] == "T2D"
        ][score]

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        results.append({

            "program": score,

            "location": loc,

            "ND_mean":
                nd.mean(),

            "T2D_mean":
                t2d.mean(),

            "delta_T2D_minus_ND":
                t2d.mean() - nd.mean(),

            "pval":
                p
        })

# =========================================================
# RESULTS TABLE
# =========================================================

res = pd.DataFrame(results)

# =========================================================
# FDR CORRECTION
# =========================================================

res["padj"] = multipletests(
    res["pval"],
    method="fdr_bh"
)[1]

# =========================================================
# CLEAN LABELS
# =========================================================

res["program_clean"] = (
    res["program"]
    .str.replace("_Score", "", regex=False)
    .str.replace("_", " ", regex=False)
)

# =========================================================
# SORT
# =========================================================

res = res.sort_values(
    "delta_T2D_minus_ND",
    ascending=False
)

# =========================================================
# PRINT RESULTS
# =========================================================

print("\n====================================")
print("PROGRAM ENRICHMENT RESULTS")
print("====================================")

print(
    res[
        [
            "program_clean",
            "location",
            "ND_mean",
            "T2D_mean",
            "delta_T2D_minus_ND",
            "pval",
            "padj"
        ]
    ]
)

# =========================================================
# EXPORT RESULTS
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/T2D_Program_Enrichment.csv",
    index=False
)

print("\n✔ exported:")
print("/Users/kmeneses/Desktop/T2D_Program_Enrichment.csv")

# =========================================================
# VISUALIZATION
# =========================================================

colors = []

for x in res["delta_T2D_minus_ND"]:

    if x > 0:
        colors.append("firebrick")
    else:
        colors.append("royalblue")

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(6,4.8)
)

bars = ax.barh(
    y=np.arange(len(res)),
    width=res["delta_T2D_minus_ND"],
    color=colors,
    edgecolor="black",
    linewidth=0.5
)

# =========================================================
# LABELS
# =========================================================

labels = [
    f"{p} ({l})"
    for p, l in zip(
        res["program_clean"],
        res["location"]
    )
]

ax.set_yticks(
    np.arange(len(res))
)

ax.set_yticklabels(labels)

# =========================================================
# SIGNIFICANCE STARS
# =========================================================

for i, row in enumerate(res.itertuples()):

    if row.padj < 0.05:

        ax.text(
            row.delta_T2D_minus_ND,
            i,
            " *",
            fontsize=10,
            va="center"
        )

# =========================================================
# ZERO LINE
# =========================================================

ax.axvline(
    0,
    linestyle="--",
    color="black",
    linewidth=1
)

# =========================================================
# AXES
# =========================================================

ax.set_xlabel(
    "T2D − ND program score"
)

ax.set_title(
    f"{CELLTYPE} Program Remodeling in T2D",
    fontsize=10,
    weight="bold"
)

# =========================================================
# CLEAN
# =========================================================

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling.svg",
    bbox_inches="tight"
)

plt.savefig(
    f"/Users/kmeneses/Desktop/{CELLTYPE}_Program_Remodeling.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =========================================================
# COMPARE:
# PERICYTES vs ISLET-ASSOCIATED FIBROBLASTS
#
# ND vs T2D
# IN ISLETS ONLY
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "merged_cell_type"
COND_COL = "Health_Status"
LOC_COL = "Location"
DONOR_COL = "Sample"

CELLTYPES = [
    "Pericytes",
    "Islet-associated Fibroblasts"
]

# =========================================================
# PROGRAMS
# =========================================================

programs = {

    "Stress_ImmediateEarly": [
        "JUN",
        "FOS",
        "ATF3"
    ],

    "Contractile_Mural": [
        "MYL9",
        "CALD1",
        "TPM2"
    ],

    "ECM_Remodeling": [
        "THBS1",
        "ADAM9",
        "SERPINF1",
        "MFGE8",
        "SPARCL1",
        "FBN1",
        "MMP2"
    ],

    "Vascular_BM": [
        "COL4A1",
        "COL4A2",
        "LAMB1",
        "LAMB2",
        "LAMC1",
        "HSPG2"
    ],

    "Fibrillar_ECM": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "COL5A1",
        "COL5A2",
        "COL6A1",
        "COL6A2",
        "COL6A3"
    ],

    "Pericyte_Identity": [
        "RGS5",
        "PDGFRB",
        "CSPG4"
    ]
}

# =========================================================
# SUBSET
# =========================================================

ad = vasc_adataf[
    (vasc_adataf.obs[LOC_COL] == "Islet") &
    (vasc_adataf.obs[CELLTYPE_COL].isin(CELLTYPES))
].copy()

print(ad)

# =========================================================
# FILTER PROGRAMS
# =========================================================

valid_programs = {}

for prog, genes in programs.items():

    valid = [
        g for g in genes
        if g in ad.var_names
    ]

    if len(valid) > 0:

        valid_programs[prog] = valid

        print("\n====================================")
        print(prog)
        print("====================================")
        print(valid)

# =========================================================
# SCORE PROGRAMS
# =========================================================

score_cols = []

for prog, genes in valid_programs.items():

    score_name = f"{prog}_Score"

    sc.tl.score_genes(
        ad,
        gene_list=genes,
        score_name=score_name,
        use_raw=False
    )

    score_cols.append(score_name)

# =========================================================
# PSEUDOBULK
# =========================================================

pb = (
    ad.obs.groupby(
        [
            DONOR_COL,
            COND_COL,
            CELLTYPE_COL
        ]
    )[score_cols]
    .mean()
    .reset_index()
)

print("\nPSEUDOBULK:")
print(pb.head())

# =========================================================
# STATS
# =========================================================

results = []

for score in score_cols:

    for celltype in CELLTYPES:

        d = pb[
            pb[CELLTYPE_COL] == celltype
        ]

        nd = d[
            d[COND_COL] == "ND"
        ][score]

        t2d = d[
            d[COND_COL] == "T2D"
        ][score]

        stat, p = mannwhitneyu(
            nd,
            t2d,
            alternative="two-sided"
        )

        results.append({

            "program": score.replace("_Score", ""),

            "celltype": celltype,

            "ND_mean": nd.mean(),

            "T2D_mean": t2d.mean(),

            "delta_T2D_minus_ND":
                t2d.mean() - nd.mean(),

            "pval": p
        })

# =========================================================
# RESULTS TABLE
# =========================================================

res = pd.DataFrame(results)

res["padj"] = multipletests(
    res["pval"],
    method="fdr_bh"
)[1]

res = res.sort_values(
    "delta_T2D_minus_ND",
    ascending=False
)

# =========================================================
# CLEAN LABELS
# =========================================================

res["program_clean"] = (
    res["program"]
    .str.replace("_", " ")
)

# =========================================================
# PRINT
# =========================================================

print("\n====================================")
print("PROGRAM ENRICHMENT RESULTS")
print("====================================")

print(res)

# =========================================================
# EXPORT
# =========================================================

res.to_csv(
    "/Users/kmeneses/Desktop/Peri_vs_IsletAssocFibro_Programs.csv",
    index=False
)

print("\n✔ exported")

In [ ]:
# =========================================================
# PLOT PROGRAM RESULTS
# PERICYTES vs ISLET-ASSOCIATED FIBROBLASTS
# =========================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import numpy as np
import pandas as pd

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# CLEAN LABELS
# =========================================================

res["celltype_clean"] = (
    res["celltype"]
    .replace({
        "Pericytes": "Pericytes",
        "Islet-associated Fibroblasts":
            "Islet-associated\nFibroblasts"
    })
)

# =========================================================
# PIVOT
# =========================================================

heat = res.pivot_table(
    index="program_clean",
    columns="celltype_clean",
    values="delta_T2D_minus_ND"
)

# =========================================================
# ORDER PROGRAMS
# =========================================================

program_order = [

    "Stress ImmediateEarly",

    "Contractile Mural",

    "ECM Remodeling",

    "Vascular BM",

    "Fibrillar ECM",

    "Pericyte Identity"
]

heat = heat.loc[
    [p for p in program_order if p in heat.index]
]

# =========================================================
# COLOR SCALE
# =========================================================

vabs = np.abs(
    heat.values
).max()

# =========================================================
# HEATMAP
# =========================================================

fig, ax = plt.subplots(
    figsize=(3.5,3)
)

sns.heatmap(
    heat,
    cmap="RdBu_r",
    center=0,
    vmin=-vabs,
    vmax=vabs,
    linewidths=0.5,
    linecolor="white",
    cbar_kws={
        "label":
        "Mean score\n(T2D − ND)"
    },
    ax=ax
)

# =========================================================
# TITLE
# =========================================================

ax.set_title(
    "Divergent Stromal Remodeling Programs",
    fontsize=9,
    fontweight="bold",
    pad=10
)

# =========================================================
# AXES
# =========================================================

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0,
    labelsize=8
)

ax.tick_params(
    axis="y",
    rotation=0,
    labelsize=7
)

# =========================================================
# CLEAN
# =========================================================

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    "/Users/kmeneses/Desktop/Peri_vs_Fibro_Programs_Heatmap.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Peri_vs_Fibro_Programs_Heatmap.pdf",
    bbox_inches="tight"
)

plt.show()

# =========================================================
# DOTPLOT VERSION
# =========================================================

dot_df = heat.reset_index().melt(
    id_vars="program_clean",
    var_name="Celltype",
    value_name="Score"
)

fig, ax = plt.subplots(
    figsize=(3.5,3)
)

sc = ax.scatter(
    x=dot_df["Celltype"],
    y=dot_df["program_clean"],
    c=dot_df["Score"],
    s=np.abs(dot_df["Score"]) * 5000 + 100,
    cmap="RdBu_r",
    vmin=-vabs,
    vmax=vabs,
    edgecolor="black",
    linewidth=0.5
)

cbar = plt.colorbar(
    sc,
    ax=ax,
    shrink=0.6
)

cbar.set_label(
    "Mean score\n(T2D − ND)"
)

ax.set_title(
    "Program Remodeling Dotplot",
    fontsize=9,
    fontweight="bold"
)

ax.set_xlabel("")
ax.set_ylabel("")

ax.tick_params(
    axis="x",
    rotation=0
)

ax.tick_params(
    axis="y",
    labelsize=7
)

plt.tight_layout()

plt.savefig(
    "/Users/kmeneses/Desktop/Peri_vs_Fibro_Programs_Dotplot.svg",
    bbox_inches="tight"
)

plt.savefig(
    "/Users/kmeneses/Desktop/Peri_vs_Fibro_Programs_Dotplot.pdf",
    bbox_inches="tight"
)

plt.show()

print("✔ plots complete")